# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.

PubMed dùng EFetch GET theo PMID; các website khác dùng Requests/Selenium.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.

Khôi phục checkpoint PubMed 6.000/110.000: giải nén ZIP đầy đủ
`ids_5979207_7061937_synced_vi_en_zh_efetch.zip`, đặt `RESUME_RUN_PATH` tới thư mục chứa manifest,
`ID_FROM=5979207`, `ID_TO=7061937`, `POSTPROCESS_ONLY=False`, `PUBMED_USE_EFETCH=True`, `MAX_NEW=None`.
Giữ `PUBMED_UPGRADE_EFETCH_SNAPSHOT=True`: chỉ nâng cấp transport EFetch; XML parser/schema, core/runner/probe
và dữ liệu cũ giữ nguyên. Log sẽ ghi completed/pending thực tế. Không dùng ZIP diagnostics để resume.
Checkpoint full ZIP được tạo mỗi 10.000 URL mới hoặc 15 phút, và cuối lượt. Nếu có Secret HF_TOKEN quyền ghi,
upload ZIP/checksum vào `checkpoints/<run>/latest.zip` và `latest.json` trong cùng commit. Đây là backup;
run còn pending không upload corpus như hoàn tất. Có thể tắt upload, ZIP local vẫn được tạo.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
REQUESTS_TRUST_ENV = False  # False như lượt thử Kaggle; True nếu môi trường bắt buộc proxy
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
VOV_USE_CURL = True  # vov.vn và các subdomain dùng curl/Chrome
LONGCHAU_USE_CURL = True
LONGCHAU_WORKERS = 1
LONGCHAU_DELAY = 5.0
LONGCHAU_ATTEMPTS = 3
LONGCHAU_CONNECT_TIMEOUT = 10.0
LONGCHAU_READ_TIMEOUT = 45.0
LONGCHAU_REQUEST_BUDGET = 180.0
LONGCHAU_RETRY_FAILED_ON_RESUME = True

BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
PUBMED_USE_EFETCH = True  # Mặc định: PubMed đi EFetch GET, ghép theo PMID
PUBMED_EFETCH_BATCH_SIZE = 100
PUBMED_EFETCH_WORKERS = 1
PUBMED_EFETCH_GAP_SECONDS = 0.5
PUBMED_EFETCH_ATTEMPTS = 5
PUBMED_EFETCH_CONNECT_TIMEOUT = 10.0
PUBMED_EFETCH_READ_TIMEOUT = 90.0
PUBMED_EFETCH_BUDGET = 300.0
PUBMED_SELENIUM_FIRST = True  # Chỉ dùng khi PUBMED_USE_EFETCH = False
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

PUBMED_UPGRADE_EFETCH_SNAPSHOT = True  # Upgrade transport compatible with existing XML; keep saved runner/core/data
PUBMED_MIN_BATCH_SIZE = 1
PUBMED_MAX_SPLIT_REQUESTS = 50
PUBMED_FALLBACK_BUDGET = 900.0
PUBMED_PENDING_RETRY_PASSES = 1
PUBMED_MAX_FAILED_BATCHES = 3
CHECKPOINT_ENABLED = True
CHECKPOINT_UPLOAD_ENABLED = True
CHECKPOINT_EVERY_URLS = 10000
CHECKPOINT_EVERY_SECONDS = 900.0
CHECKPOINT_HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
CHECKPOINT_HF_SECRET_NAME = 'HF_TOKEN'

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import os
import shutil
import sys

NCBI_EMAIL = os.environ.get('NCBI_EMAIL', '')  # Email liên hệ; có thể điền ở đây
NCBI_API_KEY = os.environ.get('NCBI_API_KEY', '')  # Tùy chọn; không ghi vào settings/log

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(REQUESTS_TRUST_ENV) is not bool:
    raise ValueError('REQUESTS_TRUST_ENV phải là bool.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
source_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
previous_tag = source_tag + ('_efetch' if PUBMED_USE_EFETCH else '')
default_tag = previous_tag + ('_longchau_v2' if LONGCHAU_USE_CURL else '')
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = previous_tag if (WORKDIR / 'runs' / previous_tag).exists() else (source_tag if (WORKDIR / 'runs' / source_tag).exists() else f'ids_{ID_FROM}_{ID_TO}_synced')
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and (previous / 'corpus_config.json').exists():
        old_settings = json.loads((previous / 'corpus_config.json').read_text(encoding='utf-8')).get('settings', {})
        if old_settings.get('pubmed_use_efetch') is not True:
            raise ValueError('Run cũ chưa dùng EFetch. Chọn RUN_TAG mới và RECOVERY_RUN_PATH.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'huggingface_hub', 'curl_cffi']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    pinned = {line.split('==',1)[0].strip().replace('_','-').lower() for line in requirements_path.read_text(encoding='utf-8').splitlines() if line.strip() and not line.startswith('#')}
    install_args = ['-r', str(requirements_path)] + [p for p in package_names if p.replace('_','-').lower() not in pinned]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import importlib.util
import hashlib
import zlib

EMBEDDED_FILES = {'ask120_snapshot.py': 'c-pO7dz0M8asPim1$0UUfVjK&pyHA*I?=R5N32k!OdeIKT5cdLcJ>xJECAxcyEE^qNL5%i70Pxcie;O0EIVSt*k!U(v15v)&F^Ay?~Z@*E2O(;1{l0{S)y1a9d<C&)6<XNboUGv78c$*^NtqUQOqLE_J>+9fTv6Lz+;`a*WWp%efLdm6hzwc%90&#Y8O)$CvM=whz%BZTdnmm)0|)u+D-z0qcDxNO~%4l3w&=!d;k4}Z71))-_^ABam4JQ7AH~aBx%IjS{SiTV!K`^Vw2#K?P=Jzg*?EVPLTQui((jH?`W>CB`{IQqRz>;G-u45%`kBNM2i?qI@AU%c884mMD~`3jCWfL3k$7L6ihVB8l^a;Wohmt459=U>IaF93yE92j}wGD^6d~7fWKSf_t=ido;#2~BgTiMkq50^lARn1ZE@(iiJ<CjPdxc*oJ_n{tM%TAv*%8pIjyZ}I+tmO3%_$|MQ@$>hxN0^-&{X)Hh*nhw5?$<vGDcVXU?skgIAZuyDmsxZ)^J2)>c<(*=v3G{E2hxFpKr>@%6W1K*aDx=z7eE^hK$}#okiqoBbmfy1<%lw$_iIJXP%5(%@gDUx;6M^6>VP`ycK9^xpK|7rnn+xwY`+l^b7P`C;EU_U`M6n|SPv-ar23$}gb#<1erL=z`zByxd+{TG}&vc`I~YJ$(GLC-;A29((%f-KW3)>A}zcY5MTN;oZBsn5p;d(T(<(S3Z39=;q&Ei2JWh@88(Jbra^?|M1TKt?SeKcc&lyH;^g!JGlJ|=rO%|d-~vy<}rAC^(A=Q#aDl&x6Ib5<KI4UiX=x9>BGQDg2=M{c#B2)s-{nGJUsa1m(!d7e(<|b^fvDlf`KqUu||RC1zTx|{q{fnkHah14j=ys79jfxVrn-|?(kQ)r`N6(iKhOhA8h%>-ar1~!NcD@dGPuEAFe{bz1I1Y);q_~{ykU<u*|ifPp|#olZQV!_@9SQ9zEjHL(L!Dyg$A6zxzM={=siQ%YPq!d~f=%SMj%4*R4AT4<1jieSG-)FQ)%;=kVUI@!jFQKR)@><AYD`O#giM$>U$`U;pis`&Z!qlGXi>{_XJIb?W(t&z@en%4t3Q<UjX6{gmeV@Qdm9AMXG1_Q9=xo__vc)0=;Wk!I`ciFeMtcj7H;{rKN;TaBH_4HFy@Cp#X)2X_?N6M6z{Arl^?NfP+<&JR=S?FY)pOB-O;lJuLWj-NYsLeZxa3|Sb&E?PBqPl7Z~iL3=qF$lJ;LBfeoHt^5I2uJXz4u9l?251xZaG1E9J^9-g?E0>=DW9<qpwlEIowZs+Hqv}pj_tYMV+MkmxytRD0s8ppSaCWq(4UUrcn$uCy)=MVT{pV`jc#a|f|3B$=b%$0BZj-NCEeTtP%_{@L51LE2(pt{Wx&Mr-c)BXc)xB8Xggtuz=l3fk`QOazp-x8ej)pfgE&FS$kl*#v&b-UAi);Vmj4LkRgCJ6QqP;%i8Ge;!28B9ItGy@+UdY&a%Gm|*>TKb!w0<;B@?s6pyuudJn7UC*p%|6#hMu%fOSrbhO{;0zDSSO0}CilDaQ{PT8=RYhC6Lg7i?Dm7RgS?)?hYHXAD~&?5hN-F|#9BUZgu?TwwyJx(5IC7B1-hV@B`7R;Pc&)Qglj>unr<l51ze61zxmFpnTI*jrux#?jYa?W?4Kf6tD{6gZ#VjVMS%V;RIglB_ciHaDC!d)XE1*6rAFU2>`oFngxcFyfB}=^WG@(6u|X6BfffJvk?gN|RCNH5|S%SblBk=#t*2{@?_}66%ukiQz{^1=W$0T+OVC`M{LxZ@|dxwY0&^vt(!BGwV-=y)jrhy1c@f*~CBF9eglX3s$Slkuj~(L`%*30MqImw5%dIRJ7}-ENi4bT|gT{jx?2bJH^q$L{Yg?BXSfWZR9$MZnm|&@jEuCB^TJX!@@)}P6fec8lH%vAZlyp0pEbqTSSlA%o#k_u#jstVxea{3|Wv$&oBXGgC5=1)*~P#+$o9JY-i$A4AmuV*^$o`((H|9t0F(aI7Vb$jMGqB{)QnYFl%OLsUmyN_EIJ!Ml=}m-hG$iI4(8Kj<LHkrR{=G<Rc9c!cnX$VUC~=LvQdUiXVZ56b}W-%<==Eqe7qKfh0iyGH4YTl{wDX^@eDGo*O3yb?&+lSPxn3Fn?(KNeqy<S~g=hc752}e1{oCp_|ws43LvK$A$$`qDCZW5`lvA<uui8q7XPB>w?r@Yo&d^zybaXFm>56<hG8)E}^6|>L>|nA^6~Q&GJ%+r(qO?+!PDDAqh^Z0wrwWmMb=nSK68k=hn!{1Z9#Xq$QwTH1~@ANrV&{tWmKfBgy9^Ue+-4y2K)Pgm{^-3Ah`;V+^4v$ZY7^Ua*le=?S>*p$pkkA}k^#)*yq+2?FaLI@7>*Hnk+s;HP#83<2zs2N+vH3L|644E)5yq|EVBSe}fD7BL`NJbyB$3HEGnM0PkP?B^2~jFK<i9nG+}in8+vqH0BJfXt!TNYs5ii^?GA>V5#3qmeg-U0j?c1O?7OEQZKl&cFFW9;JjDG+GHPJy(A&J9J2`AVsn$<b#`*kYSaBqTJk?ct>|4`+GZ<&$eVJmxoU7P>GuoJ4)g$HyIm*>N~pGZ_Se~5Gsnbz_Y=Z11&vbzi;MYnGbOd1g;`Evm84bs(1%FL3>`)B(uChEO-lq$7CNRcx+M!?<w^G^HudBR>^y+rp?*l4;4nlsM88;K-$NK*1!v#O|S!4Rx#MD(Jb_tqgVvfs$yv|D0vb;*JML4VS;8QJa5(m$`yb0v#lxc3R_aGv6XqMqG2&9my{6LYyAhhQ=+DuMYklonB|bA`LaxW?u;@u{K_*{DhjIptEGX`){nI^N(Uj|!;*`bfU+hqL@eLPq=1sfma?DoPRO)KE2%-m^g59PBhX%2%kxfU3WZBPe;!II&AnAll<>kZ``t*y-NiUBOb39RV{iom6N@T+211>fjW%k+x_$<_+2&1=);1=GhQ7E+o3PnKi?U$@F}iJS;mLysP;!}m{_%p@SIvx*?ZOC9kpaa+!%rsz771jcXPf~Nq>&3s^gPJb3|=BLu>>aR>a;N#M9(ZH&3umDQXl0m)`89hJEnmN4R11Oqgv)^l8O){8i(qV7A7#NRA{>fp%ex1O1^e@fT6~Mw$RQYxf6pjnei<$Wl#jXrT<f1#2zebBqSao;}*a6RnVMR6M~S|s3yF|vDaTBIhw}G5<OgA+1p+IrhGJeoyG3-i{JM_+ROc8yeqct-}(Wzb&kqLQR=9gcEwIftjh0^ei>D+n*B1k=+djhy-wxL3cgvXdvjF0k-iDcMCdI%?SLXmQ}P1E=o7rJh9M8z#;0;o{ruTe8t$_}3MFc7gZU8NIUQ2W2<#C9vo2P(3xRr5ou@wQI2|BqF>ziAe%&3ufn{K1LD~Ztnwkmm?8s75u7-?F3`hL1nymI)?r?2kh(`GK@`AKLu_R?=zLG|Ag^lY)JtA?@Ar=?M%Yvm$wsRITiQf$T@<NJ(^R>((`n4_x3ZLZQUn@}@s=@{;rhprYssITq(KQH1Ri;`i${b-$O|pvO0+K|dBt%K3#p4$WSz!5%;(akG5Pd0Kvu@5FzD1D4zkreI>)Mn?*i0$-sKlp0K!hJM$i)K8HiaLUK4Jcc!AD;d-=bk@C#~Tpw@m6Ux5!F7D(5MLIWwX?5l*)n6S(JDu;*6Ro5fq|f`t}5FN>dw=~B(Q=P$0F_2w0eXqrnq4${bBIXf0YyCt^SIGq4=2A)x=CS;`q44Ta1PFq->sm?M1JBEO21CKO;{1Pu%O~PcS{*|31wu3>KDlPsh_3>17!v_~AzDAxiiJ<PDZvm)j>X}1CO{t@d6S?dg1wcJa!84DYF%`*I!e<#$<eaxAbP_GxLa8uELz#3b9>LQ!1f**EmUSWtXFYrL#Ep)g(7O1`qUBIyO=^_aFY1(gWR1c}WzL+D8igO>daF|xjTRP`aav0^3$?O$bIO1sw~`wiWwLb}T=&IcIPt#bJuVKHc@$tP*u`^|n)STM60!n}vX5kw;hu_Gs^YN93l;#LyKDoK@{_bIs^XV2@RSx(9<~jq(P&%3Lrm?6w%o44K#@8_@7$$TjW=xj1)~rfb}f0nKv>KwUu(8FJnbY>TtZ5LC!+`FvJmgbFtpd-D=pNTfqF`>dN9d0(4tS5iba9POSxDQ;r>;u*Tm$Vm>^Nlwc4^r9=>nM{i5u)3e1W$0O?kWL-<ZoS*?h=;<mgT&1@f!Sp<42m1(L#NOH0YLF&N8U|p}(U@CHX?P}S@<f>EX@)~-n2&W=BU>YW7qOo?-LLk*OH(LTwU2f`LGzmGsRAHrZQ_GSN>~pGtPuvsM-0#|9h(Xz99cM+yfIn#LATjurDNUDR#4h4-uK?s2a3Rqir_`@8Lb7v8o9W_Jfss;mv|f&lLN1K02(?bwHA0D{F!)ctIyk?eNjk^^B+zsr%+}Hv&(rws4gsWcy-kBFCwNv&J0l#N*dqpz0ssxi*>G0b;s;CN*!Bx(F|oIerS`1gu2Ulvkt?AHF+$)a2^2C1;7rW%EYbH{S+KzsmC@>PJLnfj(`=Y-&DM55EB{y_ZW%-rT^p>4-!yvm=;s%F9d-kmboH|=jyrF!v13c(m?TZ7Ci!Ifxk^P8s|DC?kV(-R<8(A~w_%pif_7Q<<U3n`Z=vjzvV+*o`V^@aghiJF<|A0ur3E8LkdtSResH-w)!ws5*Ngmk_JDc7IlrpbQ`zW)KqWy<2N@T4iCmb{FE}me%ax&8h*{ZS=8w!ErC+ENLLcbGON2;7W@(h8Ddq%{#s_%|Bd1~EUClXka?`0LlNXbzjEsBLjQX-TK-yqd7zG1XU+pYan9;~(p;_D2ur^fqiS;kp01=kiTYYtDX-TBtW5`#S5AzLK0%xzZnNk#<mq+uUPq!UNXW{IY`EaDW`#~CGscsTXAcEQjNP$0C8;RJ(6i#x7yL1f}OL04ZMcN<=wgB389k|co0-R28IINNR2zJ9=U73iEjdySyVd6NQ*#0i>sC{5xviZF`e#Xkl?LPg@jl+9a4?erPJH2~-dhHJXy?c1~F?jy;N1s0Z*^iZ2E|;;ma3pRVr6JX~nMKu9goDd_Un>A?GK`zEGmJ*V2V;V+RlhBB2wF;@&TFS$V!(hl1eK=r6{7?&?E-c*FS!Tt8^3!@>&k!O8GU(veqzAOEEt++y-n>6ZJC?dMJKxnNg)^Hk|Bl&dcN6Ncu&Q`JK2hsbTz#cg%Oc9!swdZQyD4>-{58HekAe-waHWl)w(8(DS7dX5kq%0Vo(QiaDB#UaeJVQ0^-62-<v87!3JF#U{+-!!Cj=gVqwMlOY<lZ4)7ZnRwx6q=Ac0#RSh0sWVKNlQCqOd8LzMBDC5gf4vZ-`6=o|2yF5&zd6ckCHI=+5Q8BV=!9p3B1^wc33y=OODO9onWkBvd=!wFGgzCzkrD`vAfrV%jjKcy(gtB_PiF{N%f`ckov+=fL){cYvoaCOUJ=rlYxQXJcaA2kOP3Nu48fEt4*h4+XqVW=J;I`!^oZ46u7Xnm&!@_`MGH^Fixan+(^ILVAVtG8D3ExS0lzkvOe_dN%`f77aB6{giE_b-Obp1fJj+{_k_3Ndz8i&TamK8&qzZ}IPVQF;Cmrq`UDOs%nH^%b&I>zR*P=>cE8}-gRmvB8KWFOz;^vjMumvU)<%BkZ1&%s~~isZy<(n=6;Gk(sUFPn$7RU_z&POlCJ`tw&J2U&3%hYWAsN}8o3Q!LC~fH;0M)Fouhr7lQRNLNG4W>O4~e$i{QLGsLs0#1mpW5g+b9wCZ4^43BPR=fdFvXoc#3l`sMIO)L>v`UWm7qw^pBP%{<ZVeNyEVX8w9f^e&u3#u}5QoK8O(wGf*m;pjMle*y%)Yy-4Lxie)o?ij(?M~)*^+1FxUb1$b7gZX7z6YvHng)KB7<(a*DtoOO#aoNLtfAl!PKk(OpL&sC8I7?BUCa=BQwGzb@s+6Nk!adcJZU264jaEPx)4rTvVm(QMrCqEtja=meTx2vgLBXY^nvAtVB&;A?Oi6k?q5)S@ET<)ogW0O8r@-B(g8`6#kUcx#%Eyr)G>u=Y@OWV{<1f9@{H_^9|@LzO3x>219n}lKW&|6PRXq%(jOvd~pg%c9m|#F1c*W5@<l8qNkj11?bS4@?TYARba*PYNfF{kblN$L2ey$HHa@gt4S03cFa76W<v)R`pe=fjJBwC<wX>@g8~b9S==qj-rzZQH@k>Ydwx<b!FSag3*G!+T{Ga@-wu2~3$9M5o5EW4ERf}woa}Vy7Opt)uWn~tbvnyBVy*N&F6d{7Ih&{`Oz<e5KGwx&4<sr-K*pjilxnj}PT*RwG0nMTB}#}c^XO35C~p7~`3soeE&+Ta{LaDKb{(j;oA7Za7FL0;T?cOM8f0q3mu5hjfglaI(XJpw0T^b0Lc0+H+D%SerKb9DyUFXG%b(lI#Q_-z@=H5pM1<e`2>@K!6+gbDgGg$jjEeh#(wJp>Ioap<1PJ7@sC?5d$3g)IE3s`^n9OO4+7i`$fUBOOg?>XUB&BSlh!q@?UyuL{;dPCw{8STAdt<!ZPz35%spKJhi9}7cqh@$bg#rgxnrntOmMbXWuYXxX=AvWCvKkKL&lyhsKVq;@G5',
 'audit_snapshot.py': 'c-qBW-H+VJaev>xqHiw&Ilfu$N+%!U*^i4((#48TACPp116B(jccy2CUJgm|L%ZYE0tx((rvN!PfMEm%PBwyrV;FXj070aOJnX^$$L24|ud2Jr=8!YHdp;uqw8J5*tGcV}tE-B$zP|owJdVOxiZCwZBnylA<}kr!T!>K^OyVRj!eK6gc!d9=jHBRCMv)weOuh`|vB;-V7Gj)aA`rtYIF3YC#@_n+`dT<kldKRyHc5jlmuurJnTcV7!NVd<;#_EeJ4p$8nSr1DBuvfqG{~n>cvyWcW|0VT{`6%&3{q(xsSm(<Wf4Z^?vL{%R&>*#AWVklV|+D08P=A}s&5s_e4d-jG7g8yNREOcFt<)-ktd5xO7C7IXJFSeT+`(4JyR6NWjrjBB&yaKWu-q1hEvUnGK*-s%o_uOdwCj##hL<%Ktxz;H?-4vkY&j+jeSvq`=FY>SX)~g$uZ_1j9gl6Z?GotFOx-?#mosWN`g_&4?P0&3wctwGRD%vc(Uu1#dzbdoL*038YZ(e7#1!;HIX^O@qV0yv1{?gi<4{?MBz`Q>pcAUj?))B&g1pC71DefxCGursQ1~!)0^M_KmmADc`^zoGRJ7Mw&2+3y(hlAhskK(7iRhz;Yjj?#IB&np88s?u`irVrcp4IPLILEv51qxjPmj#3sYC4vgY?Ni6zF$Sz63}n#Cv?mbB>(Ll)V*HH7Br(Jm6sc;adjy*?vjXn8V}X(3z&RkHM$q#?V9-Oc*qvmh!ZUoV|e8MFKZhml;E*V@`&R|4nGf|DFOvwN#o@e$fP0$ZdlE5tHzjx>-tH%l{_=PoIW0#jmhI4@*QTWZZmZ3gWwUG-LyPs_uZ9QhD#KLeEvGRcFqNF>;hU8s@GJ%v>77(sYNFcI(U3g<B+yPd&GMe?aE9E(*!2-T{&=o0V)2!~m_2y@>4dmqbUI2E5i_{4L}s(4<Q$LSg0AOojv!cg)IYC2Ie49KDZ+5vj9rgy@EZ#g_)9>+7@U7WEaWQ0jNA0^4r-G)%nzjPLoC~E}cFdnhWRs`P@&gSC>_wPd696fVu_FgUv@H;4@!X-?7v47C>f;5%!$W<_nJ=DXrvIXTREc|+}^oYK&_6lbHGns|sxyX|;14E0jh#;5prOf6cIK=M3f{0vMe0C!2KVTE%k)(NGxzSuCV^K_{@*gr2EYle~5sM4PIO>V)JGkLsL2S>I<~p0TUTdaJWZ^n`6@<v?k=W4H3PBmKYe^yIZhrdY6EO?(SpXp!h*LuzxDL+N6+M0?NnVx-H&3M6g%y(w($LQ#J!OumlDvR)4joXluP2SshH5373K7W&#xC4_QNlXId^K<fLoz|q#=Dv!dNRzIT8}Q_j_o=is;BmT_!^@#5YMtwo?VACqZ0A}RgnCs&7VlduI5os?1-Cx9+p(I_->d|FG?+_$(F{FAjmfs*JQKyCFy1;N#?bsBV1RNdmaD}j}QYYYH%%CjZ{Gp6hO~zf9rrB47DZ16Ai<5%xzi99xcy+tg@9&G!4V4<`O7yP&C9ftif|-Qf9pJ@?_im7+xHhHH{|XX9S@q!O-Lpbf{F1_CiFvY{z>%5eu*0C35cCFDPFSCw`7tjPOOl>~Iuta5ccjdwF^2X3q2cJ@x^8ad}8)MLr6vnj4E#b*o`dcv<XXdRGx1hTnuYy-Wo%IhGmeO5eB$vY`7s{($VH+D`38_?a7H9=G)!Y$W;Rm|U{GAwd+m7SIA34@@8r$fzE{?_^#08luAp7eISeTmW|kLFz*j!%3(nV`$av0+3q4L3N91@wUerM|<|q*p@d<5$&QSS&K5&fMs{v7i3Q+Nk-8fgY&+ALeJE`n#i|rJ$@L??&x7npay|a2&Wv(eI~^6NuOAmThWB5Ro4`<efseB)2H|U=Hb1&Ci-!jIE^rvSmji?v;nUiQ(+qTsu$<P(abj#aNAt=8%SIK9JOB<pe`73JSl^T#67(sno;^TvjKmKdAbPmjBfGsv*uz*!!hml*`gsCW~pw2pI)ZpngK);!QMb{K-uKEVIcrmziki5xCe{#(Sk}=uPM~+$r+ANWV4hbtx=U?n`J3Fm0mcQt2=gtdus5X^>}3!Icrh0__c2AT9bK|3MMa-Oj<EB&(@IjjbJ&AAdGyf8pk$}ziL2^qJnttHYQM6N#H~XVMSfaNH#2~JV}8}!t^o3L31q{y@lvV&KW^94mS?iR^Wn6g{$R%QSkefMB9>Vq;<hAaKeU(_A5$+97jLRi`ph%&FU{k`}tKofI2q##-crspU2v&rOLGOY0K<4OSDm>wi~mMt#D@bHeG8~H0~-ZLMp?r2JQ0Z=4R{7)Ol%i2C7+XXCoRY=*;wOV%8N)(!k<4ET&0WRBK%7r8!8oxNAkLbk3>5Ockqwxh7z5>D{2IT`MfK!C^B4!|ktU`lL!}Ek8=)$#5E!E>@&>mJWv40#0AhU9kzh&|rQZ2WdV<A|ahSHuL#3DNn+@$Zcb7<MRD+h=|RnqbHd}8*ha!hfZNYGcW1UU7zwA3WYhy!$k3E+hBo!a=t~KEt65nq2vNCbzfm4ye|FeRm3^lhf}mX0Z9exs*lFDs#`WEgZ${m&8<uLsDQp}E-I{l8V?nY|1S<U_py;P>Fc~kojZ9VH=s?d=0=AoH^+)eRvG8Oun@yCBmQip<QET-EFT3~DDyt0Ok$O#%_P#zQKSM5lQvm6L{|obLNHCXX))}>Xr(99qDZS`p`XOjeAP+48m5*?rXg2{yLWa?TIaKSPoCa?^l<Uus;87>045P4PuDOHHF6#6BilLiU*2GMXC5Iml0n}2sEqS6rTwQvwtC!aqzk^o)W9cl5)9|!6VMiSKwRk6fftKbc3YGf9f%mk6qHUXgU>0nBEfYfB#;enS@xh>X2hiZnqXtUkRTS-PG}gNNw=4eb4U>Rlwh;+NG9WEHXgW)XQS=yPC+RkK0&sj)OmTO)2dlmN%$t|$TBy)Z8MCHVv4bbp<JA*k8V)8XrzKF?cWYzPmcT{)uc#zj<Dg2_cY`bxp2zMHDMR-b~v~yp~q-*3!}Q>t13uCcD2X!5d!AjNGniPm%~Spt4OzgbeO<kW;i+^xs-<!Y%xC%O2h$~!^IIAD=EeAC^sI3DDc52@B+&eX$AGIsX5qW-ce>}Z(*C&&C6A$+GyThSp<~BsP-9W1x9q%TN0*ROMzL-STvXM?cj=1Sw&MLnWs2~Yu(c2(bXZ9r9cf8ruq!3L6ru#BCNIcwj)+T)-W<MWQ{PJ(<RGWy;ZohEU&<}r~^yzH3i;6(o%&60<9LPOlOG?OqmHFtMRph1ZkqfdJBOXNmVSFy)@I`Ul?_u&KPQ!v^le6dI7qpj;36HYt6LaLY!iTwaSjCv{Dnl&H|iwO!dnu&3Rls^Wo`&OlMu%)rBq~C4swD>D{}XO*lcS%8RH)=0hArNVIl+su>~*vqpNs2s*BZxX#7~f<A}*!;3P+!El#Ea3#Ew{SB)5QPL5|&)tv`WV#r&dt+;B<#@#}44uj--E}@Qv5_t^J=SIKOmggp2V5;jdqNT+Uc;d(S3!r)22b+Z<?#m*%Qds;0Hh;Qy6aJZt4lg@I7aoEa|x)a6CgrVbK9eedR=Xz%BKOzgIS79BmuyGHfyG28x4wm4QUMiDwE?pWl*k^S2j_T=WuZe@dW9Yc8Q-l25^Rw!K@~!22zI7>J^|2y*jSy9uYl^(<ZoMPP7MhF6A^+)=m{oc@X11tJ5p|eU;;P?p9S5J+b?DbG_awUxDhZCWTbEj!s%c?L32h{@~r#5``hwrYPyF<D6>CZDrPbX01UrD**jMH$xq;@kUuwQM==0huu43f)qB6v?5CKx~c#$(9kSh?v1URx5W1ZeIPw~=r|o|Mm0T8`EZwUHd2N9xz%;CAghgaA6(+kou^G*?UGx~eCXy2-H$>B(zO`WM}QV#(fRJaOZ(g>la9ZLN{TA6*6X1NT)@oF`F1m{$G9rFtA@~yuaq4gcV_3|uArQwM5+RgiddTFs?}MruUd6_b)qZh!%1aFG$^!q5+#SO^F3{IOhJRwcb6!E#!(&C^dM_p#0uh#hHUd{)xw{YRPUfOXe0~PBWfOlKg&^wswhBpx5aS~>OL`q@o-tgqZE0*!YK6+r#7oTqJ@C`h}DI$aJ<BWs+mS7B-#%aSy3;CND^*`%dDD-6y|1;Q}Yrx7Gg;}>y9x2I*x(~LF^x#*~+Ej3Z4arp5!jFDlN$NS!(Z~Z(mi>#_LofYU<aZ5(&H5OabyjDqy#f3gDTM_-ZhUMMw*s%Y*j^x-Ec0cN4x`Hj8EG^=TAO!2<HWcDE9&o>mXwt}q<*S~V9lUU5`^>tN`T1wFw+uE5jgie!xJyEJ?qQ&Uk#NyQfJeU0;gf+OfnM8e+gywOcJEmS1I#-i)-O&}Rzc5dA{Q#H*@(bpLVGgetD?M2Bkd`nLqg`r4LCl6{(7Wer0*t72R#PiOqt;$<?bV_H2p(?!zyO&dxxw*f!@q>f+o_lC@qZ)O4K2{gn0&km&jS4ww_Mm$T3pQXeo!NyxYiFMzG7U`Lz0aRLxqau^qbJ>DYb@8v8eJ`eq!_=DKDt8Y$XoAqQtK2xD!sT(g34HKFJzxoLZy*R={cQrYH2<-x#HD{sixHCOLo{&lEx0P%D7AUt4&IGZmY8MSp}jypnX+2(>ii3eriN|mTb_$za{0;KaU!B&ew^j(6llcRs{>_wD&SLioWb|w{9-_<CWUkcHZs6&9^m{b;8}`+83yOTc(pc<W6F!G>Xg=!MlgfUp~J3@kSR<;G^ozM+<F))cnXd=PxZv>JZi%OAWxfM&MHzp<Q1`!*i!0s|R+opVqst<C6nZm#+t1*IbZ;%r4&kJf3VGP0#-mzF&2+fAi%Z|8Vj4*F$ml(SvPqB4<{tCtjX^#b#u3@%GDMm+4rp`B;3Y0z}Cn9Un%yXPQq%&M17T`nrtrZXk&Td%lrEML6<ubzciMCC@9ZHVx_r-KL!qNY|r^rtU2L_7o|RL(Oug`#x)xt(Q;K8dkOFOOK6@o}R8f^D@-hmf%y;(vg6p-KXOL-&-|yG51CT)hwyIidPA~sXJ9lZJB>nVC3M`l`ar5n&ob_Q{?r;fWfC&Wo}Kew~V|JzU74Kjza}$njbIatep}xoHd<nClnL8j<>nVP|J3{y<0OB3NOVO&L*ocBDr(Kavv!9o1zIrx5ou`yPZ;lKq)~3_A(9xMS;vErToYjE5X_?nL<ev)C@SQk4E+&<B0OP=cc3aE2Jz>K74f&<tHEZ+z$tT@oN9WcMp0LSyGzPyT5|CP7JW*P*d=Y?{BeOEob3iWgFQN8Dj*>?W?qD?kB@c7a5%8e1BKmuRG&)=BMs|;XEf9P#9OV>QS4$n8;XWbrzKS=vlkC7`&7an}NvQ%33y5cK8)t74t~aML5n7X3!Ph2;;|H?yqUey-}8=%5^PY@KO&`bpjQQwwYp4`c&5=x7jsYgRbsdR(rZ1Dfc!!{|{uA>E$)d;-iZ<uaV(-5QN*}D7<+4Q!qXYYTriNkO*@=wLP6~b!wUQW+A$k_y^$-Ryr9M_vt0MSQhut7l8h(?A0t!Hq`#mRiQ3--g&+c;R(mDcKF8Ls~wym_jK>C){Z6@x+ORnk3%Z>XZ|e6j>;6_dvN4*o#R`Fo_CfXdYZI!``!W`^^`mGdTX5n!9SfU!y7Jc<I%#9s;`p1=tL7~HJLX3-l02Yxf-%R;Up9nhWTraRwbp%8JIEWLyZyzt|?g=?LGwpDj8;;n7mqD=AB_qRP<U})ibd=$q#IkvLlp@ka<V5RsEW#Z?aD)-+?c@&~}fNV_p)=eF|rlcgG;C&ea90Mc>_Pbf!1ibwvV=D%@qHXl2(XW4qM6Nn}X@PV9-T%gMBBZuTZvRJvN?`r>0$*>w%&Dr8zcbE}DUwYF|uf!ac?dDXJ6P?o;>Dz8wl?jpSm`kISfRoGrdwhezZ!ut4sIgcBFe;S81Dk>_&c;v_jTtc)P@>&ZV<E`%GxJ)mh)B$!aGD((BgSbZ1Nu&jIEfQsX6eq{^a6e00)?_v4)yU<;gkK?WZkI(e<M0iezlpfce(ipnjV{$u=*4(cx2;5=(EieE_X9c85_LP%(PBHKEtPCD6(*HD(QZvwm%L;A2!M)RU9JR=%2}Gb8i?`_d6}Vvo)5!tmxZ#=Z*#`Q?#*8PK6X`W?jWc<vnf$q^vo+Q7Tv6iBEJ@Z0I@FaXg)6?Up47efrFd13PV@U<AMzOuy*yESXos%G6g=>N0e~kO8vNi_k=&(TQ_m~=Saad*!rk2#HtvrM^f8FU!B@COixEO>bDklMO7bUcyv=}UfLbF4BOFy3trFoV0U@hOSR`!)h^ww*B5o&WzeqWbT+4fzhE+I8tUB|{qnjI%ozO|K9Zz~f75zMXBcWRXg%v}sndo!E@y{i#_V82Bh<ZZS$zj=V%2t>Gg}y{?W+%wikaVupfdlL3Tmoewb^%e-a&Lj>1X~SfxR96=?i+PV|V>!xW2bT{c(abGP2lRXB4>0TeNqZlnilhmX`U>rn<LaBpiJ@pp@e27`dUrjStcj`pFASZ=(#iq3ft+4Q}2_Pqt|nHl{M1OabP7aHg}Rr-#rQnGJ5xz$C&u+V2jB!|fE^JXqJ@#!U>>Xy_WY2UVUhJJoM54e(<lqiy=L0h-*DxPe~>DQ9s$xG~NI{%=o$l&5fp{X9LMf_EGIM#li(H;%I)-KIjs80MG14e++bG2Szh+iFR->?NIf6dj*}M3N2O{lN$CpY5pQZJRgK8+(tZ;l-Q(Dec2uasE|^ti;9Jf2h0ub~XXKlkPpZc=Ov*#1r)~4bT6L-tB<FxpmVMcP`%k0xytc=dUpy8PH-HP*zkCC@$XqmQ4SE&h(pq`xK4Q-+X!g1_#bVqBl)0-u%~)ewsk2*EhDskkO^ni#NYk_|#5&rrY@ZSEU%8|2H35>F<jYeZ&`Uem@k4Jg*qzL%g+&M=vkl{!6G^qhQo@9#>&zck|7c03M3y;?3_;Pki%F=f9d`=70P{V4#k$hyo<=1rILX{OgS7_ce-A6UqbsGNjeKj;CT0p1<~XHq$-lV6iM;DO_!((4a<*H1+Sic>3pF>Ng1{)yrk$^&Ksedpp`q?NvoG@xOof&*BuL-K_mKD~W&}fK60TrnYiMlifjrS8)1v*CF9`KtToCP|5fA$X(+-tBbckL$bF3=VRWrzUC290Y94woe}OqR^A7hOe#Wl!hvU2TjI~<tWG7UY0@xE*3h;mafi1b$!0}|UIVOVvPiU@O>X1f#G2`^X6brunX2Rfu##0;aPF2rqQ?uay?(O`n3fvA*@(^GyW&T+;wAbQij<t|_ngP;8$$m!!_gFCoSKb5fz&>BpKYF6=kqgf-BxyILr@)q$`YKHsfrht+CTl_?G5w|G{Y>Seoymz0@&ZPrhJW_vyyi07pY)trHqw0)k3y^*~$0-vt0>G0?l@TvQc`&n@ggZePkdGp_b5S48-3bK7$ddnY4^f^B8?_)<$n^BL{O8yDKg3opqYZZLJhEugW+mlK~R6r}oC&Ie$HbjXD2KYVSpS{?%NZ(4!awe}~tmGdO@apxh{JA_lxhFANdJdpT?;bLbH4*V~^{rcP(_ATPhcCjzf2?c1BL=X(1r(+Jl6J3iP7rqF)~ZSFa(dA=g+{!8&nJS95zx?d6SE3vV$A^7hAcMi0So6Of30H=s0&j51Dc?n!O=4H>m%2ipNVJfBLaSY*_sIW%2Pm00T<BL|H{RJ=ar_9FtcelkC3)*U))8Bs_R#xq0<zHae-;g0kg}B@X7*_fBv2EQ6Hfi8hgca8gnzrKhr+4o^^Pk*%{OHLuZ#G(HD`unRp%Q)==n5wkZdRt>BSKv4`&29UeGcb+b@26dh7xfC(#IoA&m+?jkHZO&xJ#@QcWkfyA44(5`T',
 'checkpoint_snapshot.py': 'c-qxj>vP<;5&y2g0#!edlqjBLx0&P;)}zL9?P<pG#BMtiiRIu(+#Mpm3Xpn^{`c<UL4rJ<Wv8EdA0!f3U>A$s-#&PFc=&+{Ugta`>#itBwq@CMQ}e2o-;<_W6<lt~ukSt(TIGbdQWMCoB)lSZ#mL)kv*Fc-yrCH*SzVUAjSdeF4|v(sq9t1@w*_C>*WYAa+2@kBTl-wggSDtj(!dyy6Epn5ArZ_zwY=<~A9%Co1v@x+eeve}A3uHiNMiEnv5Kw=mU~MOtB@=tUe^>gbq54%>XhS%cV1<jRqemv7;@phq`$m4f0uszbpE%CmuW7&@PNS2V>e9Pr=2JytUf!1UuXv<%jzn}@$XJgqn>X{+m@A0ixV$lbWM5D36=`hjuxH0fJK@~TVJy(Wz`Kz`1Vcu>F2*+y!T)&71@^Gur%iarryIHJRKYy<ZMlDXu)&ZvJ|#k^Np|n1xvM|enIUV9a&w4<mgC`KPj4ZEyy*y$E)>w9S+JfJK>M7NqgTg!{QPaB>~}*RCP;MbzLlppj<NYPg->BLWo-UYwyS2w#Xx_cp_ydft;xokr&L!Flc&4mtQCk@`FvCDe?n3o0afgrF7_cftK_$!r9B)d#ht9l|YhLvZYlvk~bti@C(;)#x@90Vx&bUiMU+zino*FP0tjQ(}5NKe#Zfggn~tg2EaVq`og<BKlzoOJe;0<mqsVa@nZ50XE%cMUX?hf-Vwt!1hb$wh5EK)VgXi6C2OCTDoyY&3X&W-Vhqj$g3JhD)u8AI&~Z^$CAdAQXvrM5c0<?c5^GrKd&@l}F_EqCXXo_hU?iQW%xdGna!XT0Gaq5lXz-S|TZMyqFsd6?`JPyL0ihC^Fj_9{ZU${5+yt0ym~4Gb0_F+3zxeYvUJz~BotX<PW4dN3Q@UYyS*Z151Zde^YmNhRK?FBe0M`482DZHvET>t^@&rUeKh!z}l<n8J`@kj2ZUxtOwc4)J!IF~4Z^Ngnl{eloRLkZ!@L%!ZuZ#*NuH^C6M>IoEzmqTBVcWKie6?IGF8_TcUnR%E)hm2bCJhpvz1p*pQ8x};E&;Sp6HsJUP$`KGr-O}aHKLl+6x}#YeaVV7SgWA71rs5uyVlyS<2OhY^)%{D5$bUZ=RJY45g-V{eSu)SGawL9pi$xNn%&8Qcmp=125}W>gt6P3*<%nQQq)N2Pgp3+n2bg2j>}g1fi{-Mr}5tPN-}Z7a@gK4z~&(nX?@=cx&WxjYJvC~ZHjv3dq-NpAQ*UA_p_la(VR{vnt*$2Q%N<i5$_o~Og?}N|3zgDFi=-K^XGOnaSDhPru*Iruhsx9y*Xrc2Wgf%l5_1Pt@s+8)5@|(w@L?=)*Rgxf}z#I9+u?Nvx8nT(9yCf^lik$u6k7MS$J;n>hZ~07yKc1Sm<$sG|BSw*srM-Xpxdjh3k=X3I%-*D~)}u%D}&>P3(2;`sCXY4vxYfyKB^W<41AnVN6=Wf5;`k3^0xh6~*z8O0^OUtYU$h!Y7>7yW+`l1WZ@(#t(ugS0`wimJ{g0D*4Nft}I~o?2;k4(kx>Q9sz=(AB3m<G)k#l7tip#PfYTHc#mPJ`y7$N0gdvMK0qi;JM4L*4Txpt;p{@aD-};TKAQ%Br?wQbB9a5-@C+bg(N&fWdG}hnx-o)z2)`c6B~*1{gP*oOluHpLk+K*D7w1O4UR)%y7W=hgtAMiCe(gO*G|ziAjb`or9$R7l(*bn1_X=@4xW_p5EY0AFCi*-M9=)_rp$8=xSe6<@(?tG3=z-&)&kTeo&ScMFX`h}3kXm)T$fp?P<TC&<D)Ni~cbxC+;PE(B(y4;6;i!QCUfF+h45RWI{`#6g#-Bt7<(vv7X??9;1J^&@m4eB-xB*Lmj2g1^g0?u**1ZP}6u~p8v}*-e9{2fc*Gir<(ry_MT}6N`=(`s8u3;FNKy|Jn7lS89n1&vE=@5z|D0H12FtXHwiu+-Y2;nl7-FnUMd=KfPwshC%aMOns9ejrAM${WPzElPDHLp}@tT4h619L2EVNNk^$c72M`(<x;NgSVsBaLF^i8N|81Bb`L4pxq!scRv$VzJ&;DXPTITNRDB^km(MjHNanS+8C?foosr#Qb;^yk-}TB4hA0o2o^(0!p=Mq;`@jt7y$Rw6D(Piu<wksX%Bt9nr76d4p2*CJ=hJUWmHKU3gu^J>|O(DU|v@oqxJ`9nzwxZ(;2>f6_jJQS)IJ?%cp4p~1p($2m+;@q!tgH15{udNq~zCB)2YKX@h(qL1|8Pl;JJMbSwCZJDUsQH8J?(YCLPb7lLX2f;17_7sVirp<ju;UdyhMGdtEV5F7vr!ifVmRB8{C3w(Rt-@wVCceAyjunGBm<b{()25c-s6p@*H@)xDKrIhZ3&#9_`N_CEeHOX;hMs9(Om*LL?<+q8I0G4UC&0WWMmwzQDrE`~BBAlI*+psafO-wE165P&TXzV1Y+T*L(PE2o7#RV;(YyHr8>qC~#=X;8A;bB*waZlMyLQ}DE*?iW`LuxhL!1}`@IH^92V;XV>Y%|ij+-T+*255ab@63t#W9iNf4ba`p@SNkK8DI9M)B5ZhlHLNO#4x4+FHsl0LoC?7+ZN?6We=K(2#cmODU&+^gRI=qlGdhFg$t#R)i$38L8IpRyT^*U_t4&>s!2U*XIon$&adY{By$ucuQTK=R@s98xH4Fbh_yf>-NMb8Meh1W`nfaF#jwZsbJbWOpebcAhv_L`1>R2MS`;li0KRyqLwfKlAh^`Wwetz=)Z-dl^qtj65mU5y9IMJmkH;O4t6mHkB))7;gODosUGu9cPt(qrSGwY>IwUvv1dAF-n_Zj{UPGZT}>+hgVscf`^Cz@h8zv&fR!Sth4QU>^;?Xs9qVHId<=fiG5aFV({XYykD1?8T2<xcUD-%q9WP{+#7-@h8RxOeVuGPeNalIzJ4s6bC%R$rxSu_id`TX)8V~z){4jZ1*g>0w!<ZLVU75C9xb|GeXS-p~fCTt7SQ~Z}QHap1!2m+p4@|-cG~FPNCl~4;=zTV180(GW&pV0%hMgT$2SgT(HGu4u|KjM#tO;M922Piwr1vYP+A9XijzUtg+df;&Au~i5&X~D~oqqXzFQ;#MVw~I4Iy#QWta`GlzWrPiC-2y_6?8If75yci2sV7HNObr9_GHpFnqOi+6eMn7z^&9xVEb$4f`A9syhBUP)ZRnG&XkWB)!pK8ClJCeV^J-tO{JaxypimL6JK2>QPD#EbB`0uUz?UPUD52iYpm*>bEz0Jb1dBc>vdoR)pDW&#_ad2<cBZP$Y(5b(ij9T`?LOW$hS(-_4u^OFfz?>w_RyKeP|9O;a=n4>F(OR3~-aU7ZH1k9n^#)l|wM%(tbPC^yVnzs4?y5%v$*_a?Qr+vCL7|VEfyjh^1dqECc0`ENbvGsg~8a&arl}cnYhSj>FjR6vtR)sJ^?zlTR>Tk<JA(SGTVg9lGz~@IGGO2ncAt6EPnVr=av-LN%|yZHyWMZpUfH%!byJne1BvyHVB~Sqn6)9CHaLe!U#zzno0X?W{EGF49<`hl8iNgaAIpc5*wZRh*DK>~-)}2C{1yW2gH6vkRxWp%ohL>ig0RCI6$YF(*GK2B|SLbHH!|#3Zkou6+T4GAeTYVXAL!MF;-{2G21<',
 'corpus_runner_snapshot.py': 'c-rkeU5^~caqs&pTIYwEmG|cEL{d?Xo&YpSOM)npIUXeho5gr;w|94Zvoovtxa0YF{~>t@@)|=33?wiT<dJ&FgAD%<{*zSoclY$n?NNr|hsc0=+aFcc)m7EiRn@b%w|CMOq*+r^uhWVy$fDe&SwXf{c}58>R#`!7k`@cHZO`(oUXw4M>~AYt)9NBCR%BjQ+qNe2HJzVt%dBYXXm4+Clx?<U)sVDWZPTizqh(cYU~Hb#d6ShzO~io5Wm`0~nvexurfuFVvUwvX&+7~MC;unU&Z4%-a@AVOzkXAfh5UPyHfxS=3xB~JiR&eHh`;NrnvZF#9KkXw<v?hww|Ul#M&CR=dG+Os=OiWpU+8|-76q;LFFp%KPoDnj(d%cYub}sOMBu-m1+FCNiWW_9OoFe=-)4E99z2K+N%(D6EXvE8JU=Bzhtc6f0u7(ue@NbZdVfqFZMQl7mY#i;H3tvw{Y`Z5Qxbml#p&12CM3_!Dft|jTaL-&HBd|sj_v^f{2zIhF4HO#Q-TR6qN3lmv~KF8soJ_pXmJ6=oK`Ir{R^6>SI~46O-PV74c%;;8h+n{pG~%*WsB5LKRJZ{`wyZSG@rGL6~)%0&!APJskgA13CKyw=Km_qYl(w5CgqlGHgsOgCI_KtotMB>0H*ODJrIaI|H)YuO&7+Ldq}bFYpSb65&pMk-!jeewxUVX<b2wVAp5L>ErKoFlnaVGhy&>&DU19nnCNb6o1d2jg}yo4o8&Aj(&|b!bEu#=8-P;NoEBNTNtRhvH};;0Apkg`OWMrW-fq=(9h{}`S5jxcMLLfT4}pTqvO1>~GWH05t<o)E1imh?C3;}*^@Bd%ni6n?JOjpA5^(QOvnwx))qI_{jP5yTjZO7MdC}7&C$gp3bW7=cFf5%%!_sN53Dg=8lttRGUFBPa8_I=5?J@{W-EOF$)fI}ow507mqfmBUrdea~`^f4vmhs7CbTb-_z}gWey>+?))vv>Om0sp(2Lbr_m?7SzMYaS*2J<3%BSr$+x)6qu0}{wF5jF;6HZNPM@k>}(vQF!@9M93Q(RnZ?AH@pGRP-fq41Dt7(_nV27+Iwm*v4<tyroa8s;t7`6&p$rGFbk!SkXl!uwQ1)n)uO0<(3v9E#~C{PL4Qeo8|szKt)=Ux&dx%G)lo1So#zvvxV1T4rkGrEX#`EUj$AmJ`b0SEfM_8w&8dt`2Y$RURTo~TLd#UuVRxp@Ef*%93mRVLrZ?r!ig{k2@=!c7_8-*L5$i;!8NzH0*R;=*6vP#kbHqZ_;`U8&nj3hXB;1af9zfJQG5+IjFo$Gc}3Z<c@ce+fx!>W1#ptk3JDUC5U(g91SL5k9~gl`g@|%IA!Z9|@|f{0H0hFSnX&=1Ax18P$I4Xz9v-|tc}6x_y+L~h)&PmQvDwP;3$6+$q{CR|t_c2Gd`-6*th#?Qq;2J$b)cq8jv#=nX1qpjnk*J9MV}>vVH1+h=d`Yqx=EY1j)QL=y-1$Ce(^XEa5xWnFUWliUIl&ARO!5-i-1r#K1uM^7cZWFo~W5)g-&kVWtBD5G6ItHUU`vwICjNDu$H$luVI*1%C+d_mIMPFbGc;;q_B;{_@n{74V{Ov^?A14c28aA$x7TqL=y5daspJKkDKP$AcP<VHx_AtJ>z>~_`VSrIVJ@Y;XF^EzpQBiD`i3&2_XU40EmWS2@^@xc+?dx4#Dh++U)|2QD{q?jXCDiICFSYAp6EX2S9DF0O#fIL=Rgx&DJ~C=m>@}Q8Wt4IXK5Q1l?|7WX+zDw_xEsOVGH)%RN_$4}3bKL`-aptM_sPq=*VOh@URY<S3J4*3eryX6TIIf;C@qf!Hb|m+UB#cg1~JwFaemewNP9(eH){ame&UvxrloP=SC`sf@M~_WAX_|1s0ZwMT+?h>>bE9;=Pxf^Ojt`gT!TY{MM8>$0p%MGH8fmDLslr*hD&kC)&dHgIi7nyW31g9gwYteZ{#5WOh4fHpDriTb8X$+a~=wt{j`XQ(agmg9`!x3pf-0>Ud;Hn^%CtrzY6bHQWZ*38s0_cw-}FBYM#r1lV?^l(gmZZp_7W<u_X4gIqne?G5*0~NNkq8J1iaxJ8%xGlGBp7MCcfRkRAZ8fI}z(Idpxa!jYjB^7QUx<Hzz>K)2HSKl?k}@_pkedMkj`?o4j#7>y7u)L_&Jh?6=K_vn9{0?O#(>uiYQ+Yk2Q9o`?M8bXqCpSBcaj_aEFW+@a<wKaTB<RXml$+8r&q_OdkVk#^#J}r3d50o4P4-z4?R@8XR<IC{$(AZ*r1NeK)0Fp+17*LyEcVb(Uk!}wk-2>#UV25EdSugn(5v!Q)&8FeoVfD*P3R7x0_vH_Q7S6<3P7goxt#zCDr0LIJm|v(YW2<CE~{1WdJvoJf5*V$E^xPa05drutmvyxy<lx2<{pa3!LPTJY&7A$KJykUQi708;3Dvu*C{y_MGBA6$&QK`OEsIVsB?-Q#hvqKuuvl!j^(M4n)PK!C(R_jg=6_8Jav3P93G-#G)}RFUK7o4ldeWDp&5Qi*MF}&ydz=|3HW0Jl&ivQc@k0aKG{{3T&%MzPM`(o4u{F5-e-RFY?!N+_8u;qZW1`9(R}|2Os(mAQ_~?ply;t45vj5c&%T7gl>B#afTo%XyQZ^I;^_RC&8$JKG<~>fkQV$TmOe+B-Lyfj^`L7IA}Jt3{hwXeK5}jt08W+fxshLMds}=ALiJKZlGq#n~Uhe-Nts50l4Tm6Ue?X$X%w8zU(Z=cp0D%tFA=GmKrM7^*}V8)C=ZaDZt8HSl%4h|69=R>})b%ejoOut`EEOYG98yJS6-VTg$*o;tyDf8-4bzLv-E>ONER@P}m1c+keRvDeU~DDAsVPY0J5U6@ges7jTvbmzS4OU+?%<+Av<dJ6ZfXlGO=J!?8T_bt@8Xtpo~frPw^zR$O4%42%V;sIYRJ6SdG(Z9jw^>i3eRTyZ<W{lSSe8@IXGZ3}^KT7ba35fn%O1_j(j%xJz~=DyF}>Ke44?t|UI3sc^aa%XP$!umWfmxdkvun+gWMt;rQ-4mNBKfXveaH+w~K!Ka;TV!%ktDso~DcE4rBg*o>FgRa>t|b>~nJhr@#H49u>jHCIZgE;>V<YV#V*4?qd81bl*e_OJz6c9?iIM&|Fe0N&UNMpgh@&TPo`1_4LJnZUWfIq0$6P$!sa)dMaBG2C{!Uy$h}`N1WYebHWb@?gssUZ)3d5Pp@uMAPiX6iA3v(Yd<e88CI$H(e_|93D9ku3X<nfQco0Cg=Rs&ix|M7nclE44qUo!&f+$%DF|LzZ4+*MZ?f<OINOPcJ*|6s}dk~Hss_!kIWVt9T2{)gYE1h1P&7r=qtB0>$pLCM~~`+h?f@8A8th<uQjdw=|=AHN5v?|=9MxmF&*&B3+lEZiWfhveeNe=Er!|M^e<{r-pV=j2-0m#OJ3%x)fv73pqa72(G2hM|H}wQLO+ut3D*{kwl}NY0utNDCOY7?jPSbDd`7{QbNCZOC^mERR&km=$E(>4yCM>qpPNJpG5{<mt;7C#TV7;azp`mOSB=5O2w;=A~d8G~zG0G00HxJE8R%YdI0RMBaM9?(gpt_W#G$KQLBnCwhs5Rbfc;s`Y}&igetUdvD1#j@F0d4be?q=h5i(IL^GArQ8<nW?P4Rr0|w!7(K2rbDq}oEQ?vjA9nujo+-sYM=?g0K3)TFGPVm}oSwd%2zG7Wzxy5V3-|DqurnAUM~VGU&gEnB>#j-pHQ%P$=oZ}zzJ2=3<mt)Di<4Kdw>ak?iy-<9gy&iQuy(XiYM{*Fafy^UyteSifgmEW%vKZf@q~!<b$ovUxzsla7^s++1PC`4jwBZU5v_T-N+>qRkVzMDBqTg$<texZaTqfU=+A%ru`DzJd>`}n8<pv0Kc{x(f^n-Zm&;(%DP&-dUqVwTA>dMzfJY<CBG{n;(bZruANsmDFUre;n1~~tfyMx5D2a1QMZilExs<cA%!iQJ6UheZSqM}m;k15So$8mn;BzPMgwtQ(u{&Dt{JUT_>PGI2RnOfCW4GpGcZ{96J7KiSNE~DyB<650d`Si9mdZvNn&erav8J+As+uIMydoK6@fZa4)gVxX7CJ;mpG`S1Za#GZaqe~5MBA&tjeZ(abyEKpfEFBMaV&bwr-q_>Em4Ix6BDT6(>P3EQ7eTA+?Wi*WeFb^c1LqaLY(~k^@!{p`HSwY!wc(*I>=!;*;6yI3v3IdTBncI_N{rMAa|K<k3^0OTz&j@Cg|Y2jC=cQmio$-3>HW+L<VK`GArQxmuLa_jD^FD-{coDIdr0`-HHzk)ZZvw<WTt?x&$XTyXZYQIt5f({jq#R)8ZLPenF0gjFCG-(U&T+5D8s@n<2>4RZ0D|TYMc0kXN{SvTz(6_B8}E;mR(mN%2xk8>}qAd%SQZM%^MuR%t5hv5q}OT}nedv&HD4>9S=A40C#hfbcMg0>K><=m8h;{OFTI;6Pej=|oH@YimQhVklDA6WCO^+(GW^5-($1oB<Gk!$=MaF-Cce7OhYyQho55&a-T0jU{SfaKMgw*qDR6Kmeax%#R>(dxx1*yewHrkM~&<hcg3pTVyMkt3i6xZLC*mi)OSIb^cpgF=k3TJ1tBq3k>n$gEmt`>@k~N_4+!4>mIoCe4_}fuZHDFEC%PKvP=~zq5{##glc&+XwjK|W(bcx5S?>j2(YlZibbl1P#kVP@cK+2f~?L@8#b$yl%qk}!C24FLtX!=kBG!74wMHB@8f|ubH|K1J-cO8i0TxxF^vVh<tmRmMxndekHsSZxC0!Bi;E}=eMsP-2a^idB69Njd2;&bbEYaMPanSka|pL^XngtT^b1oWXl`V)8Ku@@jWjkZ1+AOVtpylItGql5gOA0GTgCMmU8VCYyzm|GN-=3l(r#B(sxLl0QB;Gq6UEo=jMtA)e8hof)R@mYBTruZ^2KTL=<}!Wn!rCL;)W<V0E2lno*s5#oSwXXb(%bV{*3{_cX?71w*WdZ3Uh02pbXn2?S=4*@OT!;(7K1U3X)^ckE>=>%+Io@$Tv}ut)o?GB4qV_)*ag3MSk?u6t7M#i!UwPJdtZ~Di2KRJ9l$gA(AYzD(+S}_DH4z;#6VxXdToOvhs3}EtTP7Q6n@22omvT80UoWPfef%m~iS)&??GeQ5%-1Ttqe>(z+g*Shi^ftWS%wfQscD*S`hu=@O%)X~Ddjs%qKW3~+dwOG^0*AW<=;>lyXCpE*m&M#E>5vpu_)>KfFat}O!2C|b`NS%4H?u1kjTqT~dDIu-N0#pI<7nI6dMP7KUwg4Z<PVm8h=6@90nMm=~U8qkb;+O9Z+{nllpCI&2OV^u)DONE{TdTpRHjEUx5EMP@1DTEOr(XA`5ZRK!_ZP3!P?4gCO=l0<ld3__x@VJ`Ni>z$XW?a|2KHzxA5}|R&#OQQ84?h9SY?1Jo=^~MO%tNOft<1PfbAr9PoLj@&<|+8w3xi6peUp76!p5ku%3IV89CfFAL!S0MI+;1z!^u$viUTAm{{oVcwxJ!}k|dTqnhNkUlQsSkPBG+1Rt@{O>xa1Un-5;4m2dqkSfR@A!@=U7ht^SvCkA3vf`QpL<9B!423Ca>GiZM9O4xp8jb|O|g|Z?2T~?{}*lT5*F~HU?7ExcKO5k~gN}|%B;v#LKaZ8!&Nl&nQJla*7Tylgl$MgK4^VZ-1i%IR0aAoQ>${GwGT4=NzEF;fJVAe*Y1LQLUUURv@JcKb&Wetq0B#%hY1!N0SaKj)8RIp!=l7NGo;er4{2y@}xA4|fF)MQLsK!gRN7Hpb$&*S>PJlO&DYdJMv{G5yONl}p}a_}kzA7JRFUh{I`NdRVBUVJ93mRbH9o5q%{ngs%lQ~oPTw*<2h1YZ8+xpa!LQtWaKhLJmt8K&&!DYBS=YnKy5Qln>sahe<+7t^S7%p}I@YjHjt`&gn5js7#^#$ao%dOMP7bDwT9+_Q28hP~qif=>q5JaEd)oprogq|r#hUCWI)maYTK>I(!(J%?v?*r&asz!wjarC&+2&M@Dn2J=!9C3IM9qNEx+K|DkCdtwjuMU1$o3hYjL>d;u&gPogW_TH1rG?wCtKX1HgM#$smZM}~1X{K!;O;!pu)Fj$_qkU55;`_Bj?U5wN_K%cb9q;7gx=*j<>8QZSS4cuD?SbIi#Cj|@M=mH+w$)Hp>gCl5rs)89EHN5a6R3TM=eN?Bqw%u6Xoj(7RxM`4nQo9ZinT=PV!=?^F?P8*{%$YPQy7NP75`eT`N~SZjalADkCV?O(En0$H|8;IdyMngquxR>Dup@e0&~cLy92h!?OOu*cd^d-9=pcDxgb6TTPJQ|2ijL%v{ed)6JYbeVo~!x#$U5>C!gpSsvLp?8{N#j;Z6#kM&fSy<r64X727xm-NidfW8cA84GBa!3cQM5W^-!GgK>}9q*oX%PA@1H>LkkM#JF8tZpMa2!aj7fFI=_`T^74BzU~wez++12+$q$(zvm94Q}^QE$*xtSZj#>(EbI!0-2-sLW5-096$9$8iNW+(|2cLJd1;9A45Ut^<N6&F(YP;J+7a?jAk<Ad_C$wVZ@Ay_opnGP!xact6zVMQqfQ>GLmZb9T^4G~iD|N3j1$j2u778B7%uij=I}%?s~Y0IDv5p!_SZEu0Q;jGhEzM;>W6$cG4$V4nCoZnj`@DN)43m8d(?qB5O;JXYxr8#IXE#h^`UnuKM`Z=*6$SM;7t<iak_G9KW&5qaG7VTwG*ZB36|%JB(t4u4za#jqF~xzGPwgdGuhms(8@&5+!LRb$V0pt&pGtrj3~0n(4In@U6}4eI8m6nZNQgrorA|AWjDjqgMjZd>@Nlb6K6G{m(fE}Y{9SC;olwgephp}v)LK-9h5r+*ag$YioA?<Z%0ZRBpb}zOWns;dGR+NOgb+~doKTm=5vHSrP2DiQ*W;0X5dz%?E=q$2_kor5-EK@$ztqg9qyv_emrI^N(X>dr~=qzwMN`u_~FOGzAL0=@vjU4y#52P$C=pS5HwDfKbM60T^pXpgzvW@heaauWqbBDm{3(&i^(B$?S(7;5dLO*P5NF53oD1$!6<M&O2(Qy9LA1iy6q_WsTo6)NZ0B0_$1o-?f4|7;U0Co8xup{0iZQkWh8e{U5CZGyjc7@=@aCRV%(+>x4NmWPHQzL{EJ&!2l)rMdJyTu$Y9&#ui@C44EwwIaf!Cz;V1t!%!|_WBIBMS)=`&6rcQVK_KsmG9Ko6OVlZCih*lJ7fBfEz;cKNIjNx|#c<1Q-aN5X2hiQD=US)rLzWb~Xrz?D(a6^uTX)H_F`L3A3PxfTp3MEZgRhwCWCImRRHHYPXfJ7DdLl74$L=*AGt}W_x2|?4v0)tC|=V|htc_>pT&?{En;hi(fQ!XSV;_NEPOvb@Yu*179$s$*i=<h`hki5%P#Yp<x{bZi);E?*A!R}XdOgEKfgzS~}d6oQoH=U>JJAEC8?}19%jqmay9<0kWRW}LX11^4@z6)6GJ~+$wtpBcwfuG<)JA(E9uJm8lm4+}sNU(GNYkU{N9SV_tN<q>N5orAqK-B#44+yyhXg&3EJ^pSy<nNA$!lc7d&<`PTAfjfJ>vV}ywXOVvO7dIK(Q|x?yG`d5ZE%bQuid%&dlq;8Hj@R*D@bg|1`;v-Sq)$h1o>@C(>VbI?RPL%-p7Y#{TF{w%qqS3QqWJq6rA;4><C*E{H!&vI?F|h*PK4mFO<6T#hV}o0bzysf8y=KRV}*fDT2df9RIE8Va3F--}DVoAqeOHSWhg)EWSv*c7^H_s1-lqm83kXRxM_vUb3EWLF+ku;1!n?C-8D{E=#HTOiU*xshEN#bHGjZ0v}$NLe{E|r@WY0Dv)>9{yu*y!;3K?xTbfiXFNBAcbXzUc|2jyT`)VQKDia&ADju7seQOi88ZimgK6DorqzoIU!V@nmdUJrp3D*|4kOqtfX`w7;4G;`w?5$#%G+1r2q{kX$$?T=f8#l96l2Pd3*iUqd){7Qp8-WYqrOVSbCwP;-$P$qQ^a5Uc8cZM>ev9j`}m0tc~fW2^1429Z^5irW*Ix_L)P0VPS!^y>o?BJ(Fp8wlAuB+2|F~BguUlYM3w>7wr&=%3<efy1(VeM2($R&G%Dlh{{W_1m_`',
 'crawler_snapshot.py': 'c-rNC>ysSUb>H{>D_USFJp<0p&Mp9kTw*9eAY`FLf;0#!aV^&4*_oc5=FUtHyJxVwTI>f|B|j7^WyOz@6T3`Wam6KBrfe&YMLN+3u=szVN+79E`4{p#k9+&xe$4JdRN|_zq(x8nz30BpJ@0$&rAwE-7Jn&DUcG%c7^MBf$$B!I90&7wk}TT6og{dB8KuX;q`wzUvQ{unhWzg$?Z>ltwjU&E5YL9u5uV$ZE?rt1rO7nt^+wCZGL3q@AfC>XbP@Dtvt-d<#K|mMTQl#{{dqslBB?V-CX;BuB}}E8$#S-c(pE5xM*ZbvF^mTbQ+PI5rfD==w526m=iWGt`oq_gWOC~$8Y~wHs`s<rAeqi5(IOh!`itmjF^Ts~J$*Cn&-(jODy4?~MYM>gktt(-WBB;{^<<`9kEZ>2(q1m&iR}gtjLP>GNzc?X1K<C8JU7qdel|uo%uo3TFWS>+(MQF;{W9+lhZtyPUc8khGxI#{FUHcBIi8$hy-pwWIqlDe$<+R`-O0w(b^9wB97GHIixw?g!~@%Gc5ELPJl9xx`@$LIauyGOEIb76*)pBbxOiqv_weqmB%ZbCA)8O)g;z99_L2pcPfdlp^y<s;Bzm1+TU?qS_tP{v#8P7h3NPEV_{&Aa9|y&cfDaU3@W;3F$-%p9k&iU(kK#!mn5KhSOopaQ`_ZB|nhe+0zIN;G8@FG(6Lf>{!RBVT_T~+GG`s6=ltuk?FuqBfEJ`;zZ4|lt+LvCt*Sqm+x9;3qQ8rw=clXUV?)7fn`5H>yOPA5w+8Xvmuv`op&F!@S|D_Q+H?t_(o+XD3gQY;tL9-oa$tX#IDUGI9AIAF_WP_H-hoVT?&c^+%YtKo6_BcAy3Z_1$B<fMw24S6oIE)u#Mmu@gPUg|95vF@#lcsnig(})&VA{Z_6@=1uSk-QSkxb)3Z|`^!W!y?DaK=C{Fim+E>|0uEA{o<zVVpMPN7kkJYz3fT*`k*m@K33-h^BJ@;+iz7-mE{37-XBCf@gzK*gmnFpiPJn4*|2$JZ&$g^Dqa~K1^fK0_idLHUE@G^GSaYX<uc8Qk-_UjxX7JONUo6cq-Q(E~oRXfpQqjEF<FE&jxYa{c?YjVSGfSXN&GuvmMQ7UpB(!VzmBz*fdQ}lKwDn(Z>gFfC^*{XJu*JJtCS728k)?7J*X5iS`nkEvp|x7`BNLG@P-dRU>I)+Qa*^FaRzG#Crs8-KfS1@CM|>A5X)O8~)U$LcF2?83N-b$zhaQY@5-hnZ#d@8Z=e~M32A?hsd?_a^amX-waz)MnRfI4)3Sve2a(Ry*2M#=xNl>mV1pf+_}F9?k_u?erNYtDi|pAraLHG`u^<XO6ybr+QwvdkJ{NHjpvOfPu~$uAHiwMWN-(XO;bk4KSwXIRT^N|X8n@o!t5}5+s1F{N<0c0^l>8y!Aav0c5Rzh6l3KLmKk!z{qD!$1Yhe<meDPc;<ORIwp@Tv2mJ}5)-jQWpdT1OFJgpCVjVoGgK?7JM?aWFhr#+f8g1+3WyvxfL|*^ol@fvx?diAMuM6ELUea*CV+iw6I~$C_)qBDs2sg6ebKPJkjE6Lzz?yKEdI6yZ^r5{to<|u7Y%hio0r8<xgfNPu$*@7Sn_PJ1EZjPpLlBC<U<6ta4<SDd16meHTxgtxrR8*%nIXr+Oa>)evcG4?)G|f8-1pt4{9{InitX8QGJ$*qQHw@2AGG-yq!?^}Hdr)JrrCxFH9rR0U=k*v=k#vX5MGV5Or^u&61X)0TQUoC`(@j2Zi%|Cl3qhzkKz=xxaQ+C!JBtq3Hr;$I02D74&p4xmZMQTh(X^Nt><Y1W`gEl3?5)Z5=#t5802xV7vVd8M2o?gD7LhQDtmprq1}Lm1LaM7`~t)&pFLi4XV6rQmI9Tya9fHtI#}TZlQoHnW98*_#tCC$%nbyq8FoG(v}j?Q2Ai;prvwiB_&3Lk#XS4M_QuA}Z*K6$&k33PUupVB;q4iZ+)O!UY&$qX?a!sBI`!s1NQ98&iIk-iLk2d`)RpGW)~=<5ZVg$N9jUk*JlhR6i-XLT(}uf<?FZC3Z2o3&J=pAYI^{vQ#mle(Q#LZlYw!}oFG0ox=4H^#D*Uwb(<ikcpehC@p9yZgz3flci79{*O_BktEh5*X5p9b!-dplJ!T~Cm$=4A`5pg5ItGAcXcrdY~wX{iSJcy1nY|9i|TYKMZ1s!Sia2!)VFfOj<O_&{(k9@jv_#am(VH&@+Q+d6(ZcI_m+n)K)hiR85#a8dCe$l?m|0v;&qM>0@5Z50hSY?`$T%T*uK2gFQMYA$3ShYKw#Tgku0Bf0T2`#I!Vr#bieY@#>ENruj?-CkJ{T`V^KV=4QCdCx>N=@QvOpMl>_2M!UIVNzla$KRjYV^6?pF<EIHcp8CZ`-M2vbG&C849HZ{z?z0{)nLWS6~Dy|1=bkQ$hgze;>@p8GW-{XQb6?du)2W!{v4X|04YsY?7_r4RpsfQeh4>^CUIeskgYFi<CT{tpFRgJLymNhW#Mj4pQZy0)5d>7eu)eQV}5e_ZN$laUT7Nhdurd1i1tC3y`V`By;oKicVaMI2ykuX4~Kv%$rpt?F|SG@d!LF0VP^rkB935jrO_z1f)iEG?*;2_(9aP!ow)A?Ggu~mhy(OXd=Q$oqPdSi!BKb*-+u=rEb^~h#0tec|DZLYVLa5EN`Z`S|^t;r|#0SAfFmGe44M7>EP=1ayE-J#j{&O>8cp&%Pfthz!ertPaH_6O!P^GRJm<^oq5oS1%KLhxUS&~Co;C}pi>~{La0yu3hm{b1eu1Rju@TJ&8{;S(xIji($A~uikhZ^y8vOaeL|}yFojFyYN;vASkM!ZODdtxiS2txQ-kFL<u(Q_2%OWN%ouHiDK9U?PCR(Brhe`p>F55D|2c2NikKh`)K~{qnJwd)*2ywkeyc(TzR80FNQ4j)P<nnzS2}5#DJqoVWfEYgb<^c5MczaWu(yCbz5ase8X0y7{V`uzL~p_yw|{M!F7LtWdbDBeM6|PDxE#g%RAP{%^JPZwm_yMkHq(aA2xYzS2~v!1VQlIwnLLQV^jda7)%vpS*mzV)a)Hq_K#r!CuvBJ)gq0+P`9}~&BwWeNSrY7a(9ljmQ$od^Wom2aX3r8a{+7(C-;gx$%Ss*mN+^S3fg*CT|CM8{+6vC>tU_uR0Wc2*%FilnFI~<I?awI>^y~+5vdlcq)s}BMb~C&pdc3piNk3%%v^2mcXvu`>w`pL5`T=agX2J95(g~^xu{`CmUaC@B0HZ>O&y|81v_%*uJvz#bpyk}Q1SriSbSWksAgXM8Ln6KTC@Q!4%D>K`359prWN`cC4N@AXamKn;JFMvgnebq%>>u{XC`A@aFwd<{C%E4AN|0hkXu9C-)*;dBN|jOrNdCN=(by(-OLg`VSU$S!BWMpJGF_9=zkwRUeZ^?jtdy$MPD&t{zq!pRmhV;zTXi!>BcL5oJ9i=-571-Sk<$J^T5s8=*{=d~0$Dx6-TeJ1o`DVsH!shd%(b#SfIq7U5fvB%xM;cTXwu*Bk?Mv@5F)r{xfEy&78e7N&3}%ik{TDZDtk%-dtRV{%#Vro&?2`K>RBTeD$panvP-nwAel$aVYy3_0nmr}N|(7x8E~0RIn;r+J=wp^%2oA?dSdhocQfZ@tC}FfR^o@2!Y*gZq$&p%>hOi%%MYcwa~R1Mt{M5HjuF%e3Fx7=O?J&wyc&VqjCBpy)&`S)mIdsak-c*3Xb{cGAfSYitfoCMuz1nyK^2;eSRMwkAW!B3zQQg{!k&l?b0HTX0hk~1kIOmfmTe19AP{Mkl6jB5OlGYpt?dIwLmim(kGq>~$VlKz@IL@oue5^8WS@uXkoD|aVo$9yvCv}J*E39Nk9<r3+}*rJAoZiD3Hs$q2VZL8z<8Y=_x6@U;C%Omjss+bVUqEGv6wK7;*9Zd_=(4)&!+o^i_#M6sPhr~fC>XgP5Vfkq3~68F%f49A6-?02wtNKe$z-ru~=Ly`{3EHstI(l%5({X)gGC|=SdbXARQS4vP-kw)<kq<<+=cj@V}y4txDdJ{})yx2W`>H59fvF;!_UHeS3aLIl8%!^0()O<e+B`$a#6{31?7ETLBvyRA((L0@<~uS%tmWu@!T5SgqLDqvqoB=LQy?C|1&6G;J}84}gDSMD##Q@;B)bq?^vU%*rPi#;HX#fyOeD!MHk1(^UehHEJ-SYEi3Ud5h(Jab}+^QdPyvCH*$`mwlF4bFc0zh=*ttdIF>HL8fibAopN{_&sq8$Sc}qr!D9Im1BbmAx@9EY^lZm#OoKoAt#-o_S9w%xTe%TjrPe@eavPN2d~&tosYFZWsECP>OzxRsI9ia7+jndr}+$G23R1s?lh;`17t{6gYv<!g2q+C0@mkc+FF85r4o_RenZ=7o>8a0NP(xlX!d|&CQSUqS(U~Z9;{hL$~p<(gpRs?V;?ddcKMq((CZrm;`XdvW9c`>i|M2V;fE~{0I@M2j$VAb+iAbhy1a3jpRR?c&0^R?xow%sFLK1q0D9#(8QRG;eyoP?0g@;qF-zzEY<KRZgg7H3<vM{VFk~oT4^!C8U71|k^@r;#2v}+1)f_U%VN${%rW%fL6vpCfVOhVbZyZEpdd5OW@I0h|t5>fyF9I5)L1m(aL)yXw)oJdT;0BiHI9MdcCq`r|dO)#lMl(RT8@&+sV8Cm$0cr99`41u5&f!2`sBg!Rb)wo#&^c&|ltT*Aif5ywQJBk?H(XS0V1VTn>UPbdh=B2Yr&;4eG~r<Fc;3^;26boW>h4Zwm%@E|`;%m^KdGba?nEwUS$`B6IAN=Z3C=#(jo3?w=Lo4`D?D5BjeIG~cv%AFNC=b{L+maXjG}JA^ue3K!}0tRv6!w2vJ!tOQSV*`g=w~!9E(^Us3*>^OeB#+YD2Rvd9)N9&^??qJv3sTKY3KE-5<QY1im-~D4OV^c;C&?$#B`p#~!Tnq_n3AY9xps0Tn$H(XlFXAU6V}6bhmHU0I1hcpG%haUT>9^x&`$W(=2{V^*e~IokoW-EWerV5YRME@NYh2h9^3Ix0F}#7=h#t?`d`8@KUNchyXs2UjmP7kkHM3I>pyhRNZqV8GxmWRobGH>iV!kHvYiz8~Vnh0a%K#<Ep%>Bl_L{(rs&(^9n=r(y<c%r+Ey%5k{fv~mwCE3m1{)Tx&HKBJkhuoKnH39R=~gRK85!R8&@2)4v4XC>!C^$KH#6>p3{(qs%~!eO>#0SZT8ME$86ZOG|=XzfLri93VW$4o%=qz%T)84;BQ`~VZS&tl6|mK<>a6H2Qml7WhATkU0y=dNA5@?5iib)=DJB^-xgHEYX~Hp<0cTIFx8U-txDk=$71$s2{QQV7jr+@HY-M$k@!+)T+E4))1#b!Q42$6ze_cl8?YjanK~s_M{_=0(R<#^?HO%i<a<*NO>WMFCg!{#UjU0~OgVqScJjn5a|)ZW&9o#ZM{FqkzX&E{|jDEC(Ed-@XZ7)oqlVi8A<d%+^+|tzaqa!o+&YoL2Y7P-|sMuAPVfAvEdKu7Fz8NTR&D^+GGS*156*JA}G8BQ;t88~g8)0=a%;gxyKg+*yGnmF!~0sLW5**HtmEhaa(_(b36Qw{~WCR(7fK2!nR9P0yI*>AYcO0{*<B^UeSo`3PFUJs8UL@ih<{%)}x8Y@XeBQbDS5*%4rNNO;+;;Ma&GHlwX!Ridd??Ta-(b`8FaNtL8f)eVczRRUJ4s+TJ^KtA1-onV19h4~vh)3+7C$|5!gV;oN7{dnd^n9x_!tjHt6A1U&BLn^fwM~iUH6Np$3trdwZN$lFv&(>OX@-dMn1|vU5(aFhx@I*Rux_x4?t~Z0_>@+kn;LYaF_U3cBy9WLP#J+P#@JT4Op}ot4jU=+e#T#z6=!!ReAwRJ`R+|b#LmE+%M1WAOrxGck;R!rkilf1AmM?O9&pd%C0jnI*CP1G>buwm0P|O(#v93KqAfYHj@)T5Uo8=6S%-U>K8=RjFHp`0U#rTz(XJb6!0d&K=V*lcPmZd};XYT-!Ufs~eg5D}o(j7G)h$*>I!SxQ-s=BvRSdiV^;J;Sagg>#~VCTua5$0p?V!9mbgN~n(A5raecq&pHgkYvNLO7{g<SwST2;$hjeF8E_gzfAH<H{*GW9}pbK3jp2u+@2v6m^0o+Gp4&x=FOP9H%MbLY^3>rJsoIm~+I|oC}&gbXwDdSrxl&c8!$9%~vy&y=b4~G0)V&&g6r2Vvp<+*Vti4zuDQk8eHaoT4(a8GygM6Nrk=Kv6xsPx3)nYfNLu~LpC!aS-reL3t+ah6ZFtEljeH&yPni^Q@*SdcPFtaY@TzftZb{`x+e$Q!2@xkz=c9cYZ_9Y2~ny86NHQJPrD=y6Rfbl&`z7+@6b@}^}Z1UX^Y~id3w%a5mspoM%N9~Xs!_Z4~-L3iT?&(!C1Rtn4yr5ob6FCPk3Uu(`m1NqMP%ym!4odp0@YD9@e(!9m+BuRtkimpac#_vB4-3;xQ#G?RJ`H;Ex+Z!R$ZfU=|AFgV*S&PGnS3(fZa&|HixR^C7G79%>79AP-#Ybaq<`6To`xG4HaEmI}bVPG-(tuen+atJdc0#J|EcKx&&)&_V`vd4|A+U8dSg5V{J5m<b#On6IRoX1IlIX@hIJmS%Z7RXbFO%~`uB9xE8<6-W5!YIjSke;#%1N(U0kmChFZxk7*bE&X|3La{sw^NQJ=D-?yCJ?cs0oQNx|ogB{Y3hlotysci;s1KK~(dT{Hj+A*LG%4$sqX2}z(Xdh=!Ypc_noed-NSbrJkk*Yu%<^-SOy~GaVpzLa;dOF(h34!aWNpOD+Cvb0;PLwB5QxQCRa0u!UNQ23BU8~%vCbZD{UXMa(&T>S-m9+!<TnQj1c?dpV4bE}a}<F@a&)3_{Jdd>u2uEq$a^C{Xi0oLtEGE(0;mEHogOOit6=BoT!qtYv(w!E)eT{%VvCgffen1M*4|RPz9WgMJP}Z*;l;KNV8&{nR9X5f_wKzOSZK|Q>`zDV_XY-wJ*-KL;@1j$?YU0p|K@OWWIq3R#l+$cPBhE1Zd0S*RH`-?)`e=6v-5I$qZQ+$iJ9}GWbmQoOj%Ro&g<0Y>QEa7(y(&0npSP^V5xdTpf}G(p++H2hhbaLRwO}W!yB&%(6bpaF~deJ%wl(&=ah(vbi1L=#w%Z?qwTD3?w(1(3OHXDx2mbhs?}t-eZKbOFE3leYe-Ld2bA|yr7>YOW2!M)tepYDO2fH?TIJ}^yX%;9y4^}E&h|NWTEU@i<)KwhbuHvC_g;T+kc{Xw{Z?@Ka?mh+TZbhH<|I80iVO*qb%eai?X7FgI;XP&`z2k^Pvv2Dv6~hW_ED@)q>Fn~EyNpUxFUmPj+Zhm8@R#?*`xejW%@ybsn8YZbxAL%bz&D41&d8x`vkSfS4_IKN{O0osrh<sVbF)2syB`o>NqV%K3V&wc1<mSPJ{_dBvLGoFmc14Y9pT1P)+RhWUetar#L{5AG2ryb>3mLCQ4$35h-^j`_UtQlwZkoZY=LTMevH5)W+9J6F1Dw?S7zWwFB{bMff0Z8)M#<$j{lRSw&l<#Tvhbz}yi)yLlK*?m5tbuT{&8r4<gkRe-BFu@sjGEpKzO#k}SSOM$SN38<lKa~s=Kts-#<sdc=R!$yA@N>v~E>CAS8AIObXt)?WVgR%!vzRK7@oxgGsfw%(2NYXPiTrJ}{M?M^5k}_O6S<<MpV!_R%Nn>&pYOZ}Y2<G^LIQu{jAtzh1D_Z#$E)Yiho~E%j3{|L@)1G-;D<b+C!Cxx2SW3>u4<nFAB83*iZB=U<^k+Rb2R58UV^wKH>h-EP<8T<(tvpgIPROopit6_GrP~<qwT^Rao%g8LG1hf@>s4}x!&QhB(1WQ`kaSF6XuqnDluPTnu@Td$Tw+OD$F7iNMKXC#lp`H+a|2C015^HXEZHIoCp3Gx$L0CL0#v#v%uq}Kd5+wMJ5~y2s!QdoSugWAxeL6=Y#({)VAg2`nJ+~jS9Zssx`)5&hHa80OT5|*f)$1>Zw%M2F=Wwd>=>G|TeZpR&9~*s^0-!XmDT!CS2#~E?mb)OC^KC>uZOJG;q_F`E_HPokC&ijb8UsC+f^cW%V*k((IDs0vsYUGjeWgBtGZIusvL|gDpU1ZR6Uslej*0`pl9f@ug@WPk@JHd4c6*Be=4=J3Z)&W3y7*QK^M($Vx^*(rw;wu;^-&(b5*owGwkc<8TB*4D+y#|@pjp883?0@0W^vd1=aVFAr_u7J|rqm>*cI{9x?6d^-p*F7ZAk|Wcb|0qFyNkitX=Ek*spa0e3bq%xZ<4H&^sf)dZWr!lf@Ol}ssCd2B0OLwTet^R3U!tNY7)aw}Z2;8y$Wn(b$cbz~!sM`HKQeYMTglCJZa`<Lul@uzbu@jT}lnTSYu{9%#cr?f$?W+9~8I*bZVoJ%26>AdL~Sorb6*{WhqHL8QRKxlKvCR5G*IRVR_nY@@t0sU1VwK-(No+g^vJwA%DNAhz&jU!2zMk)cqh+$KR2a{+W*~q4SkdAS*OgZN;CzMu9!V$Dwpkiozq%P!ROvCsAqd?4Onl}0G7XQ7%f3Nc2Yy9^)`nymc{=j(m86stHiNe!d{CfaP<O25Fp)2w|6HJb#l!9?e<IZrj?ExLSh2S({GaV50#o#r{NEp$)4`XVh?^Gb#NkY1gTQYF2)#nJL_S=9ei2|z*?B(Y=`uocMOr>TiI~m1wp&0M3Y=BIk3!aO`@q{;6JW8Py)6*<*K0n}HN2a`Rbr*8*x=PsY+$|W>_R_FemloFSgggPNod_BPre{?xTd29_$JfV{qj|<a{Q9!2T%bfM_-&hx*jJoI@!HKxR?qE%$r7azSvlt5VGuTJXYcIkTFXadwxXLA*2zk%QP%>*X2^wIa|}ynZ@%0!M;JNUw}B%OQNFA1lCP9v)(ovt><u%?xS)iug?y|eDG-EAq#uNzQeq=Y)I&SpZfXTU+nAbaAI&+`Wn<&=&KKih_mZLfh!R;`+Rc-d(SZk3m$RH~mU2pyMPkj5i83{CHqNxvwoNVIla1DS9L1JQjMNzosY^W!bdn7lKx=fYgDIt|=`uB(-51Pw2(p1(pLg~jy>lFV_}ve`IorRVUEaq38~2ANo2{+W4d>shr_II}K6iiE^b!DTR~hh`9d%WMV$yIBjyZ>MGpHR3nF&8X-1z)-<MY3P%(aXbej@QRqSD>geRR|H=n?lR^N_pI6k=q+XXDc;pJ8Amq9k_U5Qi~|R+(nabOI)kEt+jK^vq<{Ae(0WpC8*?{@_yd+eJk_L?y5Ny)@$Uu|qzA#EI*)0<Y~b$_Chu{TXFODxZ8lCm}C!XpT^dMD1TRq%O-P{6Ws_Qf6t<yqm5~L-rf6k+#mdBn;{GkoF{6rz~mvn8J14T^yjHO_eFHl5MlqnUjaS1a7^8;hS%YK<f;`ZZ+8u8{?}x)H1{$%B?#?cN&+1ORbztt!Hz}i2X%|4Fe#)@?Qpv=iP!p4L8q0ZEO9fY>epvHMa9b9C#lU+stQ}axpZx3#wbqjvJN+aF^ZfZxU-mKIfz^Jh}I7-w9s1{pj0wf?t30#+~p2!Ha~IGA)FbJng>aOuFeW56{yuMG`e4y6A(EXAXe*8B=)9FVk?P6EsCeK;Bg-Alt9mcKM7JH#Y+$1J2JQM=#B0XO`lc9}Qp0_qmqb`1OBtBha;DJB72y(Sta31<Z49a#cM2Q^KSi(0k_Cn_HcJc0gf{#JLX-58KWgXTP$Mf-fqmVO>aDm_b##v>TQzv$n3kEUQ319S-jyYyJUMEo_?pEISmI_?fjc%cyFvqA}%6c63GVRK=m(=^%=16b%c6j<*+v!zE};JGMAW?H=XUk?+>lRa8#}=WPuwK%yBQcI#I3>`^gUS%EZIu&-=-NTX0rBo=JjmF76NzYv-o(*+#4ce{)Ljk0<kaxWxVuFRBli)PiVm%L&6w5jnkVFe`3_y)rll3~bJ>Al^jP2YRKN|+||GTrMz?@dj<E%UOfi&{^|V$RL5;O7i)MNJM#HLX#WYaSH|OttIVtREnCU2=#e3~(oWMt3+o3vIj6XvA%QJ0PQCnam7k2QVYMWX5^pkQodju16P$9QV*a4_#Ix*m#q_do}D5wgNjsk{qN*R!Xx=ilJ-6VQC|1e~AzJ_iON#DRT0K)P8gcZl!3Ws2GW$Qtm#q`=_K$nk%g=H5iFmTCAc<b$i%b7_IN=6u)e52Nm75>sr;|c=MNPuU|hsh78c`m=rM;RO9>;XR*yeF+v4{UqxT4#m5~o@w9b0UkdUc12zBT-^QEz-1jk*1|$o2UM5tb6$k*20X>y&aB=;h){6JkdV)zhZp3@a-<u|`U?1qv1Rh;p!V>FS2lJW$Y&yE{X%2JI8GNSfl7dF~$$$ChC*S(VkH7ubzxc)<{qKMI&$OQU>Q8_7lOMiAubR!OILo3WyH*Hqw~gxfN}^bmVX2G-MHWo|w0>1w)FH=}tjw~p^D`^^T+c1zDek;u#%5fV`Ko8>fHNQWCkyXA{jNFESJqOj1m<iFAk=g}Y$}5+8wy~Ln$j8Edez}gzW?X-S8u<(o+}WyX8YKM<DkEn@pQDTrTO*c-m54u>pzJ1B~w>Q&K&X<=SC^LJdA8JxG~1u#In)Y*<61Cd6n?za&zMgVXK0}S%pF}X)VSzW)NdsHmRc<oM0(0EY}f`aXXjxG-IEa3dLcPBXCIqxAR4U)xC5=|D0ah-T0zW^PQy<_-)YL0TN}wELajZ$&=RLHIlMTi_-erLunb&e9FhP?V!RzEcZS3=u2)><8q(VTh;bqteH1l-+9Sg#hQ2}T-kx%tn;hwvN&|ILM!lPji+X&V@a~HZcPX7luJh{r`!xMpK~P}E6K=;K^JGxY=jkp<o>!TRI;<0%4eAsO+k<Dn#z`6YLCr_dT_%eu3mI}2+f6lscLOBemV$2lmOQp5r&=SS6rcnmoRT(lemn+mtVWRO~?_Sx<C<=zo73s;SKF5_Pgoxu4i38v3x*#v071M@I|G}SECHs)8-syiVgIc%}Y=<IWYGK!!PIV4duqH>WcC$tF5b@;+?$BTexLk(u!>Jf|<Hr`5wFh;-lT>*!lwGIwohv?<C!~Q&L@S?R=UWlS!tM9)a4W8Ls%o%gP_0arX)7b>`Z{**-e7?7<hk1fNn`R=UoWuUKWVg|aA;_$zmN_AG#qGGf4Ur(u!w$cKM%nJ(Q+&}%vwu-#M_&pVr|pC@_5J*{o&Q|97-em7<gS8|q9bf)c0HbY4G7*_z#kDWXpxZeb|{Xw<qE$1N@c*YOkFj)@`XQcn6LuDB9UZ4=X2K{l80On)i_2D!s{M+jr2PY%lafF+H&3ec`-li=<GuMh{2QXqh^+Ia(e+|p1UHSiU5&c^T+(;Hzdc-92=~qV+*$Y4a`paKlSB0sN&{eb|2Uz(atG!luis+k>e}q6^%3ZjcYfd^yv&~^PA+^F`Zg}vuk~vA|#OZRTLa+F~S!T3!XOcuriD*M#`5LR?Xtzqg$+;GF%3fi@a=l7bqBWk)&-SCNSrY?P;A=7k%*as7GwKN#3hqG&)NyaF+K92+&E<X#85)m^%UjD@DCd!N!&I%3grJy%%UX~YU}t;FJG`vr%<<*mtP0y}D|4#C&%@VQ^}LhzK3bYv2VZAxg#9ZxlRfEEU>$S12sfYFYq$|d0_%pvt<aqul-mtT5dN72zxJiAE1O%g`oN=`$7zgfv<Kp*QBZoTEZ;5-#YW%4+uX(fckW-hAMSpkv2*`$efQaBSlmCTdD|1@wEDIi`1};D$i!|wcjfu3yY@gj-}%b{Df>XaC~Ugzn+u20kOH^5L>@jivfW{+*$S5wjBM(&L3V$SCXU+L-#egxj@IeHSzV_+l-Yc~bEP8`l@J7mlljsds8{ul#(*;dspjU)&)7PRZ`Y?|ErB)DV(N0lPKpx@o-J3xY2TDfiC^X#npy3YBso~ltMaOq`f5)=l5xm-%g*B=TPq$>EvnJR=D)$3^t1Qd7-zLET~-MT+_~)&dJfJnZ>!k30a`{d8w30@85l{2dft98GsCI-T<%Xv@GJMIq7Q5oW9hg(8|=mH*<{+D#n6P42To`dJ2KfTt6YWfp&nb*+e!y3e4`0f<EuyI>flC>R?Wg`j;h3bTsvZ&QE~-Tp-<knXElqh54f_pz1l}9UC-Vio+G%Am|g#8vx=`Na6e=aYXT;Vz{#|4S16wco`x=CPr{c^_awBX8|sZTeDS+sk=?QeQ~4j?T{a{pCn10NCG7H^Z%{OM1R<Y?xxdamF)5cSIXACs+?IIud2meD^rismJ%>aZ=AQFx!1B0zvx>3jh3Ly_E!COS3f0s?5{0zoD4+I|zA3g+LI^U7<}}Yo0)pqgIDOg;hXidiT;`se5h)s*q9^#!^<GADb<wgY_h7K&i}K$_9Lmaf1LScGd03~ckGZt5$H!Yz94MSj=wLCQT5t*PMd)rTIJZl2HDSs3a!vfk`RPgB{#U`lkO3DdtMb%wsU0n`D|d78H0m4wmOK7tNQ+i>v$Pk_H$L^6a4f3?vJoN?gE&9&jyMXXekMyO<Lxf$)asDUvj2DoAwncPOQaAJbg{rw+%yvu8)lDSZ)|anxn!LOad4}=Nsfn&oJSXJ3`_?BhZk(fcja|UuDslBR~c(q;lg93R!;2IJB{ohNd|OuASV#M=?q-vqu|CB$CTXjIx=@a0U~PAOIID`51PD03O%dMazfZbmOCHyshw_4*u`O#`<A)}cg4846|UlFB?DcTW}i9$kj8&R{d2xb-&aAcGS79z$y-DY_3u_8=<Yc>F21d<^=uh5^Zi=pt$4MZWvud8eO4qzr534T0_Uo(DwHP7h0E^Q6WJZ!9huEm`9tZlMPne6AUfozF?YebSHIBq^UmU2K>B|gH9)%{^1s_n9Al=lTNr}PhlVbDV_5St<LJeVpT%G}>w>xGk*scAub;Y*dPIh<DvS<!KK&UcDD3&D8<3mALNW%G0f;i@FE+Vlyw|n&Y7$&`AUp{)^bNh|)EQQOQIHm(|JH0+sW)0zF_JFmWt=_2W#_DFSg05lf>r&>`RP##KvuD;bb6$&4khK`HgsFL_k(WRfc=*BzTv&ex7G>@(lQ@&TQm2FX89>3;uqE-FGiNP__`U`t-|}uD_1+6kdlNA=nr1JFdD&-$Y)bbT0Db4oN{lfN4O}YmCa)W1$Bm;^OMfD<LH{E+kn+@AyJ&v+MrFx^ylNgUSglaFIS}8b2Ns`L?d&J%+&BQ)Y&W3O<B$Wy`@KL(QRB=O~5vB&XUVv*0^fZX_!8kmjsVHz<jNvrmr{nm{BmM`;`J*wS4>{%wTb5u20~zRq<ZRG1n7FwM(*av7A7jrL=7@lcwJpFM@&ETRFzqnpljlk9*Xb*l;-KRBo>dcRu{)qrdv_yN`bG=qDe&`_Yd-{Ow1-{ox-z`u3wg{^*^L-rF@NFRB4Qh&{kAdvJ?EwE%ZO7AY?u8SX%dr?^L&IVnx+R)=TIR&;tUGk8VQ&XK9cqHV6JDXClOQ=~7E<cx*PO7EK*v!mcDoq1(jk1b#AuH)0VK%VD2-ch<6xC1vyIX@{~$_j<c5Im!00SP$o<BNK-2=`v$+AXoG=K&lDf0!)`C+SMlnNL>bT+#FvqFA>=e{4#Wi5_LhU(1rJJDfK(;W8cfqh#F~C!!d$U6<5D9D7DQdpw|pTc>`ZZT<+2evGr#Eql|O%u7IV@NDj0F)WJ#)rjH^;EeGWLh!2AwGhl})^)3MHb%JNyr$k}Wl_f;Yzkg2B2JDmcBYhIAd3KQZKIRqd^DYeaa*{gn6JQ+8T=03XGh5+R$w_B_n;?slNF&EekEA%5&E}ZjAL<`0SH*9vsni~Z6Dc^+-6S8x%tJnmatBx%&B9U(^$Dv=c?sS<4X2Hy{YYsvC$AslT)KUm*=gK@Tn{{tl+5c0dcTokCO}VQB87wWYgpz(p&6_EC5;PN;3`OLhgx(vv#@XKZJNHJP2Rm$N%*EPk#6ZPk#EdC*S?{lfV1^&;R8eB6d%{^-n+l;7>pP_Md$E-tT|%hrjd5AOAhS{eura`QyKQ^5=i^$@@Qh@~!WD`jekN`A>iI>3e@g-#-4{fBg7+fBETm-h1*#-{8Q+|M}fde)wlk-ustNfA0rntttHZPk;R6JAeA*&;I7g-+hlk|JUz4`MrPo<R8BG=}&(0<afUH<nMk;Z$JD)I^-A3#*g3s(Huwq9)Ix7VDHi2&T!@660g`4eRKTiM|hDvdKZha_xSzqC&3iooJFQ}lp&kN;<d|P>3s0sxl*@mdkOrHDKAJFkBp3QJ!oIORtJHu%94VV#wSiZL}u~8=fZw%DFg1pQb7d@yB<NJ1a*MmTI|gz14Azvq(16NL5=CLfUWsn%F?~FFtw`T!Q{EEuBuF6RYh+XR;z-wL#Zpjl6QRue0~Mrz7Dp%n)cbim~(Wq>ZC}C?>4Q@w33a~&LSWw&Cm+x>6vF2;eHh?CCzhjD_q`v?X3B7Oj5ogR0OWUxka~&2b0S%|LIXK9cWn<`@j=&)9}@J00UJrTHFSKO_9gu*3o<dM{D|UnkDO;4uGE)@_7Ao_jk}%JbHMYU%d42I?(7Pm4?+fTQ}x}vmp>~AgZ*SgJSdzMoTu;IVC_z`J&(OHydBrMmFn*NFx1EB&|TwJ3oAjEbe_ojSzQyNL6O+2-iUmA0k1`0ogpw71hy<JS})n!WglQV@xX2rX+{hTMvKn?tlNqU;Ne2|M@#le)K&s0R8>(pa0<F-~V4ve(%Q*7nJTPI2b?r2{Eq6AN&yr?c?{~jUSp<kN$QFp7r?szdjDQY%?s+v^PH0IE^}`(<VAU(1v0QjK#z*6wBPl)N{@pF<wslvzh_PaD%t{5BkD{8JJGW!(aU14?cbG8=t)MjfaE&9CV=XFJA5H$q4X2RGC1k+BedQ$-(O==+?l`s<NhDR#m(Zde2sw3fQz2x%&2+pSDWW`nSNNOm<xx&r+}eY}-Uv1!KgvKqij~wyO<7nH9WHMvxW>dfRGXd3t<8KxKH_uE<7NNL+5XZMbNm1&{H$fQV(Pch20Yux;-9U|~l?#x!AnvuOemB>^9Nu*m3+3p?ZvtCZcEy0T&}As6B0%UmYZe6IdQSH<hA?3{*#4kSp5n{DM4Jf4qQlZ<JX+LBZcJy0W7OMB+Tq2tO3+!#L2@BQMtQoQeO3|wd~Mllr;rn5^aCM_V!DT&KZ9J`)-+ljsCgrT1IwiD^K&8&;sdM}1))wv79JUG{$$XD(_S3MfX<xGxxse>_f`I*%!?szE2A#oKqKUpO#r)BXx`_T-hPT|UsV5Qm7pCGcm)qHZzbMBqpcTHw0(c==<r0F2##uzMs1D#0d3&boTo-ob^XAbL<Mo5xg+aqly1AEDY=~@zXA`HzbWmhbvrt`<9`M2_%<(jn9UQXjB(Wp;=UkMdKW=P1&6tU-}1nnZJVmaOVDWcu+;xyFjIGSr^$XH3^Z0ssYICp+%dufdTc2dBa?62Gur+*z?I$;ukkn-JA&i8uBCj?5|<&{h1;G5gel%`FR{hr*CmS?i0&zulUKZTUxt7C;Ao)-JDX^aMx!-f(nSb*2P^A5bN^5~dG4&DkI{eX5h-L6HzqJ5A$B{*`?F_S9lT-)AU3|1bG=+f4%T*>IpmHDQ*a836)K9{JADJQBjmzsp^5HK}&^qak$XVrM#;GzLjOE5=p;-Trj=7LS;CPTr=L3F%bm~XtISe8+rgH$-loAZX4`pH_({`|c4U@rbGhrqKLc?$F%G<@H@*6!rrDeTetc%_4nCFpKmeiVA4<3ey!IV*rulZ8&$n{q1(-jk;WU&zMmr-@nl*^B{a?=SlkrAsy>A$4)gp*eh!azfWne(l!XH*UXnhxn_QODwa`7ube&bL0z}q=cb)`{na|&U_a77A{Gd1i@r^=FJ-wl_lq^DdHszzI@s6?tFbl@qPu6`7v~^SuY<a#cAs=<I9)4LS%H{bYfD*{H%fr^;biJPv+ZB)VYmZ)&?k2*aitVazTb#JMII`4<+}TAG@T1$<5PK_XMDBE|IEaxUv!LK{n#%(Gj9@GYuThcQCsKX(yV+U$yaW%K3o}X~gWk7lS1qZ=AuOh+LY1Ff2-b1XifoP$=~g(j?93@UYpd4$T!=#&p4K<wN%VIB{istDq#4;XA7_b(NH*Et`%*9Za;IkwQJMk$8txSj^pJ)B>94G`i@88#D}EW}d|C{W=$;So0FLnc8+We2FOQKj4_Ae0lOJ<xG)OSlXiXvkkN9RKBs0V?Xk*?do*H{xavIv?Kv0V0MbP!|INRA;4asmfJ*hg|tV`4Qn-XaAue2vOzUSyPYWqIOwt(<q&hAh$Yv3+~t38={8Ve4Cf_+#<YfIU`&}{RffIJaUSRBqE9>9Zb3mI%@ElNe0?v<@;Amudy@^vDTyytk8Ad9#=SA&_3KFFbPFd=me^tqYLl}1QqEHDg#M7@87S#!%1gdpldzmVv@;mmD7T<d#wdGtQ5$o2f)3Wt6OOEK(NRlb^94M!<uu|f>3yQT;>y4QMrj=5DyE+%=v@WSmT1w!-KaX1JB-9e+}#bHHU6YL6OTXBR6)?PpmR&amhoq<fF#BRBgX<#38Lug#-LixSm5=UcA_Jrn>-b&H|4-xZAY@w%dwJ9T_ywGXSTXHcWhLey3S=GHE_*XFnV%Aqw<<QbInqX-Q&!;C~s@dTG(KEK3q=c8S=?;K5SChL7c2w5c1hgOVFH)udN<_j??_PHNabhmRuYquPf&JT%b!voLW@75IAG`shQdq&Fi@U%1k<d<b}b4d^S#@GU4zKq)R@;qEX%0(3530mV;VFoaQYbeBRY(x<Mx}#DISBQ7EOQ=RyNZD0&o4fhfHPCMEH<Au%Ck+^6FULvs{M8@K>2n6p|!nZqK%z{e7`ewyX;exFv%CX8p03rQPc*Gc;hR9-o(Pq!;uy^HdmN?L@Y1r#L+mMN>25VH6}pGIylLsE#>-r#FMD6!-ND0Uec7NI5aI}*+%-BC=pKf$I=!P5wAhXaR?YE6y<zAu=>G&W=78UAEmgJQ-BRt}jvK>Cr}H2Bh?Fy~Y(aq<zuu>b6y92SJIICNdi6sF9tkP#P&McK(;un_5Y78G&nu~VQ@`k80f;Tifmi=u-@$Fowo(mTf#AWz_xtwN}lhC)xX2ZMBR_RSYkjwgVlqqR?z9X=sGuHRqzYVXdMUb*>IVt`dwVCd}uwU-BoNOTu60aQs9if*WxNssjgobH~(k$@pD!Z^l^fi0&H&&4Fd2QsE4gO?NjmqjR@i_u`VIL~G6R1C{Os44T%(s!@CcJr&fTmK#jgq`1azVxqlCQ8FQj-$zNZRIt}HE4Gc)Vmlyy?Lh+pDOQ0I%8?xywkKv0tN?k3bbgwk{~8Scr1x>RHM9xg376!0rHs{mq4XxMyiUdFH%dml|)A@sFX({;~srx%H;nmyVs!;?rCD*dU7#4Nv?TzU@{%?aeap7UsYGB(o%(5opxTM#--lc%uD+r%{A8?K8q)g+A3ept+(D4uhH$7H{QJa$|{XDEo!S(&<}gC3Q~+t>Z+}+K!2JAtGq$T)Yjl+cRjZ2ZU#~J5j%7!97<g}#^dAb&Ww}d5kkB(tN4Dm&bl_lnYYoUn@O53=d4=8aHUIgdZ}6`a0k=w4~GpMY-yH75n&^W#3IN5Qxm>i@(=1TT!-Sh)}3-Z%C4webJ`x(4xj=1=KLFWO)7e>vG+JAQqml~81LOR=Y&W-DA!5jX@F9nkAq9fL^eS4ZEuvHFW<8a&(`x~sMJ;Z{Miz~JH;urVeY!nIv<m`FUk3KP6|HVF&TA}K`$P%{LFa*8u+2dPuOzDQQ6X<dqoPLu^CcmSFpKz=5Wv<d%(13TI5@jq)mlS11=7Vb)#l!Ld37HWRdBwp~(Qh<P)z-mRos;n&r|UEb}E$NLSw0F)u2r0Ciw#LIhxBOJ4-hKIb7?1$;YG`3Q6krSRTykB(dTR!xc6LYkChzMO~?l30OEukuW|TTxFU4(L#XE~k60JgY}_X+cn3o^y*4`QU9M%J{stc8WE05cvf(3Vz#;B+Q3GSwid&zBFYb#tjOa2y!J;705JOp*uReRMLCY?_y^fRzl2=h6up(P?+<=c2rm<AI5s<_SywyZXP;};{CCZF6z_Kjys*ZxJM0dax4Ve;kTXE-CT4s$OatnN|26dSDdg!X&%{;7UL}6+UTeg=dxQtCYr8Ym0eWM`tp$uMyO`hN~2+VZ=yoH@_Had%9H|S**3Iwbfk-BZFOf@>in%y0oO$@YCa!k%fl4T9A=128H@%josCgi@XUVMsHO9O)S_nsm1<^}p9@6^Pz9u-sl|gdDi9?4HR4~j*0icNypXDLcQq*2i@R#&=HzA->Q*8E$Guv1C%nz9xDxVmHG>O$6KSuo11zFo*SR~Q1jTIoo(B$4EIm-(&Ff%HdS=7$^${@HtfmFcJeN(m1I)WQ4iy<IhH-zUfpDBT2(A=<*H9{9BnhS%=L0%_;hM||)!E^jr3X<wX|!Ls2BOC2yc9O=|IOWVwg9zl$wwh|*BOfPVs1-Qk$@bXWNah1XPoi1S{$S+bAzh-bXpUBVsKTI=(eW#Q0iw6D%_0#ez+XO6iTTlIuQ_qG)Fj_hLF-DQ5bQ92cNtQ=CG6n8cvXzQ%ZCRBDXK`ntDBlxiLdeiRQX^K7JKD++a!l;QN|u(8%?wCP-W1UqYL*gRSJM^PEp20-Pzp6tAxVu-M#e^b|sjapqbu6)FmKol;tilb9|jwMDD$*Q2pp`3Bbx3V}J$z!ZrOi1~spQiLfi#OQ%}#AC^r?OiSCYW>n)!ODkk#>)jkmk+*D#9CJ(T?{v!(Lu6KZ;|x2%oiAvXKelOKh(-8w6EzbF`R^Z5p7eA4j3vx!JjDApFk-@9J)DZMFN&$EgekZv}pKD@YTof|BP<Cz-bh*OX+2oUyHvKC$HYV%Lip)0K2pv+#Emt;5##Dy$)uc2CyHzh%NKz9S)=Z>xF)2G(LSX7(Dt1n#so>`~^N3_oS6BU?#a#m;!3)EBEfb9zc?Ct61q`7f=`JqaWc6&C$|nF=ruE;PD54&!=$yY(~KsC^na~Ii~-X{#W<l=HvJO+blQ&B1|YY-0g@`H0s6-@Z(^(L>qgLKll%@;*<+3<F6U%y~iK?4>a+?@A{nt0U^5#j;ns3Cy%Gy0<B?eyW8!p&sR5dXZ+|Nr!;Fn)ho2jEGYBDSROon|A$aZ<HsMoCkq)22q$j89PB^-;Jbqo*pQ)q-^q?MdGt<x%Oh5P{Lzp5bVBl&qv?MZ902w=sTcJ2@dy76xPf^4cJNh+yPu=b_hXD%im@{X>P%QMoYw*Rqj$N^qj%@|3>8d;f?<pdI>|V0^~0r!{RetrwIH(K3*|Hc$_x<qjII*M1R*(u)T+*2YL)Z6hYy2)7wz#CAP;FJ{{%p3*&YV3ayt)$ujKoJinv&d-#5(6J)~XsFj!w-5BTr)>VLvqNp+<N=TI_jaGMbM(!=27C@0jXf&6y07xOO1CG#fN59}+)C}UpbdJo=Qa;Qw6b(yp6K7%>Hr1O1GkKT=yxP-s;Tfe1T+i`VTboWK#-4?q!XN&GuQ>uoo$`t$ow|5hiNA}htn0)l(#~*xuwtv263keUZn>7}pa@=`{i@#pEeeXAVcW=G^+TDBYDf1<pU`Ec)m+X2qA`Tra8j%p}TljH<d&p;x`=xk)we6eoRe^}rPJ^-}#~x7)(<|2YWAn;}qFQTEmCdKhPE#q}A*#<fH;Vr8(q+WDp5A4|Y%wzOWznzCzGvSNZJjjY@pb()aQNO#>ClHp^=Z(e_{k(l4P2GjHKm`YSSXGfvRB%(+$XUCdieGB%hro*lEW08LAto`F`=UsJ|c_}&lT>5Do22EW1BN(IuK1gXCoAiow(Bfq+LHxd~;s(vvTzaz^vZmLvt}bz(^XBOgVGn3;5Ci$|fT>=ek{L)p*F96HH@HkxZA!(D!_7l3-c32ww!@O`Q=Sr8l&Ud~85dwd9GUTRF;kk^A;-&x<ndR>T<B5tJewF~9ACIqM|R!C_StqB8+}T}!D_%lgR_h#Jr-X%8P-Qp5Uo?#!}EM#J$vW==wrS`<3B`_Tl0b5>=^nxkrwI|B9WHQW1>WUmok*5z*&PjC3{J1?m=1xCtP#S*UvQP`EhE*v-QPt}p13ZES&Gz0BWgl|_R+*$3jQ%|QEsOdxQGb>(;#m6uTzBck`d7ibkY9Ae+e{zSq+VfTs&Z-VK?Ju=kyhqEV&QkZ^>haiYNwoJW$O1VxfabmApdzNLxZ#<!3&hd{cxSxIHqe$efL2^YUllU)nI7UQCtRvL&{bco`fyiO9;X$jt}0x+7tMynr)*cIMypATRg-6}CR5f5AMsP?$EtFMKSjQ*vpQ6jKdYR%@2ul*^_XzxQMh7YcQ@CH%V+d_j>o-bu9v6YUv>P9gXx6%`iIU|1reI~LB75TLYiT4lB<Dl8P|F@(tVb1C2?|t?tvrz3wf{GV<&9ScwY9%eaEH9?~g1Td-5xzO2|V@IfQ|QVauL!cc{&rT}*piQ>q~g1jf)$%^B@Ef9q>^8sYl7kjKy-&FWH`-u?#PV%$?98~|{3W>E`+j$KuEsN5V<P|`tjy5u0H1Jb5hI)PYQ38y+MD;aBLxk^lzgG2lAs*R(`yc^!8w96Tt%R_eGY>|u4;Ywi1yaE;uXeulVCE)9v!TxY6vKC?*!IQWS0WLI<?$R3@_<?2QR0l|u$ZS`#0IVw+L~i0CMdIOlbIJ=unxJh7fk-`_o(x9<W`nL`Ks}1E`R=n=Ts;_3y}q?eC1=^R4#Ma9Zmo7LQrOdNRjpmAop=@}S63Fps4UCN@{U5J5&VcuV@|_ow4tXK*UiDinh{>BAEE*xap?>wbYk&LD3A`;_55Qslo!s1qMnU)Iw*PZuHY!mI&?&5*6Y#MjgqX#U7^rVgS_Py1=h;8gvFdeOm$zMYs*};gVF)jhEhJ;r0jzKA=p{NDr1E+(E0w{<g_#-0~Lyx&Bz5h6$PA~BS5_C)Tkqo$lu!7hN)l=Mw?^NF9zHgWG3trx`Xk7p<-;5pb^_xSB%A+T4#}_*W;{6Jr!vv@;6%yNna=SO(!RTq-AAp&E$4&G}rzQ#nR>W',
 'diagnostics_snapshot.py': 'c-qZeTaP2hb$*{;Q63>^_KL&V*?4V%a{<=7BY0z7R=cYR@nq0yHj89yn%#6?W`>#&NT3HJ4*{}BjKDC2NEQS|c8u6jf*`vOc^KLMWAaP#ol{la)!ig#Bnv_Y*k!XWr%s)7>fGuyH#ax$r_#@*@I^NDqo^yA(I^UIDTbjxj*~181M%aBkHjcVrWh2Y{#hi_BJPSX4x(Zh#$z$_(>(OKIqPk1Zf=CrS(4_$PsixW<OWX;k|>fv9wu=n^nm+G5$7`PilH3&MU)T2AUBhP>{QJg0=+z(N;AoP?+W_={UnxV$|Rpg<~dBtC;o`SL2oK^AG3Ya^In$3YK@tnPcXtP@d#hdpjqVSqa>Z0@7X*vkF>sdem|Uz!bomxY+x?}G#z>bzC)0^;>=HDP%3l#8v=iMI#&<$m&&|IV@9SICH^quIXwoE%k#V=;~*hA?>j|4+Iq!tU7jn?13Ak@=glXFX_|nDf9^+x<Y!m-8TD6-7zBF$Y$oGjXXIS)DzQJ6i@vzP;DzHhak^ME_Y*DOP&g8Ck~0~5@;uD)tmC$Jcy!DyX>yi{ee>-s%qMDI0>s*TrJZCs)}L1Jzzz`?(*a2i%W4=4i3XWMG&*W@SL|3AXdG#95B=e+)97q*#r`9V4dE;{5j59r&Cq<QxY1&U{kNVm7iyag4rzK4TADvNEZ$(A%dA6<T`>w}G&Ian1Odszr1u1ykH-(+?8Nen@ZWbd%(FDaNyH$4>OH_Ye1{uKYBEGZD%)4UyVz5n`kBv1dfy45EmB#7*5i)}WCws$2eA$)62}NrOF*p|=;>UX$a!BKieWLGWt~%^Cl-yfB9)$>1!1`Vsvl+2CC-Lf7-zX32ht1#d$Jsw1zT}lAu&*J&r&XwVue+^YMe%~2YjZGW?BTD)T;KssPRIPsW+B6Pwu!qmSovca=ev=V~|EM3u*&A@Umh$_0xGLDS&32oV5;>Je6@yN5wjQ7%aAh6D8x!p@%R&O#+{lYL7=a78uQaXlFifM0bn+8a(zz+XUtvhnsNb_V<Ou^Bm#F!>S)gH08zg7`6t^v8@8sa;>v};SBsN%$&Y(ir7E(!^j^*639B3B&qpFlgs~rEeS5a5ZIl_Cs!{%48=I~V-Z}v_{=Bha*!k^VtV!B7vZirQ4_LaE>d6QS1+Cg;>%}OFMdwXzZMx9#95M_K&qTYMG8<t;dr#QqcJaOX!?#Ux_aSI?{JnT<5XtLXt{<%X4VS!G&EF6>N)~U?y5=Q^OCqnCO4Mp)>$}|FtBV*)hD!OQlx<t4<BSA!MPBAkS0L>@Ku=ylkK2LqwUA?CxwKv-hLt@8HdHRrvV?vW74Y{Vg^etiIYAWhrI7dj{RU>@+4z9!5JJ!a!cyTa3?TVfy|TKkGwS$EW2zXm_{!PAbMKpDl$}(R|ns>Syk^+2P+<ajRS#MjpJ|w9swAwR-K1T28mjp+DQhF92xnuOb$KWr?*8aYuUUGbcNU|-Jl*hFkzOEkK*SYEx&&59J_8$ZC=mSDj5C@cn&~@CWHhbS`v+w<%K#kK{B01G6&oD^-Q(*vfaxkVdhy(=`J6K_S@Q8X2C=bi>L*&Y&Uj}9MgRZh3%3IK(}gU+0z{d;<m#74RQ=^4OEh{QblTUkJXs91@8=^>k)BDLDjk3w^-b*Kr-1u4A8wVEFNI~D2&4l8$$<<O4K2~yJ&B6y$``E-~s6GUGb-`wLnF0KldUZSQ~a>V%Z6>6%qRtp)Ei-_n8Mh?!gEZta{b5Ozd0Yd$(y}J_OWIlG5I*LXd#~S_xqX{GmzL0)QfE?Rx5;cd&FN8e7$_+!F9k1(=y$jt(=nKC}?`&=*^@V0UGjl4LE#sO;`9pVbx0)v%@*yY<vIJQJDJEEhfhF3<)n`onLpUVM`47;1d={IgiZldI>S=DW&TO_IxxFo#;i5EEh5(#v1P${vPS&;KV@>-5Co(|aNS?7tP!<wt_x{vy!u<8jYXXjaZUVweCvQyNUdQ`vz-gOCNTM_1@3?QkG;djyG+=_!bqL#qU4?i6ZJoJ|nmh<CzxNFMwk4dp1B18F!zpt)ebpTa?eiyQ?SpUTf^#hmS=ACG0{70dZ<Io&08J-v3vAXyuxPW48^>3$?r=)w$VbG7FU0RnS@RN6rwPRtps6^#a$Ni19z9w6Jf3rW6%e-LMD8_@FKZ~OA8A7*vml9h87NjdA_sC91lZr{cEx+8AgGN7Old#||0EgTeKG%T6KUSXX*b`#Vh1*tR)x`ZEVWFi}}U)czJ?U-n}SLA{A(zPd2MPpMx4oAv5R3WVPXiOhFGHo~0r^C9`Syf#t{2zHfLsEcE!37K}JTJIgebF$@3dBO|^B_q{Kc?DyVf4FZ3p9%&D4v#(QtlW;^i0R?jiY4Hac-%txV2D~hSav-2zWOvIU+$(GwK2X)rON`#Tulvc${_|NDpZ#D+xfmwZ@QUb;Wn~%K=BKS5XLF0}VK?uX|0D|M0=~+mBxp@fdn0gd6?**G1_@=a;{UC*t(-qol5C9C!WBNJ%3EB=&K{W^DoAT@zICd8$F8ZFJcj(Z?{E!nt&mxrEE2oAWSNQ36bAUS#w%$@7_4ef>${M`1oMTk30=-cjU_d3Z4KQ;*$R`o<A#kn_=Zj1FiS=5EcrH&!s#ox31n{+UN4H_zF`zjOC{ycH}~eyzh$4l@n%gMi)La(hO`mo?QWQHNNL;Z1ob(Al-A<z5FAg~N_l;7v<|&Zb$<ESGZ^O;I>9%sl2}R7tdMW@WlusO#g}fHnS_fdn66OUGWr?9d^7;4TF*3^8Fn3^im85O7wdxfp=fFJ>#H3>>>XxW<$gRQZG2B4&`=a91rAXO6L6*R3VMI5bhQ=c?`Z8x~Xb8K@TyyR$X}x-d@a7&G+cr`Q}>CGgsmEjgJ=QrtzEC{cSNolHRYBY!#=`lR7m-75^-de8xnJF8iSdeF@fl>pL~$oXi8b=oqiuuenKy0^BDHXqOwV>U<a7@wD`#fN&gtf!4Ofx9TwEH$Q~q10G9=rbDHbR?F|RNBj}^;sh5lK0F0soKI|?x`b09zc~?@91oMD-ov%R0&Ht|EOH8QqPVH)04@FPcJC>^1j$<W%d-M`qsfIR3_^>(2W^KR`_TwWeq^yt+SfaKndxl6L>#RCl&ymdLzlI9Qv`0If1N$xR`Q(WmruC8*(ikSyFW3z_U@c;(>7Jv6b-nUL=DjhH<6Oq|JV7h}(pa%)r*Q0BR%LT3n{N(r1;X*h+|3MC@D2j)PErY;=eIwk+;6(W{1923dMaJroA@m42o8&X&DSjbF?8v_z6z8>2dxZ?evMxwvjUORe2xF<T3&m13y&n{P`U_Qnd-xJ(yO6Zer~UZ*$6j@heeHHXs}6cpTLS^{*tJ(L3~<f}>JkS(L3I6xiNrZr1w)X=i6S^!HW06-V&NpDw?SCL9}bdBz)y!mpvNzJ<p3%E+I1M>A*c6Ny1<(mdfGLN=6bLtU+{*xGP;RGU&!29{<<R2%O9|J+{K+n+nNho|=iQ*>G&n5^7&$h|y!?(FOzx+7hX#Y1c13-vx;&l<j(GJD1utRrNseI8u4#v0PEPq{hx$cy4L(Ro%t#XZjz0{}HGInDR+qDV!GEwjwce1|?Kikz84>HfmT3Od&TCSH}h|$&j5NCS-j+(Ac0^Mz>IY;J@D07>fBH<HC$`>3}V!!ttsD^w#lYFawut)zRH-?+p&C_u6V2|$71&K!{^ZiY}<U%Wp;K6<SfT(oW-nQzpXz1)^`5aAyWH`SVA+GH2+?}1%{W^~IRuS$3SFo&q=ex7>UAn8>n#gcG!LZ)lMNZd87Xzpk+<s7H0f|)KJnZ5=B%kzmZr}cs1%mf{s7Lm9Xn`aOhvKE-aJW13hh)6^-^U6hJzEg}aomsODBlHDQ2XFVx(#M`0XRggQIwo*ou&S)@5l4AiA?1J9J)YyESFmuJayT}cOJ3OB)`CQ3)EMCmV~6w7kk@k?-n^m&fa8a8R^9T{`EhK3!;BdW`RHB+z@*oF5qBJ9mwp!D5`#aR>Prj7d`PqE8%!ydj^Z`3(JLA?BZJP-&t?H_)rZrp2DKHw>_(Kb#`!Q9AVB*e);V36IOX2&qb+0c7^ePc?=YfI0>&_d@w~qkT3Ior}7U_iClg{*MhkB`+2T1rJTFawq+Yc;=v!l+Z`0+QX><(p92_gazOc644)O*GL*mk`^#U>8I{bad64Df&}4JJe5SEOV?r41Qik^5P*HI8;->|pM<YK_$&(rF;L`w2(ENnj&q5#-Dt_XI?#pMs!VffQV>7vPU?-JVr;Wk=I|qA^wF5`IENos|ib9?BYxIPR`3?|CWgC^0aph&t<y1sWmTi2#_$T_Lts}qvAtO_p%x4rjA;rJR!NUj80TX_FngRypoF<nahh^(q_shroBxibwvJt5(^}G#;dZFWH=%M|WD)iv%cyrrlqTH4T+Fc#4b(a(jIO4B_(|Zqx`8uZybnQ61g7~Ao#3OAvkKOuBqtf>(8Q^G)bds+Mi;S)|F}y?6CE$)m2MQn`9@Ne~t*a^;fYRosOIibC941>?>Y(VP<$!>?HRFJ)o}3muqF%+<A>9W@*}<OHaR*Ov0G@%ZCzrnxfA-|fH%NY;f2Iz=XV7?hm=f4-O<t9SBaci8;of!$eVgr%$qbOgHl;N4$Z?#P#1eH0t!MNjE<j$tdhyTV-EtQM>s<uA1s}(8$Szxa`FX4r2njfj?$59b@H6ghY>9U@-kd9bw_nSSq6Ezr<*s<^ewV8c&|>^ot1w}kuWhcm3*t3RvINCASklz3>ABE@e-|Q|t2HZ~zljV^*tWEkANa-)^})~(`D9TBov5;%%J(>MgtJXv>Ht)U#hCP!KHIqDxqALzq|a#c1$nj%R9gWJRHv7N$sc~}H_#HqhE=Lm?^tK26s*Qr-Afqf<n;XT>C;C-olweIYh}dOZx0`6cshxv0S1$T$KZ1gkg=HXtr@efnkD1YFim2Lhz!3U&GQMadIb>rslw-P3Ne96;cPkF{*bQd6wPt~6{lnpl;}_e&^BO{HbonNt_l;Vgo=v?2~bsUnQ8kly!>y_mPL^5a7Lg=oQN*IxVwZ=+lmkb&+djye2kl}lgodZ67LXj5En6NmnB|-_i7hBiipV|$s5b=+$O-kGa`mhHL91NV?$G3pTb0Yig-Qz0RuFWOd!p2T{1485>5YA({ykN!3flUU%=o$|Ic{0(Ws1V@cYlfN1XhUz;vvUr>R?5TAAu`XjQsP4e0t~C`o!zr4>rtB}K%uiE&d13HzLjPGd77D7Zmw7ckx4#s>Q-d?~9B-#6y4h{u?rK=9_3hKk3Rzr@zT+6i$5Xzgw&gjoCPt1LCG$qa~kJqmjlsw^R9R+?vH;3rT`FaBXl9Qs7Nf57z%f19u1Be)ijX%bVd?#k;VmR)GpFO&~dtIp!>)f8ZlX;)E!c%b}@^1$d$+exg*R1dd_d*)V)7O&J$Z^QC<`!{xZ`%RyB1ICBVk-l^7q0E_7#`FYAr=z}P`%hCYSft3+OHTN!4#><EH|&^!Ky|R3T{QQ&Y{AGJ_wwmXTVqQ-QSQ>2v2LP#%slJC#?i$$CG~X4GCIs-&pdD@6EVt(Kjuu%#bRMyn{u<t^3UczT#_?8oaG=p^>MnPcvGLg&gRo%HXLzJ#IY|GmU+Qw6w;UIe9EbbOxKq=>u&z7-5pgZ9v|sPx@l?Ga+_^;^eDGk60l^-BQ^AmB!87^Pq<XzE2C0g*wpsH<;=Pky!Yd9_9|WeSOpgj5tMSORJlWSv*p-_kGuzmufBHg>EQz|dL95XJ1ftbd2UlhvgFn|j}PxXc>S<9CCV$;s9tyKH0qOE0BW8Qaq2H)?Ma#*_bWoQ-6a52p50f3VJRUZS5@3>uKsm>^EF^BpQi>!x8D?u<>l1CnAG=*^qhPBE-RO(u}#&{f_%d(H@8_Tqg|n@&ZE3VnIxH{dm>GeqyF7x1J+){sUeGHa9_Ue7Oim|S1xw^O(CMx`LMlnN*xZ0A!0A|!Ft}pe|NsCKkD^D{GJOeC7Ep`(GacG>AUXcYfW8AqbLQ>qs?k#3qC-b>&k^I7gq;o=0>*)bZ_0_i`>P!8{-w1-Swj6Rg7BxQbVgsq)rCT-+~r5qCg8q;AfJpUvsl2n%9l`+i^SZtp_w(kGm^h4bUihMic-+XHYl=-E3YgC1SPp9<7L11mG8B;rmkT?c2h)oTRq)=vc6z#5PTMpb5F%qM^H-OCaP?YinfN$W>{-*HWuTUI{2W31=RpoYX(TGVNrJC8|buJoul^+S>Q3He$v5ID8OI)rGl+G9?$VtVx6ly5*bZ@=;2J0!O$paJ0NUsVc<$Gsl9Ut~vF1Rmag^DI^9S9dgXD!Baw_0^0LYJDX(T`3e+Efzj}A;A|bV8Uu?Lwp6a<XvJAgm9|_~YI6Oub@BAix~A+e+otrC*^9dth^3Ra_Lm^EcCt;P9`gE~o2~iv=(Xx3w4t_-L%B(PB$gC`++dY8E0_ygTu4FiGQ1kXcehH<C+WPsoVJo>y%Atv>$_O#m4@@8dn)gR-caBOA+c^JuXT@>WhF4*z5`QeUz6MZioRzv1yo<s>iwgZ<yROfz~fj;N4cI!WAtWocJwGo)NE8Kqh_m>J*s!D7E`=k6DBDcK)gvXkWGjx_28)&jGCQLUpZ!}d(=i(F__{MqH-Ru8LjAqE=1KqYsE57Z}vpUOyI7uXb~to*|~GO*1AMN*SrlP`ZD9gcKI4DUsH0*_&S{8BNfLNaxEUOszm0LS-TMV2zQpp{gwFrv3`TbS*P^~>#{s}op?5nppsWy@5iHXTsErr%}AIypf%0L%R)6jXrP<dNab5^-+S%h)4%W@A3l2X@lzGrnv+luEuDLBKX~}GKG3|7)>Ip77A&Vw*ag+a<uH`Zv`$-#0<{Cc3a#p@AAknVl_;<3pu-WBinZNt>gcPU1|zH)sxo7YWZe=3q<$U0>^e<Om(2OLPgI#BUVW-8%~|P;Vn(lJGdHT;TK%;bF)c<FvvG<yl)d*Lzo@3ErYZ&4Ag*pj6qs|ReF?0tOlz~&2p>7eNVLq`5>^(1wCzz<O@fzTW~x-=`lZYyXmoxjqzMd}8vODD3M{S%Wh!VIB3u-BX%2DWC_FbMTBhrr!ykBuj~~DJ_z54>wG`2{GpCtn(uU?NoFLmd_D=o88x~2xt^d!Y^Cj^kyzoPJO1Nze$5fsr^!u=O%^vk<B*X(sKuUUjF~=`&sQUozj_EgNmt+}K_}#{ts{!#k7Ag$*aFnDp5w^u`tErU1^8J|Q<fzeAuXD4IeJRa3w)(neZ3TO(SolIAh^&6~223Lk#<dLxjg(Pp*k^uwyR^<t=dGGZ39d@cTDl2-uN<a`K~nr=#8~MgYV+*NC+>2OSGo@6YW=S($Bp0EaMHo@-_>Y^1?oo)GAU2Y`<cu5wPmx*b#KHkcD4CVuI7bw&6<=Z>$vt0l3e=^qptYFiHod+eDykE5u=C<8!^$5eIsLj#X%`F{X!loFhf%Q9h;QzYt(Pr^zNl&uAF_uJ)I#c8SuLz{970vmDPHlYt5vH6v(S79cOC`Qsbys*x+&uEmT&F*SlOk2Lnln^;G_#$>xpks@%Q)jv)={>9-2$r7@e3uxD7<ihM2{gF?SHyis3X))-oq`?Wgra)4<%UhpfBkY$fAz&wv%xA#1vzvnr9{bnLDW8?n-H^}Do',
 'export_snapshot.py': 'c-oCtOK%%D5WWl0e_+)Exv;c0j3h8r=un_8(4t5U)X7B^2x>WNG1-?%u5H!ufA0**)x%2M?m^;m-ro#A?qo8#e(GB5nPNp-cdGDH_S-MB&VmE`V46GJwq5UjVqXC|X5qdE=b5*v0LM&&JE)-DIdjk{t2|WOd@`AgN2ZQ`?H%qr*llc-+f-(~d;j9vW;`lwTeD7i60sz?!PoI<r`T<kwe2HQ4i#VT<`@K1^Few~R^yR|64O&!Qiw{B_iR2MF?@bx&fDB|mGPBnz=@2h%C#))IQHr6^5XL9{L|UR#npSP(isixK~+Yx-~amaBN-DkxPxfq(dDD@n@~aCcF+iDidLi2g7?0hUGa=ELpxA)%7GpWzPAlqQigfeD(wX2ld-b3j8+73v|^?VJ2?gKM7G)&Su4|sp|pUuWs4p1`Oo&Quc7fbK_WD`0zG)sHVY|rTS%E5n&wJtsgjN0vsv4FJ`Jrm7WCrQ_K*xv%VDHs7yif+;f!*$L^1VI&~}dRN#fex7J$No29fqT<8k0}RTR*9(0LH@WCu?j6u66)&vwa;=Bqig0WKS`3VvUrG9uQreuzbgQe%>bjQy|(8&RAcc10Nn>@#)u8i&gYeqDEdn~nlG(mf7JfzytM#SOWY1}aT>2+C{9cyBBED@~&+nO;0j9E1ft5v*Ld)IM6_1|BQlT6*dFb!FTJuobAb&n-UOsz*t~Myj-S8+G~q!{GPeb(d-Aw9$?>Rqr=3jD-gSj~HdW4H<VDRYS~F6rL?AY5DvWwup3o2VUSZPAD!2^jP@IVHYfVNytt~ZBeOutrfHABesnOIG}dix`Q@g`%w43$8+bm(s|{3$5+%gJ`4w^W>Ak&F#x(6j>r5MveTA+Y@6BOhP*&%aip`M_>@JWtURJB_OZet2a#p@2A;IJLuwRZu9C{oht+>(bBo#`4y%o;l`l3!4zAcht->kG<612D-l!K-5}C$-c!$9;UFymLK|+)nR2T@wI7?oNz21@H>5S~`BGaM%KIBQ`qiB^>yjoWf)4io-PSh^DEoC5Rnx?6wMf@AM=RSY522%+3BK%I-F(o99$3dmL3a=<f9LE1~e%G;AB$BveaE%}f&hsxV@@1f0-PfHHcFLe}y+!hJg)xg?@h(6PK4bjvhVKWIlFvx>6|<*<0z_q0_<bHoEf(_a3bNd{118yytda|w9}M*-A>E|w&bED{XEgFBgJa6?lWW4|e}$qO?~k_vVI%sAi2fPLtO(bJ<MY6+2{If->9-x=E)N43NGKgX84o_qolM^yc(CV;?wM2KO9;x4kEDF$16YHTxU*qAl=rquX)2&BpI=_~Jk+Q<+_>Txr|D_`4iH1eq~n<oCKp*8fc4hH?kd~^wM?;BOVV8h+=6PtjQkO1i96=6=_kk!3FqZ_^dWrE$yzn0M0u#;_y`=1en@GGMQd!^;`00R-60Wt!<3jIe5933N-9Xoz$sGFwI(G`NX2iI@#tR>HBWf',
 'longchau_snapshot.py': 'c-p;PUvC`AaewEhnD8EE_PiWYy3^r8X%DARI-g>lBuhNU0FM~V?#}Mcadu{JW|kC>CSV{>L0)1wFM%K8i(wcEV#Edl1d$N<L59Bv{Ymnxs_veie@p7@<X|s%d%CNutE;N(Uyru8w;yNe+&Nf;YiE+BWt5iAI9sRFFps0c3Dc<)Iz{x=8V}+yLB~8>MmzCx73D>ihGmS=pB^4QbMkCmM*jBp_SOtt04zzONf~Ep;h=Yxm(Bs;mQlW?eonI0x%pn4%CKozMrFK=^cekn=+NiaSsF>dXc@+dzb@m%fM11q5mEo3%mO`5PcZ*FUg__PuvjGVSpSSO{q0MvNPl03<wB-f;TyW^X`ayo^*uN6?YY)zJjtenN+=N4c>(}FBdEcktym>-xg}kam&-&yT9nIVYwHIOpTBtY^ocWa+yq-R!4B=5-gdV>efr|?1zIoMbOBV>*+f6~C)v_JP2Hj6o}HcfeUC4<?ms_#bny6L@Z#a)hX;pGpVJ&igziqch_Jg+g3q)ZZHIXoPm*Z+Se_P>Fo||%V3T8a>%pTJ5AVNt7<}^R;o}E{kmpXLVv@%z+H^vqIA5;HY)M~3Y(|!3bMO@X%;VyO{!Zg?o@PaKKb;=tQCOZ+V-lZ&snGN+k4}m0P9t})_2A(r_dkEER(_V{C*VJK-?>Flq9i=W_d7n#RF)B@rDyNqC+!Ou6Tgmb?cw?Rcl={CkJr<AM6I_zMC+Dg{pVJ^9if9eVUB04qcT4aW?;N%8f0mZN5y(ceGc<=bh-6ckDnfVFF3saUBPiB3p`~?h$P7B=yP$4KX3Erd;Ix6f8OEG59qVZ>2sW9lasI3S;;8O;`t;r1fG`sImMQ4ZB3(@6BofED@qT{HQ47>0q1p|I-X`Xe&kc9G=!98x#PM6KgrIb+#5J?>M(mLaA}q=!zBJX^2+FCsUXr<EYM<dzv|~c`Q!jNaUTUck5)-I!J{vDJzu2QObbpP`Newd<?hiJrSrvlZ!g?C{tk6?6r6#*4%7RK^y0nYr2zaQRe=33vpDsVI7P1#jEEJ@fCV+s(g>O_RA4j)1KmxncbY8&>g1(ZnOg2DjwVoqBj@Ote^bA+Fgfv*kf?6-UrZE(J}Kg~D8m$MR(<?3oDUt$HP}}RF=Oxqu&sW2FKnpP3HV>QB3{f<<$1y!QJl`{Pdv**;Zn$x>4nv^_>I?P36{BRO<w`GxECa2w4fEnIoMlB;D@VKlukV%c!gj6saZ+I0j4MI#bgmDQ<_h8+^@+V6PCLSYVGwoxC3YvC(RjbT9s24q6MwGu>$8uPUoO+MFYl}8OCN*OC%Iq3WR~?HSE`%2w;nr8cX`$*Utkw9*9lTxo3!a4|?UJd%N_BMNUX{Mgfi&QrfI@^el9C1@V-$XnF3PhRIs<b(>lFX`M}-Cv@QQ4)Ebha<3JM0MoGKd8Js$aA>p{EyMFFP1Bx9{Z@l%0ZK~UU9+soI;XYLDIa=e9s%u2<<VY{z*4*ak^o@nt_a1^$MJNu9RyDv{_rq(aR2ar5Nsd2W)(@4@>UiD=PyRiErZ;}<<L1Y$8fNW3y^IdXf()fa78`fgs5106`m(qNK~U9K7B6){?XoXr--b=d4%q2;E}t^vsK|9;|m@<lOLutt)4s2BOj+3&R;aR16oL3-76yEG%wC#CdW8R7a%935Eh9AAR?&(OI=(JY>*pN9Ca8v(|A%6*Mu?P{eU>*VV-BX=N<@&9A!y3+4xJGg^r@21$h#t8IBZ~n9ZGo`aU`OIFFLiw&}Eey!-JYk7gEtoQ%|>WG1b%J<-5kfhS9-j!N2v`Vc<Wm;gnx2Aa$dW`>$>qW|cS0qYp_gnimzO$Y3M9ps>{2)p@C!yyKQeL7iK%%DcdcBr!|dO3+!rQ>}sI_K>eI)~@0NPVq#$(p-o-W0+fPn-NMvIs6%WI61i<eYhX=p^A7d^Pe03NTrq#MNZ*`_m{|(HBc)DyHIYdfWZo-SCLF7~*N<@RJbyx~0z&nVD2cYp83jNxJz{NKZhV<!MD-#M3qN8*B~H(E^7AGf>WuH&7OXF&12B|L)LNj29-P)QIISc-8(SETUPKOuc4zu|%j3AuJ@iWSzrbMG`Mum~u69M{u;j(r7iSR)$Fwk}*1mRp{keRt}vd<d`@?Q2|cb^Jtl!M$XwHgAfWI&0-)9pJ0VGr)9CrmS!<@Qdr%&0~32%IP9A&qbUm}lu;ob9;1kvTANCZn!GgO6z54ihbg_zl@Vo_ljuP_z~qP_i8R*Q!*`E9W~U>Wta6eh`DO6srb!YO1%HdD;1he(%Q7$Qt);10qECjTae4X^np<=nPs7%9Z5C?bN!Z4kVu~*NEKJY6GCPS<-W;Bpz?&w|RqAhN8W#WupM>Ws;<DKku1hfM4oE$V$-rx=noa+~LrGu>On20Dj5R<_*OPK5kFY9O6j&fR<!l^<W8jp{b`-PMC}?e>HhV5GK}fPsZc&Xy=EuIQpMXP(@Ut?6cn9jTHMp$XdEBKbs7gS{!Te#gbjrn>e`BNe@Xfz3oke{0`ft}&qv=7P|B2WH0u<bN5iXsR#nr1{WUBMvMqIe6(S$XX@R3+sB##B#F8drliPLpd+4kb$)LF1cxzyRITch55a}*y0bacv2%VyA@n~<CjkAw_ka+z|a$pCoJ&_ox7-9W@FEcI5$H>H&`tw)B6)aKO@BAi*H9v#=(-*FB;djZu~WLSARg^TVqSlS5-Xq$0DA*p+-kM4yZ`M!T|H-x3U`e@hyMkmgf>jIVu@4N3j1-TVX)eh{?4mhF~op6TG6&~PF#g<LtQWojg{TFy9k+o3p2F*kK!4|Q3;wz8ClQ6-WY%VuTpqDt}W;1{u(%AiqRGI;*o~@2G(i<GQ=}$)z!)?#^=ixXlcGNXs>Wc7n&-pFqt<+96%{9u$rm7myB^Mh7=N;#9G!G}|%&X?)gHm+zc)loifYLk#cZ-?_8I=7nK5um?vUNU*h<_qq!G2!rppRuT#q+g<T6Zb#31-b<)#JpCa*z$78XN@$)E?MTTLv&p<DB@DyTM`qKi7sFv9Ltnt6~^-l*myH&0|(6)6K_;jT&=;5Nn?)>(Z(O3lcJk23bzglll7U^-mEds*Z|%w;^?_JO+uXMSyyzakefR{J}1;jVzCAV#|Ir1GM!78-lLef)|KIHsT-p+^8eRVaq^G?%j6ojl`ohqX*RVf#{ed@mf4hoyawfi#0@ze0AM*g0?VOEkcjIr3uk;d&m9bkN;^~;yTno!{rp9Z`5K<J!mN4!?=X;L8k|-P{>@1mzj%+ER_Z6JBP%?^K}UDterKN7Y*p}w5Bl$0|gSoc^<A7#a#zlTdZ4zj3W{hEki<EBK_GS$|YL3h_u)0Hx(prLCw@RY~;BoVtBA!boJ_&Oa1K^>B1Sm`6lC^i>uc^K_~kAch2vBa`pP}(*=iz|L~i&^X7kAmRNg#D__WVZQK0Bkr-h=70_C_+}JH3()I1-)BdlTvQiz`*Nhx~d1$SK&&Aqtq?Jeq2`pC7rygJo-sYqd)f*T{)}c{BKiau<+;Kd}qWVsSjmOjsY=wA9pcU}9u*%+XUMwIJQ>StnoF$ns*<`X_orAmK3!PD%J72~rIc%Eshs+LPC1+=0o|2hldVCxG|JEwmYw6rWW28H$Z+`9+Bqh!Yy7ym~&iv~2FJj1o!<+-r`FZS|##gU?S1PethBWleuOU}xc=gNDR^4uU_xd@FACao$V61b`xwW^qcXM-_{J5a*$Gzp&+JtY=TmqLrViAHo(iaK#(>ERL=f|gW`XjM^`o__JYZaX{>qrB~4LY256=V=ASp0h(tS;6`S=s5WC4eYU5S!8?+7i}5=a$W&EesRXr@R@b%{XhuNi)uwamtKrY0j^4>a3P4tEJI6Z^mh}{icUO8ES_T*%bbVbBvXo6}uz~kgjN1$f1KXg3pIsS@<|qIPhqsNn$DOW23ef#?->=vSLt6;%Fn2m^ZfS+C{789G0eUz_bX{2S|x_Jt5Ydik|E%KB&Y3bF=-ex3)tRs^}I*Dh9*TJe-u?IGdi^(fnFKo%j}LsRW71i_jRvpeP^dx$APa^P!~3azv$FrGj)U^mEBuKH%Q1Q0P!#M;w8EMk$EAFGPyeG7K0Je@ou^yIM*{d7~<bO0+z^)e+IDx{xs$odu#;D1+S*gT?Yb3~+uCo-U=bO%KkP<N?f^B4)j$DhMQ^xQN0jXw^=PK%`Vcmey0#=X#9JRy1r^Qkc5SfcWiJ&3Khiv|0>B3cHgw-1j6-Ysn=$ZJ~2NI(3V$KuE%`;zq&PtpfJa=CYY;(ipRKl02aVsb)YTbkrQVLaYqr+dS5IP?Y3XlkF=^xJZCOPlGX)B<P7K4!1^RqFCTbNSsnlYaZMOvv#n@dNpMiw=qmL^B{{0)D|IEQ0q;xOC4IKKylHj?D1VH6@ruKyo%Qj?YM26;ylJ9lfblcB_Kihpk4%`<&TF2O})acwJZR0;l5@YlW=Y2-XH?uavaas@V){O#TuMyz8e55!c|rc7iOf)Zy4`j|3G&;O(lLsTaa|Xx~GPQHSx!-I+~h+q>@N#29<<tSRp-2QPTvfc<a2Pj%&yCil_mP&wX)+Y*rw7wet!UR**d_!O%dCSxX~VB?Fo^6n2F=sA=ncEE8XKGc8RQ@_qq@S4v<bF4$vJN`~VvRymS3fK|9?H=IJwOEN{~RSKlzyl$mBb)K~1@qw~lI((3xt+GZ}DZO<Hj5sILuw2`F#(cPhum<>FbE9fvc~rync+j_TYQ0r;O~EBN_`i`<OsKP<{hfHn2Yo@jK5}3*pc~C)lA8^_ur!kaQrn0s)D%(aAs-CQQnyMJ^dW5**cmBOhxL{TgjYHNnR81YSW~&RvI3t|_rA&7`+R12luq8Qa@~5*2X{><;?u*=9y@4D5Rmwe^U3}17w$Dk!}9=b>qu*h9MmFfvtF#X-(+;^rK&O*O*Z9QynlplkbMvAj?ye-l0Ej)2XDjsTH|V*d7*)el>&I1Df_KlE)5lwdTiL6s<yhs5WrqJdB=IC;!P?CL2+8)OJ{Lgr9JMl^3hNWMlO_q5<;$v_jVWW>#&+isn+H=jVXr_1<JCjA2TyKXQ+tf0+?Q+4&u6V=Xe+XQai+tlcq6~m~b6?$<AC#s@B^ZHWI!K7*y9V=;AI(wBD#bw1b`0EPpyJ+)lk`5u@wOy<USUX4b^es;biI{T$w{-lc4Um{h^+R`%-Qh9mig+Fca)b?Uu_QJ<c_5v8=w*Sfx&KL5Y=MArdRLunR%MOI&B22g`bhViBXDe5%jXxiY0rkr;5Ffu46;N)<l2}f!kT%NO#(QNx&Jl?EKr2(pLt3%!MV_75@C|QQi=H6jioRJ6BXB6Fd;gs9Wr*1blck25No!@UAG0I08-OZNK;yoR6cqj_Rqv%stKVSJb3c4ZP@pVdd@m^O9`x@@QsRZ;|PoVQY2_Ck}-otpR95)p?sd1b(c!qUjWSvsV2<~6RgUyy&?JfJHmk;bgTtG)9{babiIuL#cQ=gnRGW$bgMH%Z~R~xc*ahD==;-Bx5&&Ax6XNjp5i0D>|xDOnl)|Fj>O8p~9aHQi}F1w0Vz^8IEFONygEH{^9A<zvV@X?s1luM>4JSscOZDVS%qpQLz6w(S>0d}c7mTHotYN}(JqOQxk=F>;B(gqp}RlI0s<oNfUy&JgAmCrdnuiR&<h8iC+y^h&rMudqC9%>}2ajQg|3<WB3*FtT5$H<y$F!-tufq~+%ao9{Y9x<#4tf`{7qTsmB_ul+^vH(Z?;Wt;We>UNK!++4Oe|Giizb1CR?lGRj_WQ#(sjA!?XD9*5nOgy=U?%o1dha9{G$gu3&4o8mMHtl(s&Q2KT^d4LQ*GERs~0@a62fa-OvMOV^Bw7M)-i8ab#PWY($RHfr_EGcuIgscFr5M!Su}?7IH`GHW0Ip!<1Pl_rEC6@tUz}{;8GzY+`187#VdwMO`SS3iKdqVc9WQrtVQN<Yk_Y%bv;d^p@_mr@g>bRq7O^Kj?6bY+mK~~-w`02n32gn(h=`=c~Ey`c%<UO!>WT;>Z3}#n`c`0j>rYxFp14)ysan!w%Rs~tcSfvN@fZiMXG5)=7b_A$qPp1#i*{dHozy9J*u>2ZQa$WII@4Vp;@fSrT>&EViati1)e9!GF;oM4>~i8HOkV83i5^?&JOEw#eW@>XwH=WqgrQ;0cX7weB5sHsFR(>=jT)p&*zJExb!b7ULppv0c5<Kza2oqKPVRA?K>Zg)NM{*etLY<V!B?g3X3+7C72YXdx}XMkGOPV;IjlpRN1|VUQXjVj5Tktkx$vy()2U&^HzsydsK_L^~3fpNeHw#(r}*qX=YR}-hEeJcN{wJzAK$Bw~QZ0-Hyy9$YXQafMYVP#{*ps65H3U+E+EIi@-SyFzeDUWjv9scNE0YEx&PoP@X0(W~ePFtNG<t<}v#VjW#kc{5v<pg@;I)!<Y;@nnS`TsKA7JC<zpqw^=$ZEW~QGFqcAW#IP}|dJu9qLrsq{*rBQ}LoEO<tP=QW9_4=hZjqi%-%6EBAh#X9%y90tiFwN8JEgL%J(iGDj-<TX0yH!$QCC}!ZU21JDijFR|MS2uQtMS&tI@OBmt~m|^?lh*lauvsZ1xHhRrF3fWPu$Cz`m{Rxs+RN^f=3sO~`ZuFl3<NzT2O-UWO-#Vn&EzpxR-Qhi3_8`#?~Nl#$$GXR>it7D2{=uVQ4CHpxwSBs+u2Y!*A_RyjM4T+hr8b;FjrPD7U-aIq@+q&}^$ep?&AE}|kBQxOBsf9xn&5;aUIDH*!7yj9&L!%~8Pwmb-M^d_^0?+%BIsyTeqhgZnLIg)i6=^h%vFi$G!B7Nr738Ip%&0tJ2^Lg=D0a-1QK(1)vzmRs%W|iu)HnZ3Hh!)uacrN!Pq0>fupPyDUaCU2{c%CMpo<84qszV7+xkg=6o^I>$wJ5I+pDp=%6I^w~g2>31D?gWQ^teR(jN-#1nMwR|aC0z5$syOQ?fHYAfgz-#@9sa@+$8Y+G6OV9LU`n#F2k1|-Hva}z9W6<gDSad%%ZkM@ro9W*0$Bg0G%SzBU)X3dEA&sZ$w6cCK|(40~AnO#)bn)q`+B2t;|%z6xJ5GQKOD(LY`h9$Mq|M5;bK1P46%0AIwnd$y1ZAfQbu*UM7mgxF%NhHP;FoSDRy!x;CZ5l6RLT<XzQNw|1x%QP&G=6?*b4AC0OGZD1V%e%6T^{R8@IbYp?kD8*|PJa&1l7lxqxl`0OYxF(twdb1okO77AIMtRO9=y+3yBdXPJgw7bjB0)yMBlH{lK*fSAYRz@jOP@n%I%wR*IwfnbfG3Av;Nfuyb2*~bDB?5WnQIjD(AI(MQsaP7oVkcx<6H&G;~6?=h=d-j!Z7Kax}%Ks>&W?zRA&psu8=MF`bapvCq>X)Vb>;R?Du`mfw`Ic8Po^WNy2Ytz?bAvA_I=SDfl8n85KQIS0&?nUAXpgWJ}vv-1#Rgs?C{iiL<au^-6)xO;DqIwy9Q{*0_coC{3gom%fpTQ0JjB+iptA-g_}fEJ|#{l6WOE{dpigimb~0w-Wz8BDdCz09^#{5-F;oHQ}yvdr+CNjbUBnm-*=!Y@HUQG+7<<ZCD)G(-=7e7#-~0RY!rK(wyt<E>`OJAeAR*_8jRiow8eJVngK3YJs8y5no%?yJ4&yxfa1$tOF#_9)T*U808Nrn4~G?D-b>99jX%;q+C<Mg;eW$@9%wxH{;y?OFqW#?ClX_bauYye6Y9I&5Qgx2+ETP0%EZ|C);}GK3$!YltSlFlB4P@L1h`NsoMJ`S~*jx<*cmu{!C;OcIQi?)Ld-nV7W_*xYlf+UhulA)UF`4?$|$t>cwi#R1K+($e!OPpLGvV55Z_Y<kT^y4?O*}Pjz)|<4|uBqTu@NB%%%*l}EC;y4fv_oS1qpZ4Y$|`73z5|83<#+#x{d0$`^bUrvVoSRE<}?@^gGWw&9x$4Rl56N*gzIna9X;&HQ*`0eo00B(ZV_0SHoG?jSgjeySRK1tGX+|}F;vSE8X4CdDjqm{Q~$z1Osy{xmGu3dCET|;^|TIDP4H>SB%9)^+-+mSZ!m!T;tZyte(aipC30_q{5N?GmQ!|Pa+ca0!t%2z0g6$l%MkE;)(ju3!-0!gwP)e-r~)w`uK;8b7$fe+H?OnEgNqo7CXH_B!Ly3}?)de2;1!AlyR!WK|x<*r2T?!t?o;2lH-Dw<8IRB{6_hoZ^J3Yq|62SiFwV=i&xNXQzuk#W-oUZ{TcgDf4})f3L-PGSfc@7g3#WGyW_(c`F&dfJTA=|Kq1ZH#K^e*FPCf<|+iaFVLO<g8z-be@WoIIN5&&2|)`3MlC81p$8#2K<J%l3s;KMf-{|wmRO<V6z5ShuaNYJg)hGpXwj<q{cw>v?GRcY3+^<Z_qaE*6YLZ)Cy^ka;`>H(Fxj=c6h_;%CSpLznjGx>GauE+@qN4E=dnIina}LXGg5K4Am@z0MUS0TW?o~>{}QO4LZ_WvRU#nfuedlff8X+-|`lq;>d*>K-5qya`FjO%CNfJB*Pehf@{C&$s%^OariOYzL0@OR9A6)$&}x=>xF7$Z<YuVq|P`K??CJNrwph{{)dj;LQ9k?FVDtFD>Y4nz6e3jo*K>JJbJME`SZt)F6#2zWrsF4Vv)F9cBt>nz)3axuKV>}aN&VUP`gk*1KsGgpMK@wdY-*Q2p)%L%s5yyPRWoiZIqOM?>6bx9;i$aeyM`F3YJn9$96bKcURpcxm@L*Y!H0MM#I=-CtSM|n0uGYAV^X5Zirf#IZ*biuL|F{tH{;vZC^~jr%)G#{OYWka%_!ibe9OsjOhsCYsb(MdBjDubjMW~XIBN4@d}ut(ihlubJw=;8+Z5RuD*IdSZ%{=9W}tV^3)*A+1w%E!9`=|^`Jph_HBh0q7XKoD0CD8LxZyQf;iDkI&L8|Z#rD}EoG*@9qp@r$95J^J*?PHbS!oNIf<f?+Rz$sXeqnFR%g&f<wn?VdCOFZF3h1diW(P?u*c>k&a@;l%_L~?OtpHh$*90~qS%dE+Q71v2JOp*bsgU#M@=8gv^uDuM8+&htx@Y$wMN@YYz2a<CY`@~n+Qe2=xo{*mU-E=U5W`5)_w7R3fwhuZ0TyF)BmET)4RfCJP2kJ?ClWgG@XdkA>(Tz&4i>whm8(3D`S;PP4cFmQ<KS!IaE5%DJM|p5-s72vRnnYCeLruw0m@=nFd?e_$4>C2lO7bHs!{;xsnpZm2Wi^p9qG(VR++gZtR6h;`f~0?J!|`zsJzFCtIDaq~<%F{%N62kr0^MgFjI-DXnA(rF^}Oz_hbhchE%lD?6#n&!L2Cukk1_{|vU{b=G0axRi_|SD$IkBpW$$mAbRv)+<+#=(TH95X`nAU8=|K1@fI`MRCH=HD5%&@Z~GKxsa~5nJ2nomV|TqE!Q@?6rD>YUQ$OxpqsT{Ra>pn3eD~ED{FjJJJmgDv%(5P>orsKeu2(nANzqRR<_!7U0wIQ?b@pfX{&n52G~(;*I1}7km{bY&!`^u-bNqllFwo7j`Og7v3Y1;SMHpdLw&?1^f#cQLj1sl_~!!P4^}0r!`6XNnd?K_+8);IW&lpY6=J-BstK}R5u1@uaB;G4>#`H;U68`_m-Ya92d@x<(<+=nw>2T#^FxHTkM<E8Kkgn1;o2~8hl1+0kyg?FS>5p^=}x?+T`CnRg*<hK3_`|@*RvTLUvcKY;I}0_dWtv2opD-R2jp1=a;B{~^(mk5H{9UK;GvL(UbKUvL3u08S;23xrBj1cos_!UdP2$ku|0vDj1qhH+7AelS7F|18TTz5h-2A;FOio6`kpEAlM>k-2#UUr4r5^%<`VYNM+TxQT&gJwdZ6BG$!7%x5mCCBADM-*g`>5mRlG>}xA@)-Clq<L;WU(^RLYxf*8-Gb5k-PnpR0dcqlRG}Mh6f}fwv)1L>SO)C4W4)9%g-Aod~v;4wzm=OJXW(6(gI@@_q>l?pDU-4Up0*xeJ@N-e!bwqlF8t<VyDSQ*<fhCPnK|s(@4tjV?fp5w4PSP275ENPU6a+%d6wz_T~vEDeYQGx_D#MmyI!?nJsI$~e@*re5btR~*PVa^KOle{cR}?fmh_|LC0X%bmVj<NZx2QMh{j+lf=6NnJy<Dti^cuCiWO>L~>QN4Ez&tWGcYQ!x$XWjao(Uip{c-BYJ;{u{`g<MmM){>2;k97%Td>YKQFJ;0FONcGbt-#fbNsJFb~D#)wf(JPYE_3r17oXOSeUxj$_@0*|F-aw83r{BWhnUL3+<IS}{iygeI>YJ3(s?G_$>I#c?u3rDMbMWE^yMO)S>66F&$~b3&kGpcQg=YNKIp0C~m6Kk*`t4G^AWPY${30!UTOrc%a!s$rYUA+xe*ld2?gXp<Nz8B78uCBlv-|ynp>@Nnz+<_aw5x9;6YvCxPBZmxlNt(LdfDggJHA$@KYse;y9b}%|2%m9@Y&PnhyHTf`d2*)t7d!ae*v-sHKG',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<%aYs3wcws#0n0G~P|ZR;<o2{FQ4U#>rxk8X3Q67ViRv<gMS?}RNP-Q3RVtTjeb;w6yNQ^X-7ID?tLw!qJTd>_KbiAPWC970CAIIkqeDdi`8;{@<ay_V2M^w_Cd*~8*`6=c>A_}lkpz?JG%1Q;nyogIJT0>IFgVY%>mtd6MKYNsc@d=R;9c_LHYv(t5M-M&&DN7;@IF~4>vX${A3S)_o9EdI+E`Dwd7iAxc)l&Sc>+~en=CJb4~rbydYfg-mmibqw#@QDFe%2^=`tyknW<kUAIoKWuJzX2<x<}E<khsi*(3#5NLG_{8E?yUX?lW56v=E-CgU<2<KHq}C0h65&*?^g{|Tn5zgLrTp}#X}djs#Rk-W8v<fg%5ft{PD?fE9p5S-@CjrnPhqa<*o>kIB`n=hfKn1-(X!Mi_Y>3V=aX30FCT+q*AvrJ2c)^eFlaWTx~pJ!Wu6~Gp1Ot#B%mQKrF@4J`p-oJYDIv54vCQq|GEpHBBoaf2G)$vi-`}*ax7f=E|-}VCdFMI|Vu_?o05F*?U7v*X>m~5cq=>&T_{D^P9_*j3tT0Z`9bQJ$;@bK^<{rV~#*bN?hGg)730p+meKQ9iRzaIQ~AytyKTjSf!MGm-k@M>KEy5z}$;vd#MhEjLEUJoW2q{a9uyNUoZ{UKFdWJL*6i4<1!DaN<!$tnr5JP5;nyv(kXJnB=8JONZ)OGSVYv=?8kLoAB#;$%H5uG4Z6g|U3<OXsV}WrE$0=SewTK!ekKa=k?OXWMcRJj8WaPHq6z%Q9JQaQ6V%v$7cTCMrg+vvnc^hh6k$odnMpu(k=V&pcfw0Ze`w6o64sJeb3f%0&Xtc{*99e@<q>cW=H6W*KaQb?~R`oVFr1#s~s+$J6;d4Gi8g5Mk@_M}5(wO4H>82*Gps@LP92u{_I2@xLvMX)JVNJO@lpa=@)Mg8oCkO-woH87K(Qi!wp{o6_t;^SLOq%{W_+bKJ1Ybd{P0Oc4b1c$}`&ay*WTWI4A+K}~EoxRkLeL$D{nk80o_RqOEZVdO3aF8A$SzwfpU-NhF;HlzkHG^7I9K|eSoL{SKiu~3X8C3G8@T)`rZ0tMtUo6N>oEOMLf#}`Q%h3l*wpC_=)NjM0O;(n0MYh#F$k0~rz)DMy+5R0SO=?;oR6R;`Lbx}^%(<Gt+41x%jW)RGw`?BB1sbVptdU0CJ5pGhFzVV=+1V=8QX|GID==HnFa+|!&^DK|TSL-W4MMlIa(Dzw(y$%Q(p$C?0ASmNWd8(W23}AzVF<xabqHLW`VP;>z$Ua6#@jym$z(w$%$hSa*%DP8JE@sE?`F&&|;z-eO5M04dn%|5N<N7YjNqd5EN+cTnId!JXtVpy*qiD>ZYyBt!WwcsQhsgXOpv|CojAVl73Bmx-PhQO65Lu?{#3mySWN2w%#zX|&&naz4#Y@1x@q9UpsOvc#s^bEv&$?U+Y!EH9EE2_LgVz(<ySj4t*IseLZ|s!tdJr_eDMrxuzOK227X3P=>+AvD37ruL!DKz-p*$T`1)7buVYbfkIsO$42R3E{1R@!0doae(LHFe2G)ZQKu4Qn(oxzkI!*j8PGiM1akk_{Zji^{A$tJ?_x*s<aps#~qy4bESfmjtl>fs4ExAV#La-86}1l9ycA#?NxSBfBfdIq2z5LBGD2~nk~h<flqW1X#+8I<yN-6)NPf-gn^V2rfbD1w9xOFd=8ZuUqAP<@8J<;Y;qVpB6%hI#mVpp|eYPS(LKHGBv2L#`4)4{Ts?ep4nz*!T7(Tl&a1)>hnV5GxDMK4Vf%9J}Y38C7Cn$bemGTLh`yvP@-u<JlHMo`moSxyD~Gm*@ny`Y*nW`}@Lh*fJ|*V1S7TIybGiZaTNBKN-QHb+4`52Ij`Bw;-rQt8^V*4Xh3igOfh7WmhO9)vEaxfcu9Ao5urEL88BaFxfSnV6!v_{=OLfcCtijptPTdQtxZP($l&eE5eQRhj(>cv@}5L-smb2R!D5l*`mz1Q<C}O=aV8q#(fR+KD|m_fDjVt%wuW7J0?>In3r;k;^8boxJJJ?`U>#u<X3bKeR*_*3+*lZB>3ggksrwry(^MEVrD3k;qVdkE`dS^&*ng`k;Wf|eLs32z2B4;8|3fN*Q-}N?HmORf><Az=4uo5U2dqgUm0Rljd2is0Kr*)z5yZ%0Jx^dzB9raOKB~S%P(pbs7=R}97uX}>DM+Q<XgUhBG@1}55pMcUnW#Fbdkxqj2^#4Rz4PC@uj{&c?UiZKXo6lI0Y^E8nBpcC^2Xs)%*9~2)O+;$DvwX4MeclurFYfev*NG5EAmK<C6OO3~P5|&$IQKwHkZP6_62@%5!iz;s!FFXG(PUltBoQo9C<y{Tj3=iBFOn1h=*rH?()Fwt6QrfNTX@;#SqdIw`NST<~vvnc|_Y$^`@mPk5G;z^miL1WJx#!-oTVpqGUQGQhAXmOPU1QyP{fZK9brKji{?mVHtu?|^c&_kilQM)Q;PO+?B~0VAH{^bCSH=!$?sWy>wL&Qk|k4D@+2n}MEVo{^M9OlZizf-(!J$`x=Go*~TEMGb~9s#ONc&$JK#Ky%h`P*p`D<1Cn(O%b3b3Rx1RpH_`lz(Tz-w^a3QH3>~6<-YB3;ZIqi_SJC~#?z;Ojd#X$0UJI6h0_?-*HLw1WV0L90U9>t_ELAm5Lh%^;L8E?YI@|tHeP``<&G*+S*V6>oeqzWPJR(Q4DipHv$5cCyG)^fAV`~%if2pUaN;7F<kLk|&F?&X5|wGWO#ZSs?*DLp`p2hd51;-}Jmdz7V6{bQF$t3O<Qz~4zC1bn;TZlr1-zmjg#$@GqM{YlTW0p3(rxtXA!@+Zvg$u>fx(>p?AQF|hri&!mf3bTUrzF*|Lec}2}d^xR-h_bmsldcJsv!|gD;`omlzOcU<|<4{sMGtdWp>OO}5R|mf%$PmI#<)iai3pp8oO4*~5N2Ok@@E1eIz44q}rPF(VXm2ql0g{Ob5uCnUX{hqG+D#p%Rco3Nf5btpZSjm|?9fe}_Q_fZwbz3F`3EGJW|F?;{hv9$#4xN2chd){jmy2?DkwZ%d)Rt3rqu;8gU1+QT50n(sWH4t72>&74P6pF&bL#%Rmh${T3e+oxZgxeGpuK93SO!IUDN=%$*?Mr#HO!{ZYbYjn&8jD$nIYL<mf`C3gsU%Y$2THR_Xqa!25zc)KQN0hNPZrAHa?8{C0f_x6p1(6@mvyU>GT-bM(&xJy7hW{?l`|At<|nk3o;tgs1#hT4=otZ#SBT$aya00x>lwvT(p6XbYVJ}O_Z$!@%aDVWC@B^yPex*t7@)2`9|NsimzAmE1spn)d`9L3IlrtEVFQ#UxL%+E0yQZ~eucC7`oo8}#ukBWdt&SjQ0gkpwn%cN+gOKTZ`NX9(Pph9TM_?mZXyt5(P|Z$ua3Y7XHN-3VbVSf@H{So_0htE;F87hOFqU~y^CluP4W`|PPXL&|AMp%!gI2|3Wf9oPJE;Rf`m3%;Xq+j*pE9JDb^Sz*J=(P8|gbBGHPIE;Fbnpq;QLEy>g|~_5)Ql2f87)8&dYxAm%M2Mn5!!+=k!63LiW}1Oa}v!j?O?1zjd-K_1;Q0v;`+&~nQVs7w(CoI6&8nwd=nLR#4bG@VySQ*enDAM_ZnC`loxA#Dz<in?J??>dGyVIKh*uJ~)SagwWKT~=y#V@Q&2T6E}C>of$WK0Mn`nQqtMnXa>Ik5tmDDcVn9^>j=iTNvgG$a=Plx9b$?Py|Zg_{+YTmQN#(S_SCRhz-=o7eRbnQ=>zf3c@mK0v;&T=D?)^okD3Ud@j=eA<EF%d~RvTRdeR*%ELNOSRLk0H?(9xRWq>3Qdc=xyl13A9|Vth{9etvG|Q&No^5ym9b1;qJgZ+!&^bYDxz(;PinxJ{9St)*uM)A?pj5+lM%ChRcVp1)jY9qn`e{R!-!Fjmng!&TAp;?-7ZdL+om{K|W728C+GwEO(8Oj-8rB3D!b?Mv&y$OE-G)SFO-EAh3z>`n3&*9jPvHO*KZipWP-mo=wk@Yfu$9{%q0OPvTf&nEE^x@A$tpC~=@2naDY=1u*k)k{>P?^&3uvgFhrZ5qkyoHy0(i5<Hv#c2pVFB+udR}_$uk(Un+(YrV{(}A_rdFX8)7HupMWZO7;k&?s2_mo%}@tOiUa0(rH?za4Vyr*te7F&cE~n~%XR8}JhKiB>ph>v?4yHf16o?4JbN8EVqyM-<+4fh=N?H8*NppPOzL;=FG-@WYV0IIe$2M9QPUjOSfi#Y$!i#?H9at^z^yn7z~wd>AQK!j4E?5JRJEE7yG5JPq|sk(RhXlIwc-AO{QwzliimT2*KgPLx%xE#uAVRk#K+ZPq8=4MK#u!f1?&ASX7@D3zH?3z0OJC5Y$Y21SfHen!(-^7g9RB$LjR#|3uF>`Qt(cG(K{?+)4jW1>yju*I#Yuj_0=4LkkmFR?JNZF5nx^Mp>QXTvU=Uttb<r+<EHI@U_fwuB#ICDGiWt5?D-%2G1wdCb6Yg#Fg=ZpH-!FZ-C|KgO;xuTfQ95e%ciZi19wq;%5H;XO5sS4HBw#e=@2YgXM@0~$j0P=HA4SJR_2b!sD~Vn_kdNUCm!um9eZ%AO^I&tU$AByJhGb1S7U=8C|(<utVR+PY<n1n0k0>l<)}F}NgI2rw7X_zI^j9#Q@gcU$Alc1(#9uEim0wApCe+ljN!A7U_Y>T>km2`5O_*`))ZHhC+Ir9feqUV7_ifIO97^>&Q4-<SNt?@LSjN?kw)fMtNQ?NQBo$=WSI_tRiwTl{j|tM_dM#HT^tU`!Xr2@ltV0H^71FN)wi1H9b&d9%gvZimysSa=6seWMyRCe)Vp_6Hdq~_z-Nt)I5`|3k;KDab+Gsw^;B##C)giM)P!Ew5yEpm0hYkS1p%lWU(XO;9GdeS-9G0JZk26LVOj6Sw+i1`dI6<aReKj-{P{tv^>9*VD-bNmJ1Me#O11j|xvIzN$wX~}Qyr4p(NTZL%4j1a{Skw!{U$5Yk0h+Y%VIG(`SO<|-@seiu9Hb#@mA*y`AqH@9<B$RejSfrDGmx-Xom=Ale#rsg$-TvdYieO47+wLdGnEW6S{v4nxbS=c|_o7Tt~iM8xe7VB+B*1bi0p@VH=tHM;s-4P~6WVapz!nz3jD<aUTs)HzwTSBS+Q$T=xD4+4~_XwUBig0|P%&-nE0GeAdS3Y`9MM41nDCs@`+()NG_jM<7@|Iy%9B9^pU#5&!v$%`^@qzEP^@bKOMs9!3p}G=`*N>!p#?s)4h9hZG}<oJx$j4e`ivWTi5LN9_}yttEvI{KG?MD;<Q^?d^q}w*9ly672JfKdBL?NAQfgSZ${X%%2CpLt=n>Hlh#g4n!gpE=me)p4#Eaa>L3x*3zX#d=FKE0G(Q)J*GGa;K?V`QjP#EXYUeZcvP#v_`$8v7HN}#jHu%u&juusUN;X`fgjuk$a`UAUfCR22WhOz_x2WV=8R&uf?8wC!k1d!0jz4P6%PCN@cX8k%6`p5T7T0J=sTkDab@7E`bAvy4JE956M`xFlb5-Jrb4r*LeqAC+(HaMd4sR}13~>!D7_(%+w(Cf-~w=AJWwO<z%A*LcAQ~N?be~h5%nq`;_BegqWo&e-|lQ>LQo>s1EelxA0HvJ+_MjBQ<`kVO|8<fx0mRlik>m6p2kM?6x#UD!6m+}x8K~tEs)3!Z7_q5JjZ!>+f)|Z4TD>FaThu>cUwjm&SK%!<=5o=QLBtkb_0~~#lYKijB^-UWA^Z{&=ffJK(XF2*zhYc*%S#5Y)i`W`L=Rqti|1N+ZZ<i?j=(WoY`S^L45@n_wfse5H~@-Wlt+H+=Ghl<l{%=5FQ(AIr^|{;B<u}02b;d5Uszlzwl6K1a{z9EIwigYkJ8=toN#C*Se!X-0hKKN^uN7TTQ^g2h-=Uvp#s{jxl7=Hi9<8wchCrr5*J^nqU+QM3$jxZkEo0qr=nLfWhfYLs?dn{F2<IaD0bBsL&g`BjfqZ(4nEcA$=?NJ%-kaMaUHf$mR3e!TZzGg3$w@C>Dyd%0*Agc`Qs}wbbFzT6b30{yZ=c0QwBX86%%($sr*7JcSTjgfUo4Rw<i7z^5;TvmD)$0ue6@BO1ig;)ew&vQqSbf+*euIsN4gvV`DJ1BimhUqGTL0y)L_+<d!1t*^m00`WxP8QboOj+t1&AfoVk0qC>Mz@1=y0dv`+v`7(D;q~=(-1fEKfT3-f;e|m#8O)rM9EsWmhi>wx!%*AvL_X5?>nvdtVXH3_)acUP%8k|^L0L7>5<a;jHnJ){p!mqn@W`rg|2|=n-vBTmrgyTfR@<OY{p>X?adWQLT6pFvk)l?i?l4ggt+whmJebkdKpB1u43(V{F;xPwD;)hiH1o&S3Wn_ny#4=xT%W#n!F?A`;9bMzt36Qoir!ygWN#I-WbXpyF8lWl{N0V)KQYD$scbH7d%!hUari)T0LKcnEZ{o!vr$c3smGS8EBihsx9fc6LuU3EmFPg?*U2><s>}gr`Hc-IjWNa%=|q<>!6@PbwmQoIkCxea6h4%*f4?JNd<l!LapVa)Ccu$<nS#iVkVK!3(=+3L#k<+s*Hzb|H?24nWwcsZzFv(uI}z@+cxE1uY9j2$>Mp7^wSn<wY2pNh-rIu^QHQrFwB1+Ek>4O`>l2zd0~olD6!%7wS1%5~efLcOOb2Y(>B7c|`j|wsLSwL`INaEFv8HV95q*V7h*reu#@6Bx)i~<2zT~G{71XvVcU*C`M)@QfI%ucH8C&R5wSmkpc*9Q6h5=qR%24jT5YQ5%FR*d2*$UYAx;U^+>;)lSjg<2lOgXBCfM$pDknF|4(k7FY+qszC`BRMXC{f+WPCgX&94jC8rwxm|?L=s+DG@PjAhJd?Ji8Yt7L#IeNb%Y&I9XG@*{IXdd6K({6QA+<*z@z?E&C-W$gLl!I$d2{fN#$p2N|gC@@t%^IiHpZ5Le*TfhaG@3sKhHQq@t633wtKf0>A#Mu)xj_&O>vAY$z~JLjZhOqVmHnpCtUQrzj4YSiK$m)>sCOV96nLj`16bVIe610XHX1?tkB-AMI!ZAM4GNryVN!AI7;8<3r?qiNaKO1ikadZG4M#vQ0{(5FrcYBa#fd*RXp9rzAv;M&)u*ZWY18qAPVXgh7xBC277PzE+H)JRFy7P3UuA}TfcO!#)wmz@?<UiP|;R9o+|lQy?vO#c>9zt!+&Z8)k5*>rHn$#(Cwu668Z_hVhd>fx8`N|v3>m8(y){O4JHpvp68i1#O;tvZm@UPH)G0$u_ObY%395jg=i*(M@KdvZ<f{bQ}_pQ1pwPby<O3?=Q6&BOIPGYi=bdrU$6;9?^|;$dUD`zz!hK2-N==Df#fU>n&wv0j_WQBuwMJ+iwjoxOIK77Meu{`^55;SPPc*2G+A!;1WJPv?f#O>l2zawXZc^BO8SlFz%!eP&rwRE3Ld#Aa`NBMc0h+T9gs)rqqdDw(YuO*e~utmsD9c(yaZQGnH8PgdIMXQTi>6ojY5aO?vFk{zfsW~9cQmh2t`nS2iBlM)VgLA(ItkU;!H6Hj5Z)oKm_pT28H(x{Axm9d(14@g5Q0fd}x*1+$%Zd0no*G4TbE#7n7Aci`V%5{CAQU?d}R$_=fG)FqovDsze=%$Q4u7qQkAYrC4PBj-=CGSE{<0d5#TlRG5_PBIdX3##7j3&-7f&XnwCP3HJl!Y79igBgWvwHwJR4b*`gYC>;^@PEqQYSP4pv`z~<h5_cRM^H~a!l^^B#j{A{@Xmez%6c2_bufJ6cY-_yqLLi1@=JFnS(iXyMhgZe8EO)5_6)MO8`LFoFmJSRM^|d5jc=A6WC}_aJF6ICS7J1aN+_f8@YlkYTwX|X1@1zjd8F54D$g)Tz3glRbgXW36Sz4cS`F23pW`UrH?S9#RqH1^jx(#>YSF_q{*$Sc`pR&@&t<nCpj($y*qL%GZH7b>m;XWR2#7(<^Cmk^;(O8C*LJH!fR^O7<^PNHCrAnFk{C}JqaXDA|k`oY`fYN;HAP)3WqLmLjw~1VqSFlA5_)GC0Kd@&92mYrqK*$SY^Z=LsEwgg#*c8I=e8-sZCnzEBPh#3qN(YCTjv!ID*Sw(oRRFu)eHL6ZMb#HN|Ah*A9K@-{EF0;P@I(ifNj1pd7k(Ew{zOvYuC%;-ap*1QotWU>}o%>HyvIv}3x4hc7!)HP2&LC1h~elZLU{ijXAlxcOb-dCEV|T8h=df7&=9Y>geb4O~LJCg!*4ZjKoN^*Mm%b{stYX8aVwq$LH9P@TpQ?zeNvhX^;5S&49;01{!?(GIM0k~e5;PG|N9()RAtCPxdIdL%y#bkiB1x4oJ|hQV&jVQdDJy&x~_0{8+_WdQ-*bFekcu$rpl%)Oe^J>7yTXffEYL5o)zwsSCyVf#1OXv=QqKJ-TUUUwJ&hsL=K2Pm<#wS#w5Bjzo8#GGu_X~!-)^|P*v)HKKWv`H`b7y)P5LH2E@gAF*=V2}cOynd;<YL8<p;qoIVa=O<M##QB#)VS)=@mG>bb`0h!NqOrHs_wPLX1a;C{a!#Yr?tqoQVKzf&MAZoHMY^)7uus?nqv%UW&|3l3y|&lW1Gu~;k>FdnrNFnwrsNk+zO!GmcH+~xb0zH<ru2-phaBCPN+WAM3SiSJHf%QRo}8OQ^KKLw&jsyTz*)KmG2(EWQgJkNI0hspZlRNIumOGb#u5Kt`B028FcmDo8CrdJze17tFodtWSRE|3Mu&kt*)JusUnyyL4Zx>D3g;6B@WrvM4FJLT}2b%7-lCgceC3URLBE0>~x)6Olpvm?bS$h%NQzO>v*b_o;X3j&B187c1Z|Z!#=GM@z$`ml<6JuZ|Ld9E^PY3!rH2)QmC|rns4tXNEYw7wo!6ok__b51y-Fxwk6e_S5d_1tv5zhcr+%@z46#{W`<I_tGd3anxSAo!CohT-qV5F>4t%czz^-b3CMQ+BQht1E=-{tCte;?f}&4eTUSi0+r0fe-D#G&Av;x6iE~5AK{7N6Jv0Kms6xO24MM^Ylw|w?%;Dw+a`TZ?cvHlJ2`CbRX$mv3$OnBK1P<cDSNrkKkTy~L>(w0$PXUhlej1M<+m=)LIqcB{l6}SGt}Xpm+6btPK-OqqH<xz1M#tC&W?+&@i9Kz}H{~9FDSPXVZN*TBN6OxQC9WTbx{T<PxXazor^LTD&y7!VmrIG-hHaur!7Spz=X5LKZ#x}OhQ|NdQLy(rA_O`r_|GM4MGXeXuhXHFKhq2+OBRz*Kd@azt&?rKD@u!+v`f^GBUhmr)za5k9nuc8`sXBh+NzLA68vfv>*;KqdKMeXBf9{{{0-$NtFHQq$Rg?464HvL0P&};a@M{zca@eBQMwBI<1ulzUEi`|B&4I_dx>JbNl7%9G)H5&$tI$<EUf=|fVbnIy#tsBnHji#0B&Buq|xpmwTBdyCETN651bwVU;ylv(t<2fiQU4;ElM9|nPn@j>5+Zg4WrJexox_gUwIm)AsPQ@*VRG0IFg!l<35r<sTm{3L2)nbX(wHLp-~Up!fMgq0`)}wp(nuevk0!$kptrnDr{Ky<&4s+LqSUR%ZA5$I29PrVolU%EbNCnjnmM1Uk#60-D`)X(QC>r;ZHfnSaFYxWBcV!smaRjqazGko~6FEFaq+h`su1bD~%r0$GEFdhdR`^_>o^dCb8#_1{!*9iq*3??1q+I+qCZw4Rgx+Y<i7d;mxjMO>!vf?<9t_%xaZt+h~BQ1UB3C7}XUVW`&Zx=q`BMTi54-O?u$DhL0pX)#1or@>;?apjM78xy}z*F8mO~O?MEx7gh9nAe4>>3H>y5q%bZvXWjINJHdMP)qs)o?Q^W7FX&MQ2cQKwiL?6+<hE!DbNzXs0H7=UD9|4R><b!*z~M0;Sl~qnw2KapLUT%CSv<gMjeb}pCKNyu9IikdXJsC<2H6Ue0AN>D2pO#^x|^Gm%WavcE|J_0pb4B>!8Aq9$g<XioAJwws%B4%O49LKIBgR)8f4$~7ioltG`QuxnlXpMndmKnc!PINf&%D@q8Ry($_@pl<lYu#pQCN;?N3}ayW_rE$I6@I9g7i~iCD7if@28`4^R*7`deVu-fqu-k30@ZR1jYVk!cu}F5X(k60Z6anU$+L=@9E`@B&3bUX)&4%0HjZs71|+eK%w1T<HdBJWoAdPtVhMy<ElXbP-=<SD|05Y`R0xMO(ZlpnLQKarmFmg{dRi+O<f1<Xr|3u4Fj}d?HnNM>-GpSnrO1B#)G8l)DrMc3E~U=NDZIsQ1DiFb?ghAuxg_iZ~tW0((RQ9gef+0;`U7Lhw$>Tn|o-lhTB_9MzPCCez*y1w%Ryx{&N=9Y{9p_H+N|QeFI{VKJYh!e}ZJI;fJa@zaqO8Sa;B#GeLN^VeE}x(4ctJ79hLc4v;U6Vzg?8?tou2HXt0cGjF-=Tv*r-g=<_QRgu0Id1n#VSA4ptJS8{UsimJM_GSliN(kJUgz4Gf#sfQTwA<_eK}p!qZ+OR;M|wgYIR%V{+VDqlTqF?F~&YZq%8Y=^2+Xv`xL_MNhzWm)sYHY0^kPXp$OP5mov-y`{Ufa8cZ_hr5wgeZs2G(Sr?NzU^CqE1Lz#~`#T7<9wn;oa!7@7rAH2K6yAd@P=b{j){N{)G;s>kxc6O;;`%WsJhDj<sh6CoQxmUA#hDW0$sP{k#-l<FnN!yX6S~+NlN?mO$iSF7H^aOBRxK;(IvD%c)$$kKXXP5_zd6m*`Oa9$rgN=T;XeMgGT@rB{64MU&qcZ|hV!4?OPbw~t-*hyC$f49yO5~6I%%(38dpbAxw$x|IwgJ5I<|*h--lK;=ehP~xq!!!n(u*wYZEl7tGh#`@Q&SVFN8^>gLJ`UiWPP?GD5M)s2~i(VK1E_e}xxp8rOfg)uFok4DvEA>6#T5agxb&F&5nraA0E%PoN7}mdmi)Z-Rju{N~-8@87?CH~#SQ{fF`I-@JHf3!pd@l-+O?9!8%pK)@Icl&W%OM?n}OMn31OT4|1V0!dUL?QKsqd?EF~_h@)SS2awTb*J42Nez2;A9J5|tnC;KOKy5UfV5A)!TWiZd}QtDkv({`B)GstQhb<8afqhq4s0Rroys)C7hEr&g{it+6bcBsQ^R`im(YL+?o%y5xn5WqB;jX{8MIRC2z-%oo;r|WQEp9FQ0FkEawh3x!8%#g#_%N_(iLP1VqFfOX?hk`)f{}o1QWP)I<R58u2hOZeR7>e*{z7M{%I(m012(%omX=GF1e{+58K}<l;V{=XvE?k7(e)J?Q#J`47<=O0S}3pSdU+GisM)z0-CS{)I6NcKyC@*!vKT#xXud&?r>Mq2XoW1-IBRAu`x`$6s_;88DqI31z{tg37}Lyxu`BQHWD!3)q8q$hPA4K2EXuA1CJRM6qWUbgUsk7ZAzXV9G{))UO(&Gw<cSDF*QaH_&88IC+_jFKhNMf(KUgiC!=QbpsOWP{EgiA=O$G29idf#W`hnQ+Xmrn$pK&6iv>XJnGC&GZz>$>IkkNZbv<=v>yTJ^I5`L44$qg9^<}7xRE@bb4%@>vV!xhTVRC9rntnaW@p`0Ycx`PaRSk7(szg^$b7!qEVGhdBESUI*{BQqby$EL8o4^0{U&`Qe@wfj4+R5eL|Lwou1pnt>|L_0&`(OXtG+6)rum2^5GVta6Z~rxgEU16~xBp$n?iHRUA`rn@!%1=tLu(X7!sf8+mFn=-ULYRROVD4TnI;*t4;|U|u<2V`rlFy=F+GFA95_MiEhXLfwfh{WaLrx*0SX2~An}?L)M_9wA@7onRa1};%zI=wYj=WLiD~)rO&49=>jTBbtaitr@49j1GxxetkVcNB`O-VQTa&^;7h6cjXG%Dfh1`DeCF;hGzdUN@Wvwlms9Y**3a?Vla{UmURjT`T2Z4QSS3NB?1*xiJjK1rTWzxFBK`+e|*I;AL^VGdg&IM_T?FK8w+f=K6pPav-Z{NdEdV5^G(yjZ`KEtkCv1Fd=wc)wBsHWeFiDe_@jbFK>(@RR;J~osi`=L>V$c#;^d?ld0i%5|TjfGaJ1`}d6+%j6>HtH+q)nRp!F6(-0zWBg5jyj#V20j(6^k_JZ`d#yIFe6fYwlTI3H|dq>GHOz;opbe1lF1_9Wd~-zRI{o(mt!;%wo00FnXg!xnQwK@E-gV1R_+I7V;ZD&I@^q=s~Na<3Y+#@EcyY`JusJa%KQ+$IFkqbk<OfBh39575NG7)<YB%$QLZ}2s<ZaY;>~-tj7riNt%@Xr0&OIS7{Ha4^MkL#9cd`cUjEcH6hC3c19#DXqI(=LXNQ@w5&BI!kCyx!lUt!%ckyO@QO#CnhBf<ml_wsnt4pv+Mf^~RA)O3Q<Lt5)Pz9WiQZdMPU=1U}J#Mw#_wSv08WN%o@MM8zulyi%gdgI4#E%1tvV(XdMi(#seL`-JW*&>~q}pbNvUx;bk7@=#8}DSaZr&58?J=`Uw>IFfjx1iccit>rJF~v?g1bsKw2%FXm8o5NZac601J)|GiX&F4_XiF8k~fQjD613_uEOnX*n0Cpt)NP0zCJ!HvA5yD|9P@{TQ_&L=iz1tJFimmQ*`UqU9Wn>ce`GzQvhle^xCAIM)cRtB#6$1H}%?I?wv^yuEo(vOEH|fo7<e}BfjIa%AbK=jo?E;7m!IAy)q?0-wBX0SSNc3j(Ryf%hh&;WutNOW^n&_g$3!5)SB({bfE&xRWT_bRD7t4?UjcTt99*Pt6NV*derx~$dY(Wz5`Jn{xSOXP_C>tu0`T0@)vT%L+J`+rd*-g|KWU}7;|fIEG^QYs+X=_p>EN@!zO;^pMd7&tlDbN%GtAh2{O#1VJ)kol#{PVpk~e%tb=Y|%I30fd)0`%9#G~m73<O9*sGEza;hyze33IQc`)eKGPDGb$Xf}s5y8U=J;<3c436(;&dsH-!sCxTHf)-Ly(T_8k}u2-n25Y&OVZwW*KFnhMe1GE1ZL^nhCcbvsW+d4Ib=N+byLlR+-vt|X7o1*k>$1ZQ7nfbarc9ebB`MByJv64FTQ>AobOsQ60o#Xk$>&9y9LXCMh=&DGl)#y?_T4m)nqV!?b+|Xe)IZ2jrW*_UFrGP-#!~Z|N7Z~ZXdKFw^i1){j}O@$p28Ui)J%vZIFF$udcHzt3uIp2+u;t*R?u2USVwXav6vj7Lw9qC19TE#&Nj0AcxFLKpc^K*^~$8;5-fK>p`jh(cCQ_pm$qg-*D0JyMz&;zJ+`lTf-iS-V&cuzf%{pa_~44Oh7liL{kV}261+0n~LrB9P8K;$Lq2fIG&2D#@pW3oGH|dZ;z~tW-l^`H7=1`?>Ei)_?q5Ua0QT<axhLvVRd<M93RC+79YV3BY6j}4vxxS{(>$ytlrr9Ikfh9o`S3R!AY|b`E=4~%l=Sra$OXUpdoDrAA!OT<l83yh4T@f@vGCq#rX<Ha;&A``Pek^@$y`>d-h<>+oMKne?V~=U4Ij&(U0bJ)%G7|6ST~^<@nukv)y_65lk*n_K_(|ud3<(0q(h;<yhpz6rooqjn=@{4y?3OlwX}#fae!=fKM?t!mfxftZMXFue1R-T5!Kg^Q?hA?x%waqIcCHtxgyQFKw?1B@M<fsm4hzO^HKIMVG{S-Rh<&$G{z<FTVLEk|t^ijahDzyOx^`$beJ;YqTNzO^Wd>t4Vm~+OuhbewS+p&Z;U+Rc&fO=Z<oMmxE8if=%grA(SLt#@CA!Sk4fsD2A2ewOfr3R3-G@Py(Oz0H;!IZnN*07Kz)$?_IKP=KMI2aU7a%gDpr;AR{323$C2t)q)5Y(^Y{o$r?`$2Mz2H7cVq6jBKXKHl0Oep*`|snAy&<26XUc=#4fb0uNuiE{AvLE=QYh`qiZec}97q>1I0+4rfaY>&1S2sfVixV+nxH!Mb+6WwO44jkaF#tsi7108q%=LGa?`Z(conJ^t;xH?KdueEkAld)IXL565}|z`uDue)i2bqY#exo~zMD2^*Y%<sgh8*)d70g^2_7c~Ywj3q)o~Vpt3gi)12S1Xe2r#5+O89joNqS8rd^dyvBH_wPTvc=PRt{jX|A_F46RsDp(S8HK7=BMFZbLvZp=lW_lL0jeVl-h=wx@6Gk&C^<R2g*}=jNJL;<fEJj$3FHAv*&R;8=5Q$n4hs2^5I!)I5QVyNnt<-(#)3RC=rtxDD-3y&M&z@14@h~YZ>Y@Pe=|5c07*7s(ZuA5g~*7Ybxh!10AlCxtcH>{I-U~A3t8ucgyZ1HA5mKg*_vuZL)f<_*{B^UUc##7q!s7rIhD$1eyAAY*4z%KL;x`-2YGSB@TL8;)@`vjmKu-ymR;1jJMP;{V>7jHl1E1^QP9B9khzh3K)|Xv$w^c4T;C6WC>};Xy#JyPts9uS_|ld$y((UzFlR}lhz!%S^AuCKXmre;K9JH0G+$wRdUeGw<qqVzJ>1tHtlo?z>DpbdGdiCYD=Sn(T*YnW<#Vj9P-LiIr96xJwszQhHk&-sZX-X-gM)Q;Aa4&)CWI`yYf5JJP+L74RGOi>G~|)7&d5`W9~#n=a!P!o`dJ5WW$2f;qAz!D2lc}98**8L(AvSVv~=ZI)tWLO*>A&rmG|sT;41GWE9-t*`~FWTW15KF&uYpQRW<1!h6xzOz8f?9&`wDpk*#vFo`azb(&K6Nyvh$@g!%m1bK&iyzi2vos85IS_vrKm;kIMED<+*LTK7Fa^2o%~WK^{nGaA0(FM`@qE(Rysu^#df>_p6;h0Np#BFn<=qL7*$QTSk%TpbjP)xnk=G`s8M$qLjjQ007k0p2GU2b*-Ws}6fV9N-NbqvMmWjt0LvI;#BHs%5NjysafHd%#PM+pelwc?Vj$7$V%z`J<C?chf+@3LyRG*~e;V4nj4X;P;sEfcY8r3~?&CfT`SMpdjG(1$qpG3B1iiry%;&OedSkc}fP{@FD}=57<0}I2v8ZvDI-@06GNxyN0?|y#HzV{^j`jv*%yG9KU_`;cMf+f@m)WbKIh_(PD}ySY3?cZSlja-@gPdf~glUX`w{2GFp!mqbZ3w^t3@v##q#a1s3Ec-nWh8*dR|lzPyH5KOzTcVbg4{7Ekm1-OFbmUXEYC8Grxk^@}&(-!)KSWKe+>o9b`NgYfLONG3CqzXM&Ik><5&ishf~5!LX1QEy~63e_BhcavI+^-^+s>wwsltWpFjiBh~n!NgYme<vkti{<9_>vohBJ81^`Rj5SiC>zZ&a&(PHO2|!Kp$%9}r)ND{Fsnld7HKPfz?!yhy9S~CQk4!(ZxNY<MdsUYub07tv~Q=$;M20^dJ6k_nO-c)`U&??5zQ#gzhhrY)s>T`S#y9CnuQFfj%ss^05H~8R5Mg|R>gS}gXY0v@>TI4z~<Pr2kG_kBPVF%+cJ1UX2+-0(1DrvlS6*}1iiDK%H)V6dU8mwLZ8D9)gcOBYcAZItus{u+^wKBVKXfvP~6QU*>{PDjfFHjuFAqK93DB^)~yV8dwOkr`Z#)?eEc{l!!(*a)jPdC(BrXd7WWxwV+(c+)l=suANO9U!Z3S`nhXpf4}8om3zD@%F2>8Qku8KF^}OkA403yYTZNR^39<dGv7JullXzs;;dFG2Z#>zwPkgzmn|GSv6+y7CfKORM&_=eTplan(4J}LPhV#b(GYfomP=Wk7%H;$$L(3#suG8?RiDyiv>RS32laJcDLZuo0r_mve{MGHZs95<xi$-1{6J1&tjoj<gTGI-3XMwcr$DP?4^?uuW9!5Ukev&hymNag(Bk#Ja3UYANd8#36lc(BAMb(Y*_TPoL+=FHFP(ym?-;X}D-e!?-T;1N64OqzFEd7pLo_j<y=znm14dvSLYGm~iRrHv|Ts_foMSAG+$u@^c%SCm+0FhRVx`!InA(-9(6F2a}0|r$b-d+xC*M5tyhFtoMXmD9YK9i3S`i}4w@e~t^?y0TDEU&wyMH5j}Qb7APk$%l3Ep@R>k`3nEiX?%_39yJJ#-`ul$y-~qVkBTO0OYu}5fV1})Y;xY#QpRhYP0*_huKJP*-RZ$rMvXhZDhOjl<_Z;9a!C&%A>)u__OgfsnY)gCW2^-',
 'pubmed_efetch_snapshot.py': 'c-p-^%aYs1c6)vWEH3~^9L$XTsB22pEm<1htBNhDr15<e&7eRMoPh@fcmZ%m<GEa<vdB8Ce862~o2q0Jdy$1x`49g|&ZE20-Dr@M?Zj2qkU&39pFVxwr+M$*y|?LKHfhmhN$$Pg%)Uw&-q{yvGr#nHcXsaGWX+}b`l}buy>H8`sJ*0Ec#BPy%<|NmCD>|yna;1)@Up3+d-v`Qvemk*8gE`-50+KA^5$ipr}HK&p|^Oqz!~}NB5BelTczri_&)UT-|xyov`be>mPeZ=%hf3Bq^eVFA2($z`^o7qzRT9~b5_bv-_~UzKd+MJQhqLLo@EWs@@yt&c&#<5)X){+pj^psG%0o7R5|pG2rN0mUp6Jcx6U{95Z~A9JZlE+kbs7|snvsbt2|1ZDovxaJYB(_oWo}?slBuF!C;Xtu|eFV@0y?}7il;ec<>+K{B@PqX?2}?tLz<YUsmK<k$TvW_OIMDUb0wZ#f4X|lX(glEb45LdN*aYs2>3wO_CMfEHCF*-mKgdi=@g@-c6h#TY5O3m(|`&*btr;|AUU3s_=ZDo>2eV8+&*D+8=s8AR@uPi|iWz<Qe~4H%D2MuJFsO^0(9#*LIWSS3%0_4%<1_S&9&ky@_}i(YneCrk$!*OZYy&%yL}3CmOb6DCl4qY85rfMO4vl1pXF4ISnUAr_(4eZ_+9VJ&BFl7y|G>PtAc_AfsfxPK!n0e^Y!@_~D?x1+A4J3B&)RDdak{oN*O)cP+vr9dI{(Enm^-{Mz?qyQFF&Mp+mV!tn{BidI7uDc<`C8?J@>G*z~SK1{-}c~vrR;wqg3^#%M#>^Jn_@3=qSc<lMWkXGs9Y1L%&JoQ_w%(EszmI50pdMq<U3ve%_{b%B}w(5k(Knf5qMHe|Z0j$OaS})3m=pE<@4m5shiE@=KVmcF@;gDCp*H}%~<f$A?Kxu+@jxBghxr*IGCbOE5Ovh$w^=tb~f&vcVQ_<MxO)YRobuf}V541+@nd@1iNxBM@2sR$+lIDEXT&9(4Vgs2pv(=P06v1!jS(Nh)9(bD=J}=9w=i;443Hs*AuAR+OW4_1cS%Y<V@4!L3+oyLtWd96Ec5R>wqF=Ou-I<*mh<gC5Q}RshN!Usp6p+z!ljkecGlI&WoF08Pot(fw_km?&7fnu7Nd_qSLy~XOv#Kg9*#4is|6Ac*TxQ;%zWUN@F0-G$|E~?}GM7L7@VB*>|MbH@3A&(t)B$QYmK)SUg&BItUwCJR@c%$cABm?Su-oh+<6|Uo*MgZR#l<GM0R83)5QHaQw23ZKV4Poy*C-wRaA>_fzg>$rYP^f8+^pjkCPN~!)NbMElEXQy)ekL6SuSpMRJ2HF7con7Xo7Df-0*}!ll<Q3F$eYRqO5L>PA~J-Gx2;!*F^?GCAp2|r2H3F7pOsz6UomXfYkW<<<qxsU;OUn*>j&yif@TZxtmg0-ub*~{z&W|YASLoM%t$N4Q|1RHgDLG(Z+v10ZP5$FLW*PRV=}T$MPc%in}l(#BaO$06cZ5168`r-XYV`(E~aSVbCAJnS9{Em_D6y1BC)3ogp<+l6VobJRl%KopET5M>fRb0HFLf4}W(V#%$bLQL7L0220-y?j-)VAuUIKReMWIwYHc?7CcLXmRrd=h{6Gjy2hh5bG#*`N7sHP#43j7kW;Ds@Hm%A9n1H4=8M}vkl=(je@a7&XE>^)P{DmbDsJ$A`$ZYI_z`T<7@cqu_^Yh0LE(=HIMtHb9HT9n&(n32E__4v1Kl0?Di99-LDHoDBTp>SSF4l&!n>9?N=n8*=JD`(3D}xGKwBMbc&>kiw<$7lBw#QRNO2^)Do&SZn!+yJ=4Ao^3L7e2#0YTRDv~q*@dsxw&tIJX*++be4$%fk4K(~+as#Es{KtRyxVLw8`Qv|p<@DeG{6GH&Qe%#?<6pd{g3l6jWr>@5b{?S_hzGE^kN^sG1qUYn+%i>7X5s)5*hFM`z($dEA^@dRboS=Wt2e^hfd)L>t|bTqp86pVfb;Q4>hat=RDzafM}7MPTi<@0>PNOS%aSlft5-Ps@%!Rp<lV_Jw~DyLE&&c|VEy*4iAJcq0c9*UU_~H%<hO9NigK0Y8Hu>N?U05D4e2XPK8ekd&Z?B%2y4VckRg?uhQ&~>992g7aBC?-5{Oo4pcFt7-4yA&H5>w14{orS5J!bMp@9a0#oy;8I9`g4rchQEvqxJQmTj3$(oR-Q(|359we3N-0X#f-4OwvwZx&(#!o05?Lm=w_h70$>FBCh3dcv&4&^PI|*aSrcKL^heK8;)L15Hj(QxmC~6I)~#fy;#S((q95+5wVvwg*Dfg3-u-f(@qvg^Io<;1-O!E5_{I$H#rYhl!&)E7><R>ep>-zWqf4OBrem-ie1D<0(#0YI+4)yH|j+{=+|PyqQwr=Rf`@Fc?(cA9?eizW>((RIs0Z_{SA=NPsonM8cCK#F*kGYP$x5!5_}vynXTNC5q8Co%SP^CP&w&{^0!0)0b~wzj|{X_cnv6iW(htN7oO1&-)PViesLMsEy<=py9TLuPg8+X6Pj)4{DuU1M|B+WuGY!&*1Fk^VhFlygbKgE}LdukB*O1@^41Pe3nI^Ek;Fl8C{gu$8gfC^gHqTn4yW5^NY+MJU{#5>DOPLzojjw!Q(pn4tM(W<OFsFTzpq)h1+rp-!76h?s-`(@JaLu(n14twr=qICzi-Y*CzN5kvL9IqM`TMNrXf)+bk|nGI$@I;8RdFirAdl8fTTfi^(w*OOGc`hR`j+7?~yWt5}Ws8IOr43)L6c&gwQ^Cv~0HJtM<vU`c?Q>oi`1y9=0uKr(^|W3>)ycyQ;Z1v;-6l5)4(!B<cJ6u&ux&42m!EdKn@=Vyq`(@(wofWwC$i$4R_LWoHsX~VS3>>}VlLZ9F-xE1&B%Z(X&_wV!bt(3TELx_$Oeb1($G{XT_kL<U`JM|uqJy&CV4tson8i0QMwx_q+)QvX-zcMNxV1t0l4V>jPieR+J51mj-GU|ZXX@F|P=>b4(U`Ihf!I8;tJ#=aC2z>x3GVO(+ZQ(sW2m;xmMToa9>kJ7=B1S@kEchw~Qw?Z6TWwaP_Xoyt8p3e`og|RQ1HPY}hEqcmA*ewu2Od$()!JZSI935nIrNsGG&bN~6`PMSiYTkWYzJwew-2?I<9ArH0L>CAU8U^>3wjykxr6T8NV!4i#Ct8pNgkk-&pa&x@i4@_eu$8uL1QiM`k*CU)AMVj<rBj4HKJCIf50Y6h>J8`)LwH_dOUKwmBnT?1Ipa9l@=4jhyZS;gLToXesLDKdDqZ8Q3LMaSU9*zGx{muop+M|uSf*wO%c0qVP)z)akC=r7&=#v9Y5N&n-U(mDI8iPc7@sxkLl5v9*^k-CsFiKr~%!3U^-&PI6mFCM2!c~p4CzD5kY|%DwR|y!IX1qGHFUdDfu;Cm5WWD24UouG-g-Bbms;_PLV?u?a>&*jsg5H4ig7WVM-Z|q<8d++^pn7H@vRLf#!D&kAI7<R~jV3XsN=!cy?LVO_4x6_XDLqb@Ulu;|24}U5FO(=&#>=iS7aD!}6E(z{qr5&{z>UdV<qXV4;He@Vp7GaY%sx;nG2Sh94`gKwtwm0LSFB&tJq(U%!a|?(EN~FVpw4ub#g6Qumd@8HTxGtP=c8#dTIe*aH2UIF9M01@v%Z^IQs0@K02w;GqMy$?c%~gket?Q}Bz*w{19CGz_hlI!18Z$>{Wxlar}J2cF4f39beyw@lpV#wWy-H#x#b3|vQ+f%SI*Ds0@C+!y8%lrPBVbP^3TTQz8Rdg!5Rn?7m??%2ZCq=7R7KtSppsa%4nC~pFpP5?Os^DvB>vPp7<gs@{syMY^euddSy5OQO~s(}ZqWOS3gkb&|Ze|&PHJVGjJk5uK=<%(dey_+goLu4NGk?SmlFtieF=)m`2gh!sR(WT{#XAWo$cC#k41NdaMN(%HesL(wp7F;%~T&I3G;`L?*hx_s5@1IQXKS3h*;2&%kft6j)6OcO6XSM(=Ez-c>G|QvkU{gVk5X=dqc~Ym#GG7F(LkR{?lZu$fH?!ckBLVhbH^mj~L;-(m@NNNI=_355egG}Lxp@E|!n>2<!!3MO>-y^kx`74$%?zeR*uzfp0V(l1+0V){4>~XiWd0TZE?_SFjN0EVd<Gg_qe#}KK?X9<LC|=gBVYLfr1YjrgEyN32hcYkH#&}y*y9)uWxgDG=`|!*fDWYdiVb=5BGppL5EgW-1w2AVSHn71n2sTY9=q@4kP~4oPeY9j<;zIEM^BOXXf~juPgy%{3y#95H?Vd>rZqIHKOpKLH)7H0ZM^Ln0Z;@`J37l3Z9hIZ4skfr$(jFPvJG0qLasF^yD(!Xh~p)oGmb;d#X+y&v2Fy$bpZ$M5*(E4TVK~5a0mz(K!OS|gwg&H%WA!;<91**UJX7fbl@0JM);H^GK~dU9u(;ge75Jcfoc-5QnoS%y5sS49udb5@Sjit5aH1U3nqOB(+Vu|A5(17BSq~@mCy1_<<B_idk+4VJOyS5hiE9&4s^;w#Gx$47(zCrBv63rLEoqwKbHoxYCz~l-CquD5hG7RZe;#MvQ>Pvqv}#;kKS^NAasrvU8abt8hq|3GQgwvHKbD=J-tZDW#B(8=7^6!!fC9JSS2}PL>-?-C&&<=QY;1Af>V%uUoh-3_=5edHRT2Q?wGVBGLV{BRgzXljLjm<R`bp7u)cT({I)`Z2n5z>t7B{(m~(`D%-T+9hD716xFw-eA48o8DZ;G<u^^S02B=O3if-n4S*HY(O%M|E+#(!kSzu&qC5u}@Yzz2|5xH<Quwm4YaYWKGBKaSBPsXa7e6uQ{by;NdAaof==PVvf{T_1VV9Sr+AcH)5idqwC2WV(A+m>>e_5Vkt1%VEk1MZaT0ROvzT&&c9@UguDTa!Q$fF{q*mDqvTwwr0e{C}=hx)Gu}Pt$b(23>F(4R>$Yu!TI-Xo01opaWrVCaZOxf?pS%t@!u;ki%g^Psy>d@JJh?Nj{V!hresI02Z)3lJZzBvI|&ofCnpDY*y<!uz;w6AIH$K9-mjBa)K2=ijuBPwsQH-7yU5OZ;nI0dVG%Ak-;LvObuY`IC5Yba`nWAKWM;GyqFRd^|3<BHQ{R>SBWy^a2M7QeGN&ilbXP&#U^_84H-W&5o^t-MEXdf8siGma~^~Gj0YcHcO34}4vo@SdJvL^yd$7@VQWCJHC573>{;6|Q0<YJe9?{g7qnSQ?#ieVZ|h_BdJvc0I^|V#Q-KYmaaLy5tukgzG}Sb}_UNPcKX7n*3zkdRc}NB!XmYUlO29(kWBx1=72snm^zRTOdtkn%iRI?uXAHAWv=S&Bszzv&dY)zC0AOg?$RuSRVQ9VWs@kJTQF!Ocyx;O{(glWSuPFgxs|Q5+nAP*kDZd>P%?D0IUd&J^Gs@X=X+YzlKp0AActSxke<-s%C>)Ix3s!_J`&slqW7^P2Mmi6MtwAQdikjm9&mHEl`{V|+Jq#EqgGKwg^G{~ebiE6qBOi#?JW&8l4Zp+$d$oaBB~r%da0}SLhY0Xeq$$eRMz*1RI_!ucpdn40oA}DhrdjgLdO*i)MC#mKi$7E@GgFChs1~~6Cs~upi9F5FAbuhKZB0a#tjDf@3I#(lQ5u{M%_V5Tga_b~?QDup(uUrNPCu7kbktoXXes(I-94SbdA>3QYvo7<qv<rgNoh<*OX1hhg}*aJfW4Gp(1WEeH)5Bf<=pwM=E2qmH5OvMN1&558q0J8U95LSK+k`s;K!|z3)1Ajd}Lwin)8V7d^kmyCGiFbpZP4C;F}EFwvkhKC_ZU|oD)~vz+$8veIIeZ0K<TAtODvtSX+zL(+KE1+@%t#EU9&}LFK3a<kTCmU}`*hGoOrv;f@nTBytJf98`h8MNjTO@}R^5upx%zV+?}lz>LI+KN-qgf$nfl!tT+6#T)~N>9S7&-lGJwrYpSp>AOq78cbwsw`QQMfn<l2)ZA-LjRisfFl+-_#vxH5weGMVdRj3S(A^_+d0%D=5Wlo}m;wc30{v!?cK|#D#-U0cNSqWZya%a~kOA-J5*AZ>0HNAWL(W&wjWiTHE#0%E8S|f^#Kc(qJ?Ma98VyojtQMklcG6*3oksQ~3Cq2eDlJkri}*>5YPfszEkYHN!r7L1f%FMzCvq$^7%}Gqy}KBq8!Hf`Gl5L+XWZnd&D?fd-*iA}Azcr>dt}X0>l>7dqTrlz%99uc6Afbldhn@CZwN=d8%I$%AQg=K(s2C_1hmYt^%Jj~Q!&~}8M=S6%N}-e_+5LrH-;zCqO4uW`}ivR9PQi1W}a$!6D~HO$7yY)=!xQ4qXR&5@*@j4SgOvUb^*lfxvd53-8CU1nDMg?%zeWi=4r6?Uc4hA@ZrQ1fs>Pd6zCf-x?qCl$0xrTdLKXhY>WI3LRiSq#p}o3r|=yM0Oa`}V|avAFL%hLA5S(Rwa}0vr<MSa^o~J%LOAAQ$^}H_+iyW3-U0CV8d@}%+c`<kUb6JR<Az&*a4=0T{okLTzxHI$N5I8Xk$GkjR3GVd)8=3$?!;>?akqn1rA&0_Lq^w>XZ8($IHTtvY=)j5c!Qkk*EqY3!)b>k?A2cN{h+N7evYMoN68eZS3)2LO`<s7LP{7s*g6mbo2BKmK%-&nP);Sh;9GwCr7IZdq;HGb+>8!WmPzywpVw7E$_9TZ$VNuza#1uRi&rBOZ8eJVAb>1*jOl^s!F+fq&O!`Om=A<Tk5JtUq}asIA@Zm8?eN{`?t$A+X*`8!GC<_e5lQJYW%}sXVs!Oo(&G{|vin_Kg@=FX!xxdTd3-0voQxh$x9ZzAy7<n0#*K-Srd(z7cy`;QweC)mlMSijJwo0L_^8`+x<bJ;KI!3shK?i08n=TAARm76=~#T^X9UL>H3EqWQ{^A)U&6ueX_bZ7XP=CXZ{6WA38!2xF(iVy0fx4%=jct9CG-%oGQB{$0gk}L2Rok&`yu|Ff#$TL6a&cV+ZQfD`%-c|lbz!?ImP2%|3W9MqSt=$!bb8Gv3f|hzFHG9IIP(ZMB}0NQP_J5a|nHc<=O4rQD2muwQD>~@~AJ_l%3jVg^XFZ9b$yBbcezN@4yWdi|rY|&ws0;THt@u!(t;FtZnGFi-P``UHwaZxBGp#Ddtih;=nb1Zc+uL0lJNZLyq3sJ}yDjqdsTOo!>h1TNnQRAjdA>2&J7L6fEfPFS!?$Ue<VO<O9rJZ#kOrUObxa8rKm@QF;=3f#he?SF?tCFe8X`CkaOc{#||~(*eX_>d&V8>}(a}!M?0X@RLyN1aRjKg|~@`lQOq;5*TLiZt6?6$p@}j@vjnx6_Ew$zx|c`j^xeeK}QFwU*vT~SB$z<1)VGOsZx%_N05?5p-&kHMJ0qhG8t4^mSr?>NQqW5XfEjSxKA?da|pk$Pt~RbAcQYO2CGzTkLfBN+X3c>JE`q`<+tX9l^@!C%|uq91!k5BuSq4bwy8OkhpsbYvS568w7hr6y(SpsnR2A>UVp}qjaYRjgDAo4W?L8uumLt>{AP$Uul<g$<B+oD105o{DjC^2AmzAC67pU=KgPURDD>sjS2OvOs>c|qtK=%}AeCokAVmmIB9|-ptmvBo=s?`ML5^~xvaBs2Q#)eTp=l41y=Tj;EyXdY%oi1pTpDI_Of0g5nyp=xY<0P*=BcVvgY2q=s-R9(6%MZ4xI;whqqLwgS1|J8rkoWhpHacKR`4z77Vo<-xw*9q6=Dd<<tdB$&~iHGlE=WE0%Y;E1Y8TC_QD(L6}fcD0NBrxWbWSwWKmg@I1qo?fY@sw@Kj>JCM_;7^efWZFqKW5OgO39<3TS%aFxQ-YGFg{nkZnGCGNc#5rard0zy+GplW^?s>NgiHe*gz($m5unwe#qPT+Kh@<S}uGrY-m9vKup;%=tRSSH~@Qf>~ph;Sw>9n&=#na_BT$HqYh8ir+pdPp2)-1_*HB}k0H(4YprwlW%tT6?8Yd2PH)y_c#Klod=TdO&R21HLpf93#NRU6v){8pASu#%%WB>2TUbct#qPWT4N7(gqKdcp>%OX^4iBb7p2*ESrBlF#4ubnHH0~E6`L)0$uT;zvA)+5!w_nO)ZjXDV~_JU_qp^hr;bn1+*XTsZp;(M+e&42OYuGueumI&!#3ZmpR@TrVb$HOf&bI_#L*Q>4~z9U|}&11Mj8PgkQtn#tPgq8V8uQzmvg??dv<3WmM_6RMPHu=4pLqttug~Z?VER>V5^f-bipMkFF&&3ca9WIr6)O(n3=}rgO`LYN=oJOll1^xPM}l=4B!D3O%cb+!j=egYZfdj<g?PyB(nCaGZlY&k#c8F^~m9t_GyZZz#8SUc!r8E{`bE8?V|FN(oAC&>5sFh#gLbRW)^)U#Ee2=M>bEXEm$~)O#$@Ys$bF2S@O=zK3XS2oCA2Tv$;fsPvGdpP2WhRVOl%wZo%WjF|$f?*$=S*DVR({9sQppowxZIV+dP>k4pOp#NJUUjq7}_td@zG(l=wCmCOjlcB%XLq`TkHLOfI0>adC+}oQe0qc0}4}`nfY7+NPkv|hlPlQJkH-_v3_JOt+m{`79x~d3Q?~p|Y?iJLUT{;wN5gUo^m<Y}|l+&@7WBPvENj3;4I_x!pjH^v#zs}X1hWfVC0ubvW3fiUghtf<i^FX?uwx4B|NjUc3kU^KB@UnGOC|+dtQIF+KlY-SjXpEsk9^yna?DQJBdWoTaC#QoVt2Zg~BB{M=To!udw~xqmfV>G({X9~DVG%?$n8*RQmHz%5nu;pNs=R6MT-BzgG!sIqd>CqV5`1KCV`h0GtIni5v7Qz&ICBrBwevd24f~;BgTKt(oz`4vwy~ro-)N*RpR%r=qjf;yNdtS7K;d8j!<Y`d2{3LUoplIt#ab)`%RuSK@qj2Q4HN$$4A3Z`pz4{W<akgUp#@-Uabmj}Y?-1Bu|uS|S=AdU`#OS!Z9rWV?%=2dSC^&kVXJCKx=eKswEH+Ld99P`f{JRF_w>+NXHpSF>paJB+P#6zos8%id+&K7J8Z_Ryc;u{iQ4FWK5rr!hgSZJGW==K(539_zHYf7)5G0C(9m#X^J}G?TORGn)Hx^bXUaf}yP*_oYVCp#y;u2yZ^D6lao86LqUcuoLKAV*126_~i$B7H*ZN!eRQk2n0@yxU0N|0ijm(YFmw?HJ<gRE8!;=<j*qMe^Nv{nXGHh)a3rSO}W~9GdgsI_xG7*OqbyN$aEI)m%ZlNQS_p0!BHOpPtwwA&K0dF-yr)n{AkUjw^Yya+Zl0Qf~k@7DkYs?HuJT;{#q=T0iaf;2xaFviwYOS8A%UewnD#@oR2zxH(8E6UHD2VM`?SRq|#9RAr4cY0jkpkZ3aI#(hMKU-V&M8?PIPJ;kLu&2rgU){OLIsT}jCWkCFqOIhIdI(@E#}?o*SSy+;_c)p6iuK6w!6~j(g^ps{1prZ8dtD(hF`mr!=eMd3UG)1B_QaKkwKp7YBZF8xHKv#4|)y`ltSj^Eb#C1xzS~8|B>nO_zykX{uP%Ri)%`(J(XLX?Tl}PQiq=ky5`)8MH?qKHU|P?6u9Fl#r)k!BRgZ!gRb1MJ_UR65r?k);d_T%eF=sLA1bKuwk-*`{mz28z2r;AAjOMAqmb@HtyLDO-4LH9Bhg-9n2Mb7;!dsRIdIiB^L!BX)2qFWTU+Gb4NT@`wOLv{%wlEN9Wbv!rv~Tl{F!>^1kiOUjM|4G+$RCd2tTNMNKe!qn!;d!;<GiDkJMCKO-oU=Uf=}4Jb)N4)|4WR+?Fw4t)v~L0<0dWYtP%+)hm#m8@ePN<!CUWaWu_ZD@k&c7xiA%)B681xcXS8Vj$jWJNrtA)S2qSsl}kp_%Ijo(lShqSKR)tpLuE(txFht+3>BdhaY{*dcfgF-P<hjKe4ega3vUi3r6?mZdITFH&N$#tzGoxxan#NM>cRG5?&Yx5BC>yI8&sx`~WIzFoEl@;SfNHs5dTiNMSz?bRQblwu_5giDj(z%lXLN!_%o6$F?rcsB^Ry6S^}f+bYZ61WIE!N8$`m#9mLy8#0eNmwxQXKG`V~I7%ToGa3tc(y<^|JMr#x08~+otP&zWtF9P<)oY7onu@*%<@xTJ$13kNyP74r@0-%1{AzP`Yl)iztx`j08<k1&RHTm_-C(Vbs;&cAO42?s&$=mvvj{Hep*xk?iQe7NUr;bh8{%9rCgfT-y`Xm7<_2juzuA|<^#+0{A>TgpkJ}@U?+;=x&}kQ&>Pciq-Yf2#rZr4P?Q)KfA5Ptgo%FjoaK{UBbe*O1Zz<RptF=gmCOF}Ae*x6{j4G~5rjgx<W>F*VxtGBA)O!3ZUO?CuzvqPUY30dK$UL;>%O~*E023VHZol_q%U>?&_hG~Wr`tV;*eNgf+3U>N_nYg#uE~Co+3vPD+SLj*M%HO{CBIXQpo${Zh9RjSW~UFm^Got37HGoGh>O)2>A}McY91_AW-0JB?BwTTqKdq?P&olVHv*#`=jlZ<zqQ6j!N_-)WqFSJHFChL(zRr`&MZOmnOFsi7GjcEAX&+{6U2_v$f`|MS2BUSX$FM>Py-Wy;;%B&7h{Da@@reFJ*Q-MF1u^7d?Yl;HkEDAy+DstT)OpN8X)9<8qvR+QmgY)p}aNXw<WkZ9kg}Ra&q1sZ2t6%F0R{=_FU50Ep->ylcGT4yM1huQl55RZI=@R+Ks~ez1Bom0Wx7Qz5>LgCxa0<#w%ob=YFmh)EeFJYe6+!dUY)>>B66DQTCKuDXD>e&}wt_wd`JzZUIf{z7uzQq1*Q`SGQV*s+QC``%HA8-HvZTd$gCYI?q<PEAbXpdt=j{m{!M*>u{%iS@OEI>a%hcOTEfBCg%i+9dq=nZk=T3cG{7n1>a6<8iLXZai9S>=+&bHzosz1<kxrG(7wWtV{fQ6SayPS;a)iHO;C0%tz~7@<QFnB8e=yb>(qM*JyG5lri^luzL0hRGP~|^Bbp&hlhu!U2pWK&r>3(>-~GYQRjITa4}Bhs;wNh4U8<B~PwMLW1C5&xsT%v`*&kVig20u$O2!NE-bu&>F6Q}$$TblMR%WR`QnwE*N{=tpC@Ae6v)ym9xe&lYPU`GSL=krGleoaA-^nj;N$UO*jQj~0Vf0a)7s<58b89$^gIyOgbsWa+Z_S7%n$Ovoeu!!&B-^iMi^QY5Xh-~qk6_Y>djLj;yl$iQVlZc(h>nbjo<%Vgf)QSH9ODOi#7CWA=dOfz^QTaR`9X2FMBWHR38W&%+y=gJQzQ5V5sZ+$lfNXIsPq_tBo&0DfWQ2mr5o)x?)IcOs9cfv%BK;%oY^$uL?}Ghot!J;$OF5)N^?pEJceZ2!hWU8$R~H(Dg6emJLAPjz}B)Z#=A0hhK`JC<#gZ0Y1oaX1FLF>j8ihEyLP!9yfZUB4x6g=z?ryF0&VE&9v!2V;p*9Gh3d)H`kpvSthMClZc7zz*Nj0*+mXKobTyJcLE25b&GeL8y*Q%%s^zG{Lo0LUP&2y~7-m+h`V}Bqr_oLKRqIopxuJ9e$gB7af1Z8Up8R<z$OFn-FBM`4(`fkD#^4b1ub2LEgh(K#1M?a-76W_*n@T{#d&sd5LO^%0Ia2zG-&AZs@o$OUP%yS~W)V{`)~E=W?w~?7p_$kGp3~`_51OrY<-X@kvg1Lsi1g>)NjfRhALvP7Q~lbz*=7kk5+*&D(Odd4y%(?0cqeDOeycv|6xa}cYCd`n4DyfkAm9OpOoT`f^_c0=?#jL!zEvl1zr-a_VOJd<l2ina9-MF+0Z8cRAxMxFkOB}WUlNyU9M#!{<Ro%Xsv)){Hf~KTYq`o;Q-#El!(8uqndhR2N8XkF3{}3gqBf9kf!6}=s$SNAuCHcqzOwr<U7);h6wFa+G`^zB=Yxx$JJVnp#3|7aUq4ugZZtExl+;g&ALM}(g%>dDDiK#}0^aU=!`Ol|7$Ix~sppV<pdAvE=(Q801-W+&H9lx4P5(z79Dj7@XN>p$(f&4BQ?8*~0ihq~{uOY0$}J;}Mi6J;GR$%Gf#66f%8ivjh3>7@X-eFzspqueMz=?28xH;#nM~J{',
 'pubmed_resume_snapshot.py': 'c-qBU>u=n+5&yn_g`E2!>7r<J0a_Frs6cw>pe~ZsaGb*xTSJf~u9s)MKvIsKApd)3h7Xau>vgU`(FKxSQA2Vbzj={<_UzfG@kUe)i-x{^h}4!v_0Wx#;@2&UPTw+BZzCnDUXFuMSu_eJnYiQqO~mfGRQTh!_wS;rZ=$Mgqoz_-RBu^*I}W^8GJp2$*@AaCIjW?xKX1P9(f;06a@+E2`>4NQAfFU(?db1v=ocF?bP@1|<c2Ofhet~!xBe8xJEo^hqQ&*bN=Rns!I&Idt`>{)Hy0mIFUq%{KKy)n_RkL=qasS;`>rh&yH{}*#qcGnT9ty;gJ^JsRw~&BwG*d{#iC&w*u6?n{B#uwrY6x_$#Qt}>Q8^xFY_(CZ}<(9DoJUYU{#Z#Q1d(#HgAWjkqL^Y-{we2JSD8JhXyoX#FN^*_)DDXl`1uKye8@_x*nQ`bhU`!g99Mcyp$V7m9Spr-EG4~q90^&DJGVYb<5$Fp7adGx)Ii)iJ!UJmU7x`_&u$e>)+4gTy<kS)90TBSFF5#P)sJo)BH-Xv8`%mctQz?*a&b}wXpFjkSZa~G*dUqgx+VYmy=*+C2P)$w^b`y#`^}uC|;${<A3*Ynv=Fzl3F{F@kCh<J()G}G^GS%ARz!%gN5GZ4Wk8SWh3ck>TAV^UIP8sKJTlJt!N^0m-N6z-A)Z~rj+2<6Ns<bOQX#oqCVn1?Uv&X5#K~y@?NT{2YvJzuJhJx@C)0d-)Kvq>Ia2xmf&`{#LNY)YDRYvjmWAvVE<Rt*{KkNNaFWgO7vdi<LJa%Gm+IyZtQ4K`ITgJJFfBZ>gc}TjK&elWi@gGb6Ql6tOnJE-k#%Vi9TCyc+1Lms4Ltoi6QJvopZTthH*DEQ_J>O(&Kt9Ts6V}Qnxj^7ZW+VhYdAtqQ3%}^^g$*5XZrXS&5FE3=@Py5X#5{aqa>f^aX{{J0|#sYu>X+LjiSNR}dPTNCy%L(Y69}3PxrHZxBPxqy&Oy5=gFvrVxr-z>e09;inP5>iGtO3A~^{0xy=a9n8^Pj1dWE5ZQ=8R^8Y#dfmn*vS=4gEz&TWk!PF~cDc2gSL%2cUmb$g1@RndN{WC{$7NT6*({#)J9M127wKBw8|T1Z2&Y5Nms8MA$g`16q|GI`*qEeOTF{4*H|Ugd9rFg5)V?8cMYFSWr<U~z(Swxu+q_RB<idLnbii6xE7p9E_-DV!Wj##2+S9?<@rFVNASlNNja0}=3GfYH@m0FVH?nN=)=ZXf2dU5niy|JU>y9;fUtjaQZ@awb+x%v@i?zg}`dktO3Am+&A#m|6gMUPWKxcOxLMTKcO<@r3r4mAsez}fuRUAiF0I@@R#|(_TphY=?OW3mouef8S8WN2~m>zo*(<531Qq2KOl)z#nrrs*l>_x9}L(rNNcjg~In7XI{$%heaOo=5H*MO%=MgS#Nb;!f#LjNZ3NYG60-~3iyoPIoq$aY%(?O&ItAD{n}21^@A!Q}L>hi4p-4kIwTuNecANdAQZwmYtT`S3%v70?&xpt#X41D4>V@D21GE{obd=&$zU^tnczT_FzaW*NcQtBR1dj6j~QUtZ1YxCv1+)UYem8yLF6l^6eWaZGr3JUHmP7~fUn2r+oy1+*^~6gKi3rVIuB#DJ3qcr<~4%jm-ZQ4JlMMvU`Xsq}0dI)?wbu7=36{B-f|plq;fOr<ON04-1DIRVD#kltU-rBTQSzO^b&S34MZgg2b?q`tN6x_$yxkh21lIuPjCWup?rgqwD_PU0U;7F&6XAw@A4&bt8s5&{FjfdB3r=ZJdWoS*qB#tH<(>hkQDgEAQ)JcgJ#W`Y+K2g1CMgFpnIXV%|9hKs>Kv8jC!Lq5e;sj-C0QB4xbUcG#|OJNj?+am>jf2UDVBL+c$U@AVTJ!}~yKwCLtmC^=r%DGjsn@N0<!jTc1$kwB3Fd|^qvZ6w^>vRdLC0p$VLRj8%ZGCSS<VU^Dtp4KsJpjN8au5luowu`F!J~DvedmhPnS`sB?a|QS_qrZ3xWw#1K6r>hFWawe2*BFt%Ir#nb=40&6s>LX@(~aH|GB2)?T<L<lb-pAuNBVMb_@Wc0zpuC`t(wO*6ChCcGG-#RQ-dmfLn^dhR%$T0d7hA8Z;Tjip)=dSs1H#5OWkYPNmD8xOYgEj`(@;)A8_qjZ@h%j1a{m5TSS48qh;Yjx$dvc>a2N@ez8;C+a7fADOAxc)7DIUz&|e!_;Qr9NXJSs3~?0IH!6Wv&qhYVtPEnl4j>fu6FT}D8qJp?le|@FOtPB0>>+7mn?=_7JJN)_{B^RNM)_%ErruOT*E$i28LME-K?c?wOVT1`$?Z5>W;W)pG$1qGCW4u@S809F^j-W!32#yOLG14KL(H!T^eeTrKZLbydtNBO>qu@jEmQ>Dm?4sA=^DKfsv&_+fu?(7@SR)CjFkd2s>s~2{2{hV^({6Df?=a+X2pg9%9`g;z1Hu57gf18L;&@$x?Hxq3xf|7(rLNg%220r>#}oa7i?AaHtb(mDAxHG~+AZH+EyNCDLcF5KoQ==_j+v&sC3{7;AzVT6R;_57=Z;&xFSh`{KWjCdsVD9)hH!5AL+SGmDG#h-#1qb~Bp!UBkmuzA=~Kurvgo70ieZF8+KvEAR(q?J+o%v(4{Jq+^g*YXDz8xtDJeL%b%=ZBKGSt(lYeP<S}&6%%3_v5K<#)_Rgr!#9)%l&!DOJ%ak)K&5cI!=d~gAHXj5ReTlZIM7A_u++B5Za51>nb&y|HHSpe@R$Uhe?e-g`Wt-Pv-^<(S{vYGXdBX0LHKoi;4q5roh|H~Qh6s;T6qD<EMu^wwNiG_cK?;Q#!fbT92FSRw%Rx40Tb=(=YWvAda$&Roa|~r0c|6l?d=Xo__4a1YkMct!Ojkkt<5fYE~fA{2oh+p<K$hG*m4}xfEOb7Sh%$JO-{vhsw$@u2jzV8WIK?lmzF1c9-ZB?2eke=iEGFdEu8rK@jaVIjV`jp26uGzJ#sr}srL#k`heJi4}y#+hHJL7h#50$T&;G^M;?IOhiLb#xdyQgo_c)hE$o0(y!v{?cB?3UPCG1IRUTf4co5osHOI6g#`7-{G)Fo`R9G6gM1fpprlWM%i~8`~*xVZQhv*%g57W&gE`TAyrK1fOl1|Ry9trV)M3V69*36?bKn&G^y^cn%(A~0j1TF!TP<lfb3rF+0?SvZY9CX^TiY&JPyFom}%Z+IeN=I3K4TAnRS3{UA=m>Wb;SY0{Vk#gy4fojeAr#r;(?|^@tiGSn3dUz!s!`H5^1-;k34<0+kD6-c;4X(5H+O}PPRjJcA#fL_`z+iQkXui`(?U~)zoUpmf0-)-h1S@=fWiA0BJi_T(Qp8(%WO6<g~NWp%7W-vn2obTF*D5V!EMQ*98`q#RH_|_^+n<HzgImH<D`O2S0qJTPXetGXAbD4>3T?)mltn7z_IhiWoh~*%_;gel~bENP%n*`UxPHVhhv0|T_tm5M4>NlAN81^bky5;x4E;mP!tGq`Wo81z5S-^yv$OjLB;1(gvznCS~e&AA`mN^bAD)`ET}pPC+U8)->bP?h|U5m`VSez{~HK2(zR3tS=HB{U!>X^C0#sKPwTk>r~)<uWvy*pdsVnzGX5?W9>=2z+RYyTq;4KfwkoYR>MpWjk_*=OcdM$xD{~j%p{;#!K3%`ZSO-&V1NnOXJC?s+UY_IOByTyLzSte;#?|uc8bz|V>lB}C^ujjHDMqtxZaINX@0i06IIdCwpl9@l4EmN2X5?UjZnyOVvXdgL;bMv<rpPh-(vbO0g~CZei$7J=!lBd)*KAM{vb98T5sjq-tol5DETPYTFn}k*#wSszod9cD+>z+Yro_vwSG^cZ!I4>u%6xDMvsw0o=-}ZC)8zqp*EmS8cs2UYJ*M{dEQrCqVIS^#=Hfr{KxCf'}

EMBEDDED_SHA256 = {'ask120_snapshot.py': '4a6c6800bc8dc949ee49fa9de0b9e41d30062d676f5f51f6826db87a179913f3',
 'audit_snapshot.py': 'dfba4379264469670e4868f2e79bf940d5048c8061d922c7090bcd1766c3d8a8',
 'checkpoint_snapshot.py': '8ecff6c935274743509f8ca1b1a0a85dd612b58f02240c79e6ae9cab03e18ff6',
 'corpus_runner_snapshot.py': '23427350bff4562edb6485f6d57e2bdde38218aef3ce6a005d854ec79dacc348',
 'crawler_snapshot.py': 'b6da9e76223db89833783770e7b98c4b300f47112c8784d6ac8e5d1c8e68c9ac',
 'diagnostics_snapshot.py': '0d45ea2439d7d11f42d7203b3628d18b05a54f8f86b746af235ed3e354af316c',
 'export_snapshot.py': 'e924cfa0a104928aa7d90826118955bbe49bbdca833994bb42746bf7e9c8ba30',
 'longchau_snapshot.py': '287b8cd009ff5e558f2a0e7c5592dd1b430a5e144da23619087c745f57287a3f',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '626e493ad05d4f6c96b257167c2cb81a815614122ce35272317ced8d5240f37d',
 'pubmed_efetch_snapshot.py': '983c7c3c1350c4e3a157a5b1b6b004a85f81074454ced5ffb7476f531f4cc3c7',
 'pubmed_resume_snapshot.py': '2c43d37a923335a61cc03d41632c340668ac8cc45afca15f24ebdcddc36220db'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'pubmed_resume_snapshot.py', 'checkpoint_snapshot.py', 'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}
HAS_ASK120_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        for line in stream:
            if not line.strip():
                continue
            ask120_url = urlsplit(json.loads(line)['url'])
            ask120_id = ask120_url.path[len('/question/'):-len('.htm')]
            if ((ask120_url.hostname or '').lower() in ('120ask.com', 'www.120ask.com')
                    and ask120_url.path.startswith('/question/') and ask120_url.path.endswith('.htm')
                    and ask120_id.isdigit()):
                HAS_ASK120_SAMPLE = True
                break
if not POSTPROCESS_ONLY and HAS_ASK120_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_ask120_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('ask120_snapshot.py', 'ask120_sha256'),
                      ('crawler_snapshot.py', 'crawler_sha256'),
                      ('corpus_runner_snapshot.py', 'runner_sha256')):
        if saved_ask120_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run 120ask cũ cần RUN_TAG mới + RECOVERY_RUN_PATH; '
                             'không tự thay text/checkpoint đã trích bằng parser cũ.')

HAS_VOV_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_VOV_SAMPLE = any((urlsplit(json.loads(line)['url']).hostname or '').lower() == 'vov.vn'
            or (urlsplit(json.loads(line)['url']).hostname or '').lower().endswith('.vov.vn')
            for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_VOV_SAMPLE and VOV_USE_CURL and (RUN_DIR / 'corpus_config.json').exists():
    saved_vov_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name,key in (('priority_probe_snapshot.py','probe_sha256'),('corpus_runner_snapshot.py','runner_sha256')):
        if saved_vov_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run VOV cũ cần RUN_TAG mới + RECOVERY_RUN_PATH để chuyển sang curl/Chrome; giữ raw/records/cooldown.')

HAS_LONGCHAU_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_LONGCHAU_SAMPLE = any(urlsplit(json.loads(line)['url']).hostname in
            ('nhathuoclongchau.com.vn', 'www.nhathuoclongchau.com.vn') for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('longchau_snapshot.py', 'longchau_sha256'),
                      ('corpus_runner_snapshot.py', 'runner_sha256'), ('priority_probe_snapshot.py', 'probe_sha256')):
        if saved_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run Long Châu cũ cần recovery: dùng ZIP đầy đủ, RECOVERY_RUN_PATH + RUN_TAG mới; '
                             'không ép RESUME vì parser/runner đã đổi. Raw và records cũ được giữ nguyên.')

for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (name == 'ask120_snapshot.py' and POSTPROCESS_ONLY and HAS_ASK120_SAMPLE) or (name == 'longchau_snapshot.py' and POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE) or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
resume_spec = importlib.util.spec_from_file_location('pubmed_resume_upgrade', RUN_DIR / 'pubmed_resume_snapshot.py')
PUBMED_RESUME = importlib.util.module_from_spec(resume_spec)
resume_spec.loader.exec_module(PUBMED_RESUME)
if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and PUBMED_UPGRADE_EFETCH_SNAPSHOT:
    upgrade_status = PUBMED_RESUME.upgrade(RUN_DIR,
        zlib.decompress(base64.b85decode(EMBEDDED_FILES['pubmed_efetch_snapshot.py'])),
        {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}})
    print('EFetch snapshot:', json.dumps(upgrade_status, ensure_ascii=False), flush=True)

CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

PubMed mặc định dùng **EFetch GET**, **100 PMID/batch**, **1 worker**, nghỉ ít nhất **0,5 giây giữa request**.
`PUBMED_USE_EFETCH = True` hoạt động cả khi `BROWSER_TEST_MODE = 'off'`; run chỉ có PubMed không khởi động Chrome.
Giữ ID/URL gốc, ghép theo PMID trong XML; `text` có title và toàn bộ abstract cùng nhãn.
Các website ngoài PubMed, Long Châu và VOV dùng Requests rồi Selenium theo cấu hình browser; `RESPECT_ROBOTS` áp dụng lượt crawl website.

Checkpoint XML và batch được lưu trước các record ID. Resume sửa các record còn thiếu mà không GET lại batch đã lưu.
`MAX_NEW` tính số URL corpus mới đã lên lịch, gồm cả PubMed; retry không tính hai lần.
HTTP 400 chỉ retry khi body báo Timeout/Empty Response; HTTP 400 invalid UID/parameter không retry mù quáng.
5 attempts, read timeout 90 giây, budget 300 giây/request batch, backoff + jitter và Retry-After.
Batch còn lỗi chia 100 → 50 → 25 → … để cô lập, giới hạn tổng request/time; sau đó tiếp tục batch khác.
Cuối lượt thử pending thêm một vòng. Cooldown server dài hoặc 3 batch liên tiếp không lấy được bản ghi
thì hoãn để resume; không ném RuntimeError vì riêng một batch lỗi, không đánh dấu complete khi còn pending.
PMID không được API trả về hoặc không có abstract được ghi riêng, không đưa vào corpus accepted.
Xem `pubmed_results.csv`, `pubmed_missing_pmids.csv`, `pubmed_no_abstract.csv`, `pubmed_review.csv`,
`pubmed_pending.csv`, `pubmed_requests.jsonl`, `pubmed_efetch_summary.json` và `raw/efetch/` trong ZIP đầy đủ.

Run mới có hậu tố `_efetch`. Nâng cấp run Selenium cũ bằng `RUN_TAG` mới và `RECOVERY_RUN_PATH`:
reuse raw hợp lệ của domain khác; PubMed HTML cũ được lấy mới qua EFetch.
Muốn thử lại PMID thiếu/no abstract cần run mới; resume giữ những kết quả XML hợp lệ đã hoàn tất.
`POSTPROCESS_ONLY = True` vẫn audit/export run cũ mà không crawl lại.
Resume EFetch cũ tự nâng cấp transport tương thích, giữ nguyên XML/record/schema và các snapshot crawl khác.
Checkpoint full ZIP/HF chạy định kỳ trong lúc crawl, khác với upload corpus cuối cùng sau audit/export.

Tham khảo [NCBI EFetch](https://www.ncbi.nlm.nih.gov/books/NBK25499/).

Long Châu mặc định dùng `curl_cffi`/Chrome, 1 worker, delay 5 giây; không chạy Selenium cho domain này. Parser giữ sapo/nội dung chính; `/benh/` lấy đầy đủ các trường HTML y khoa từ JSON đã xác minh slug/canonical. Xem `longchau_results.csv`, `longchau_summary.json`; parser v2 loại nhóm liên kết “Xem thêm”. Run mới có hậu tố `_longchau_v2`; dùng recovery từ ZIP đầy đủ để tái trích HTML 200 cũ, giữ cooldown. Checkpoint thuộc runner chung: lưu theo cadence cho mọi engine và khi dừng/ngắt; log có tiến độ chung theo ID.

VOV (`vov.vn`, `vov2.vov.vn` và subdomain) mặc định dùng `curl_cffi` với `impersonate="chrome"`, một request VOV tại một thời điểm. Giữ parser hiện có, timeout/budget/robots, raw và ID/URL. Không chạy Selenium cho URL VOV khi `VOV_USE_CURL=True`; 429 dừng nhóm VOV và lưu `vov_request_gate.json` qua resume/recovery. Run VOV cũ dùng RUN_TAG mới + RECOVERY_RUN_PATH.

120ask V2 chỉ định tuyến `/question/<id>.htm`; `text` giữ câu hỏi và từng lượt theo thread. `ask120_threads`, `ask120_text_raw` và `ask120_qc` nằm trong record checkpoint; Parquet corpus không thêm cột. Run parser cũ phải dùng RUN_TAG mới + RECOVERY_RUN_PATH để tái trích raw, không resume ghi đè.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        # Pass credentials to the subprocess through its environment; settings/log contain none.
        for name, value in [('NCBI_EMAIL', NCBI_EMAIL), ('NCBI_API_KEY', NCBI_API_KEY)]:
            if value:
                os.environ[name] = value
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'requests_trust_env': REQUESTS_TRUST_ENV, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'pubmed_selenium_first': PUBMED_SELENIUM_FIRST,
            'pubmed_use_efetch': PUBMED_USE_EFETCH,
            'longchau_use_curl': LONGCHAU_USE_CURL,
            'vov_use_curl': VOV_USE_CURL,
            'longchau': {'workers': LONGCHAU_WORKERS, 'delay': LONGCHAU_DELAY, 'attempts': LONGCHAU_ATTEMPTS,
                          'timeout': [LONGCHAU_CONNECT_TIMEOUT, LONGCHAU_READ_TIMEOUT],
                          'budget': LONGCHAU_REQUEST_BUDGET, 'impersonate': 'chrome',
                          'retry_failed_on_resume': LONGCHAU_RETRY_FAILED_ON_RESUME},
            'pubmed_efetch': {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
                               'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
                               'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT],
                               'budget': PUBMED_EFETCH_BUDGET},
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        PUBMED_CONFIG_CURRENT = {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}}
        corpus_settings['pubmed_efetch'] = PUBMED_CONFIG_CURRENT
        if PUBMED_UPGRADE_EFETCH_SNAPSHOT:
            corpus_settings = PUBMED_RESUME.runtime_settings(RUN_DIR, corpus_settings)
        if CHECKPOINT_ENABLED and CHECKPOINT_UPLOAD_ENABLED and not os.environ.get('HF_TOKEN'):
            try:
                from kaggle_secrets import UserSecretsClient
                os.environ['HF_TOKEN'] = UserSecretsClient().get_secret(CHECKPOINT_HF_SECRET_NAME)
            except Exception:
                print('Chưa có HF_TOKEN: checkpoint local vẫn lưu; backup HF chưa khả dụng.')
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
